# LegalQA (UIT DSC 2026 — Task 2) · Kaggle T4×2 · **v8**

Dẫn xuất từ `legalqa_kaggle_v7.ipynb` (dev FULL 0,5966 · NOLLM 0,5751). Giữ nguyên mọi cell đã chạy
ổn của v7: chunk 2 tầng, BM25, sinh nhãn + tách dev, fine-tune 2 encoder, encode fp16, reranker
zero-shot, LTR tắt. Mọi thay đổi là **nhánh mới qua cổng split-half** ở Cell 12. Nhánh thua thì bài
nộp giữ hành vi v7.

## Thay đổi so với v7, kèm bằng chứng

| # | Khâu | Thay đổi | Bằng chứng có trước khi code | Cell |
|---|---|---|---|---|
| 1 | Xử lý data / chunking | **Ứng viên đa độ hạt**: ngoài Điều đầy đủ, thêm cụm 1–3 khoản liên tiếp của top-5 Điều (Điều ≥ 200 từ). Điều bị cắt `dieu_capped` được đọc lại nguyên văn từ file để tách khoản. Chunk cũ giữ nguyên, không có ngưỡng cứng: Điều đầy đủ vẫn là ứng viên, bộ chọn quyết định. | Đo 0 GPU trên 3.427 Điều gold: oracle 0,6596 → 0,6955 (+0,036). Theo độ dài Điều: 400–700 từ +0,048, ≥ 700 từ +0,111. 55% đáp án gold trích dạng "khoản X … Điều Y". | 11, 11b |
| 2 | Chọn ứng viên (T2) | **CE-METEOR**: fine-tune reranker tầng 2 bằng nhãn mềm softmax(METEOR/τ) trên nhóm ứng viên (Điều + cụm khoản) của câu train đã loại dev. | v7: CE chọn trúng ứng viên tốt nhất trong top-5 ở 60% số câu (oracle top-5 0,6403). v6_2 fine-tune bằng nhãn **citation** thì âm ở cả hai nửa; v8 đổi nhãn sang chính độ đo. | 11c |
| 3 | LLM ở các khâu v7 thất bại | **LoRA cho Qwen3-1.7B**, đa nhiệm, chỉ dùng nhãn Task 2: (a) viết câu trả lời theo văn phong `train.json`; (b) chọn ứng viên listwise, nhãn là ứng viên có METEOR cao nhất. Adapter bật/tắt được, nên cổng so LLM gốc với LLM-LoRA trên cùng câu. | v7: LLM listwise gốc âm (`list_only` −0,08); nhánh `free` (LLM tự viết cả câu trả lời) −0,118 ở cả hai nửa. LLM gốc không nắm văn phong đáp án. | 11c, 11d, 12 |
| 4 | T2 | Nhánh **chỉ gọi LLM khi CE lưỡng lự**: margin hạng 1–2 dưới trung vị. Ngưỡng tính từ phân phối điểm, không nhìn gold. | v7: dư địa đổi ứng viên dồn vào nhóm câu có margin thấp (gap trung bình 0,059 so với 0,019). | 11b, 12 |
| 5 | Hậu xử lý | Câu kết v7 bị cắt ở 96 token và lẫn phần nhắc lại prompt ("Câu hỏi: …"). Sửa: `clean_concl` bỏ câu kết khi không có "Như vậy"; khi chạm `max_new_tokens` thì cắt về câu hoàn chỉnh cuối; nâng trần 96→160 và 512→768 token. | Soi `v7_dev_arms.jsonl`: nhiều câu kết dừng giữa câu. | 10b |
| 6 | Bỏ | **Cổng R** (tầng 1): v7 đo 12 nhánh + vòng 2, không nhánh nào qua; tắt để lấy ~60 phút cho Cell 11c (bật lại bằng `V8_RUN_R_GATE=True`). **Temperature scaling**: không làm, vì điểm được pha bằng z-score nên chia nhiệt độ không đổi thứ hạng. | `v7_decisions.json` | 2, 12 |

### Nguyên tắc: không dùng LLM làm trọng tài chọn giữa các bản trả lời
LLM chỉ được dùng ở chỗ kiểm được bằng số: chấm và xếp hạng ứng viên (qua cổng), viết câu trả lời
(qua cổng), và phát hiện văn bản bị cắt cụt (quy tắc cơ học, không phải LLM tự chấm). Không cho LLM
tự đánh giá "bản nào đầy đủ hơn" rồi chọn giữa câu trả lời ngắn và dài. Lý do, theo số v7: khi LLM tự
viết toàn bộ (`free`), METEOR thua template 0,118 ở cả hai nửa (thua 244/300 câu). Chỉ
`free_plus_verbatim`, tức giữ nguyên văn Điều, mới dương. Nghĩa là "đầy đủ" theo cảm nhận của LLM
không trùng với thứ METEOR thưởng. Thiên lệch ưa câu dài của LLM-as-judge cũng là failure mode đã
được ghi nhận (Zheng et al., 2023).

> ⚠️ **Đính chính phân tích trước khi code v8.**
> (i) "CE thiên vị hạng 0" là mệnh đề hiển nhiên, vì hạng 0 do chính CE xếp. Vấn đề thật là CE
> chỉ chọn đúng 60% số câu.
> (ii) 10 câu "ok nhưng còn gap" thực chất là gap **chọn ứng viên** (ok = METEOR ≥ 0,60), không
> phải lỗi hậu xử lý.
> (iii) Các ví dụ "verbosity" trước đây ghép điểm nhánh T2 `list_only` với văn bản nhánh G, tức là
> ghép sai. Nguyên tắc ở trên dựa trên số của cổng G.

## Thứ tự chạy và VRAM
1. Cell 10 nạp reranker zero-shot.
2. Cell 10b chỉ định nghĩa hàm LLM.
3. Cell 11 / 11b: hàm retrieval + ứng viên đa độ hạt.
4. **Cell 11c** dựng nhãn METEOR (CPU nhiều tiến trình), rồi train song song: CE-METEOR trên GPU0 ∥
   LoRA trên GPU1. Encoder dense tạm chuyển ra CPU trong lúc train.
5. **Cell 11d** nạp CE-METEOR và LLM (gốc + adapter) lên mỗi GPU.
6. Cell 12 chạy các cổng; Cell 14 sinh bài public.

## Hai bài nộp

| File | Cấu hình | Model |
|---|---|---|
| `submission_nollm.zip` | nhánh thắng **không dùng LLM** | model trong `REGISTERED_MODELS` + bản fine-tune của chúng (CE-METEOR xuất phát từ `AITeamVN/Vietnamese_Reranker`) |
| `submission.zip` | nhánh thắng **có dùng LLM**, và chỉ khi **thắng NOLLM ở cả hai nửa dev**; không thì trùng NOLLM | + `Qwen/Qwen3-1.7B` (+ LoRA ~17M) |

⚠️ **Ngân sách tham số** (`rule.md §2.1`, < 4B): encoder 1,16B + reranker 0,57B + CE-METEOR 0,57B +
LLM 1,74B ≈ **4,04B**, tức vượt trần. Vì vậy Cell 12 **không cho CE-METEOR và LLM đi cùng nhau**:
FULL chỉ xét các nhánh T2 không dùng CE-METEOR, còn NOLLM được dùng CE-METEOR (≈ 2,30B). Cell 14 đếm
lại bằng `numel()` trên model thật.
`rule.md §2.3`: `Qwen/Qwen3-1.7B` **không** có trong `REGISTERED_MODELS`. Chỉ nộp `submission.zip`
khi đội đã xác nhận model này hợp lệ với BTC.

## File sinh ra
`submission.zip` · `submission_nollm.zip` · `v8_decisions.json` (số đo từng cổng, thống kê cụm
khoản, nhãn METEOR, log huấn luyện) · `v8_dev_arms.jsonl` (từng câu dev × từng nhánh, kèm id ứng
viên được chọn và văn bản LLM sinh) · `eval_harvest_*.json` · `experiment_log.jsonl` ·
`checkpoints/reranker-meteor-ft`, `checkpoints/llm-lora`.

⚠️ Dev 300 câu (150/nửa): hiệu ứng dưới ~1 điểm có thể không phân định được. Log in cảnh báo
`|Δ| < 2·SE` cạnh từng dòng. Cổng T2 có ~25 nhánh, nên chọn nhánh tốt nhất trên nửa A có thiên lệch
chọn lọc; nửa B là chốt chặn.


In [ ]:
# Cell 1: Cài đặt thư viện
!pip install -q -U "transformers>=4.51" sentence-transformers datasets "accelerate>=1.1.0" nltk rouge_score sentencepiece peft
# bitsandbytes (optimizer 8-bit) — cài riêng, KHÔNG chặn nếu lỗi (hay gặp trên Windows/GPU cũ):
!pip install -q -U bitsandbytes || echo "bitsandbytes cai khong duoc - se tu dung AdamW thuong, khong crash"
# lightgbm (LTR — STAGE 2B): thường đã có sẵn trên image Kaggle chuẩn; cài phòng hờ, KHÔNG chặn nếu lỗi (LTR sẽ tự bỏ qua, xem Cell mới "STAGE 2A+2B").
!pip install -q -U lightgbm || echo "lightgbm cai khong duoc - STAGE 2B (LTR) se tu bo qua"


In [ ]:
# Cell 2: Đường dẫn và tham số
import os

# BẢN SỬA (log lỗi thật: chạy trên máy cá nhân RTX 2050 4GB nhưng CONTEXT_DIR trước đây cứng
# đường dẫn Kaggle /kaggle/input/... -> FileNotFoundError): tự nhận diện MÔI TRƯỜNG thay vì
# cứng 1 kiểu đường dẫn — /kaggle/input CHỈ tồn tại thật trên Kaggle, nên dùng chính nó làm
# điều kiện phát hiện. Nhờ vậy CÙNG MỘT nguồn code chạy đúng trên cả 2 nơi (notebook Kaggle
# tự lấy đường dẫn Kaggle, .py trên máy cá nhân tự lấy đường dẫn CẠNH SCRIPT — đúng quy ước
# HERE-relative của legalqa_local.py, dữ liệu đặt cùng thư mục file .py).
IS_KAGGLE = os.path.isdir("/kaggle/input")

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/datasets/anhnguyen7508/uit-data-science-dataset/"
    CONTEXT_DIR = os.path.join(DATA_DIR, "selected-contexts/selected-contexts/")
    TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
    WARMUP_PATH = os.path.join(DATA_DIR, "warmup.json")
    PUBLIC_PATH = os.path.join(DATA_DIR, "public-official.json")
    # /kaggle/input CHỈ ĐỌC. Mọi thứ ghi ra phải nằm ở /kaggle/working (được lưu khi
    # "Save Version" — đây là nơi submission.zip phải nằm) hoặc /kaggle/temp (KHÔNG được
    # lưu, mất khi session kết thúc — dùng cho cache/model tải về, đỡ tốn quota output).
    OUT_DIR = "/kaggle/working"
    CACHE_DIR = "/kaggle/temp/legalqa_cache"
else:
    # Máy cá nhân (hoặc bất kỳ máy nào không phải Kaggle): đặt file .py CẠNH train.json,
    # public-official.json, selected-contexts/ — đúng quy ước của legalqa_local.py, không có
    # trần thời gian phiên nào để lo (khác Kaggle) nên OUT_DIR/CACHE_DIR cũng nằm ngay cạnh
    # script, dễ tìm dễ dọn.
    HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    DATA_DIR = HERE
    CONTEXT_DIR = os.path.join(HERE, "selected-contexts")
    TRAIN_PATH = os.path.join(HERE, "train.json")
    WARMUP_PATH = os.path.join(HERE, "warmup.json")
    PUBLIC_PATH = os.path.join(HERE, "public-official.json")
    OUT_DIR = HERE
    CACHE_DIR = os.path.join(HERE, "cache")

HF_CACHE_DIR = os.path.join(CACHE_DIR, "hf")
NLTK_CACHE_DIR = os.path.join(CACHE_DIR, "nltk_data")
TRAINER_TMP_DIR = os.path.join(CACHE_DIR, "trainer_tmp")
for _d in (OUT_DIR, HF_CACHE_DIR, NLTK_CACHE_DIR, TRAINER_TMP_DIR):
    os.makedirs(_d, exist_ok=True)
os.environ.setdefault("HF_HOME", HF_CACHE_DIR)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(HF_CACHE_DIR, "hub"))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

# Tham số
CHUNK_SIZE = 512      # không sử dụng — chunk theo Điều (xem Bước 1), giữ lại đúng như đề bài
TOP_K_RETRIEVE = 100  # số ứng viên lấy ra sau RRF fusion (BM25 + dense)
TOP_K_RERANK = 5      # trần trên cho số Điều đưa vào 1 câu trả lời (top_n tĩnh VÀ trần adaptive-k)

# BẢN SỬA v4: KIẾN TRÚC HAI TẦNG (result.md §3) — v3 trở về trước cắt theo Điều NGAY Ở
# TẦNG RETRIEVAL, đúng cách Task 1 đã đo là làm document recall TỆ ĐI 1,19 điểm (p=0,024).
# v4 tách: tầng 1 truy xuất trên corpus 450 TỪ (bm25_t1 + dense_channels mã hoá
# all_chunks_t1) để CHỌN VĂN BẢN, tầng 2 mới cắt Điều — CHỈ trong DOC_K văn bản đã chọn —
# rồi rerank lại để chọn Điều cuối cùng. Xem chunk_passage_words()/retrieve_two_tier() ở
# Cell 5/11.
DOC_K = 5                 # số văn bản mở ra ở tầng 1 để cắt Điều tầng 2 (mirror run_qa.py)
MAX_DIEU_CANDIDATES = 100 # trần tổng số Điều đưa vào rerank tầng 2 (giữ THẤP HƠN
                          # run_qa.py's MAX_CANDS=150: v4 giờ rerank 2 LƯỢT/câu thay vì 1,
                          # nên tổng chi phí cross-encoder cao hơn v3 đáng kể — CHƯA đo
                          # được tác động lên thời lượng phiên Kaggle, xem cảnh báo ở Cell 0)
USE_FINETUNE = True   # bật mặc định — có đủ VRAM (16GB/thẻ Kaggle, hoặc OOM-backoff tự lùi
                       # trên máy yếu hơn) thì fine-tune, không cần đắn đo trước. Đặt False nếu
                       # muốn chạy thử nhanh hoặc đang tiết kiệm quota GPU trên Kaggle.

# Kiến trúc "mạnh nhất" — 2 dense encoder khác họ, fine-tune SONG SONG trên 2 GPU riêng (nếu
# có; tự lùi về tuần tự nếu chỉ 1 GPU — xem Bước 4), fusion RRF 3 kênh với BM25, thay cho 1
# bi-encoder 135M tự train trước đây. Không có nhãn document-level của Task 1 nên train HOÀN
# TOÀN từ nhãn citation của Task 2 — chỉ đổi SỐ LƯỢNG và ĐỘ MẠNH encoder, không cần dữ liệu ngoài.
# =============================================================================
# CHUYỂN GIAO TỪ TASK 1 (LegalIR) — hai thứ, mỗi thứ một cờ riêng để đo tách bạch
# =============================================================================
# Nguồn: result.md §31. Bài nộp LegalIR tốt nhất là cấu hình #18 + gộp LSE T=2:
#
#     #18 với max      public recall@5 = 0,9471
#     #18 với LSE T=2  public recall@5 = 0,9519      (+0,48)
#
# (1) HÀM GỘP Ở MỨC VĂN BẢN. Task 1 gộp điểm cross-encoder của nhiều chunk về một
#     document rồi xếp hạng document. `max` cho document thắng nhờ MỘT chunk tốt nhất;
#     `logsumexp` thưởng thêm cho document có NHIỀU chunk cùng tốt. Ở đây chunk là Điều
#     thay vì đoạn 450 từ, nhưng phép toán y hệt: gộp điểm các Điều cùng một văn bản,
#     chọn văn bản, rồi lấy Điều tốt nhất TRONG văn bản đó.
#
#     CẢNH BÁO VỀ T: nhiệt độ phụ thuộc THANG ĐIỂM của cross-encoder. Task 1 đo T=2 trên
#     logit thô; reranker ở notebook này có thể trả về thang khác, nên T=2 KHÔNG bê thẳng
#     được. Bước 6 quét T và chọn bằng SPLIT-HALF (chỉnh trên nửa này, đo nửa kia) — đúng
#     quy trình đã bắt được ảo giác "đỉnh trên toàn dev" ba lần trong repo này.
#
# (2) CẶP ENCODER. Task 1 thay bge-m3+e5-large bằng vnembv2+harrier, cả hai fine-tune:
#     0,9439 -> 0,9471. Cả hai đều KHÔNG dùng tiền tố query:/passage: (job Task 1 chạy
#     harrier với cờ --e5_no_prefix). Đặt False để quay về cặp cũ đã cho 0,5528.
# Negative "cùng họ" cho reranker (chuyển giao PHƯƠNG PHÁP Task 1 — xem Cell 10).
# Task 1 dùng 7 negative/nhóm + listwise softmax; giữ nguyên con số đó.
N_NEG_RERANK = 7
RERANK_FALSE_NEG_GATE = True   # bỏ ứng viên bị reranker zero-shot chấm CAO HƠN gold —
                                # gần như chắc chắn là positive chưa gán nhãn, không phải
                                # negative. Tắt cái này là dạy model dìm đáp án đúng.

# BẢN SỬA v6_1 — LSE ĐÃ ĐO ÂM HAI LẦN ĐỘC LẬP, ĐÓNG HƯỚNG:
#   (1) result.md §8.1, 501 câu dev server: max 0,5630 · lse T=0,5 0,5306 · T=1 0,5249 ·
#       T=2 0,5240 -> âm 3,2 đến 3,9 điểm.
#   (2) Lượt Kaggle thật 2026-09-09 (experiment_log.txt): cổng split-half tự BỎ lse,
#       "agg_mode_chosen": "max" — nửa B âm ở cả 3 giá trị T.
# Lý do cơ chế (không phải nhiễu): Task 1 tối ưu recall@5, văn bản đúng chỉ cần lọt 1 trong
# 5 slot nên thưởng "bằng chứng trải rộng" là đúng hướng. Task 2 trích ĐÚNG MỘT Điều, nên
# đẩy văn bản có vài Điều tầm tầm lên trên văn bản có một Điều xuất sắc là đánh đổi NGƯỢC
# dấu. Ép cứng "max", xoá luôn Bước 6b (quét T) để lấy lại thời gian cho harvest.
AGG_MODE = "max"
AGG_T_GRID = []             # rỗng = không quét T nữa (xem khối lý do ngay trên)
USE_NEW_ENCODERS = True     # False -> giữ nguyên bge-m3 + e5-large của bản 0.5528


BASE_DENSE_MODEL_A = ("AITeamVN/Vietnamese_Embedding_v2" if USE_NEW_ENCODERS
                       else "BAAI/bge-m3")          # họ bge-m3: CLS, KHÔNG tiền tố
BASE_DENSE_MODEL_B = ("mainguyen9/vietlegal-harrier-0.6b" if USE_NEW_ENCODERS
                       else "intfloat/multilingual-e5-large")
# Tiền tố query của TỪNG kênh. Sai tiền tố = embedding lệch hệ toạ độ, recall tụt mà
# KHÔNG lỗi nào bắn ra — bẫy kinh điển, đã ghi ở nhiều chỗ trong repo.
# e5-large là model DUY NHẤT trong bốn cái trên cần tiền tố.
QUERY_PREFIX_A = ""
QUERY_PREFIX_B = "" if USE_NEW_ENCODERS else "query: "
PASSAGE_PREFIX_B = "" if USE_NEW_ENCODERS else "passage: "
                                                            # — bẫy kinh điển: quên tiền tố thì
                                                            # recall tụt mà KHÔNG có lỗi nào bắn ra
                                                            # (embedding vẫn ra số, chỉ lệch hệ toạ độ).
# v6_2 — CHỈ model đã có trong danh sách đăng ký (rule.md §2.3; hạn gửi đề xuất model mới là
# 13/09/2026). Cổng ở cuối cell này chặn cứng nếu ai đổi sang model ngoài danh sách.
DENSE_MAX_SEQ_LEN = 256
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")   # 1 thư mục duy nhất — Kaggle: giữ lại
                                                          # khi Save Version, tự tải về; máy cá
                                                          # nhân: nằm cạnh script như mọi output khác.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
MIN_TRAIN_PAIRS = 50
MAX_TRAIN_EXAMPLES = 9000   # trần số cặp train dùng để fine-tune encoder (Bước 4) —
                             # không sợ vượt giờ: Bước 4 time-box theo FINETUNE_TIME_BUDGET_SEC
                             # (đo tốc độ vài step đầu rồi tự tính max_steps).
N_NEG_PER_ROW = 2

# BẢN SỬA (theo yêu cầu — chạy trên máy cá nhân, không cần đắn đo ngân sách còn lại): batch
# khởi điểm GIỮ NGUYÊN mức đã tối ưu cho GPU nhiều VRAM (Kaggle T4 16GB); trên máy VRAM nhỏ
# hơn (vd RTX 2050 4GB), OOM-backoff đã có sẵn ở mọi bước (Bước 4/5/6/7/5b) tự lùi batch khi
# OOM thật xảy ra — KHÔNG cần hạ tay trước, đúng triết lý "dùng tối đa tài nguyên, chỉ lùi khi
# thật sự hết" đã áp dụng xuyên suốt từ legalqa_local.py.
TRAIN_BATCH_SIZE = 64        # batch HIỆU DỤNG (số in-batch negative)
TRAIN_MINI_BATCH_SIZE = 32
# BẢN SỬA (log ver9): encoder thứ hai kết thúc ở mini_batch_final=4 sau 176 PHÚT cho chỉ
# 334 step — tức OOM-backoff phải chia đôi ba lần (32->16->8->4), mỗi lần OOM là một lượt
# forward vứt đi. Với USE_NEW_ENCODERS=True kênh B là harrier-0.6b (596M, lớn hơn
# e5-large), nên xuất phát thấp ngay thay vì dò xuống. So sánh: kênh A xong 440 step
# trong 82 phút ở mini_batch 16.
TRAIN_MINI_BATCH_SIZE_B = 8 if USE_NEW_ENCODERS else 32   # batch THẬT mỗi forward — OOM-backoff tự giảm nếu máy yếu hơn Kaggle.
ENCODE_BATCH_SIZE = 256      # batch encode corpus mỗi tiến trình GPU.
# v6_2 — encode corpus fp16 có cổng (Cell 9): chỉ dùng fp16 khi không NaN và cosine nhỏ nhất
# so với fp32 trên mẫu >= FP16_MIN_COS. Lượt v6_1 encode fp32 hết 185 phút.
ENCODE_FP16 = True
FP16_GUARD_N = 256
FP16_MIN_COS = 0.995
RERANK_SUBBATCH = 64         # batch xử lý mỗi lần forward reranker (568M) — xử lý theo lô thay
                              # vì lô vừa phải luôn nhanh hơn 1 lô khổng lồ (padding ít hơn,
                              # không nghẽn băng thông bộ nhớ) — xem Cell 11.

# BẢN SỬA: TIME_BUDGET chỉ thật sự cần trên Kaggle (trần phiên GPU ~9-12h). Máy cá nhân KHÔNG
# có giới hạn phiên nào — đặt trần RẤT RỘNG (không phải vô hạn, để tránh treo vĩnh viễn nếu có
# bug logic nào đó) thay vì ép chạy nhanh/cắt ngắn không cần thiết.
# BẢN SỬA v6_1 — TRẦN 8 GIỜ LÀ TỰ ĐẶT, KHÔNG PHẢI GIỚI HẠN CỦA KAGGLE.
# Bằng chứng: lượt 2026-09-09 (v6_log.txt) chạy 586,6 phút = 9,8 GIỜ và HOÀN THÀNH bình
# thường — checkpoint() in "còn lại ~-106,6 phút" suốt 80 phút cuối mà không có gì dừng nó.
# Trần thật của notebook GPU Kaggle là 12h/phiên. Nới TIME_BUDGET lên 11h để phần chẩn đoán
# có chỗ chạy, NHƯNG tách riêng một mốc cứng cho bài nộp:
#
#   SUBMISSION_DEADLINE_SEC = 8h — mốc mà submission.zip PHẢI nằm trên đĩa.
#
# Mọi việc sau mốc đó (harvest mở rộng, error analysis) là phần ăn thêm: mất cũng không ảnh
# hưởng bài nộp. Đây là lý do Bước 7 (sinh public) được đẩy lên TRƯỚC harvest mở rộng trong
# v6_1 — xem Cell 0. Nếu Kaggle bị kill ở giờ thứ 9, ta vẫn có bài nộp hoàn chỉnh.
TIME_BUDGET_SEC = (11 * 3600) if IS_KAGGLE else (48 * 3600)
# v6_2: +1,5h cho fine-tune reranker và so nhánh trên dev (xem Cell 10/12).
SUBMISSION_DEADLINE_SEC = (int(9.5 * 3600)) if IS_KAGGLE else (24 * 3600)
# BẢN SỬA (log ver9): hai encoder ngốn 259/490 phút = 53% phiên, mà METEOR gần như không
# nhúc nhích. Trong khi nút thắt đo được nằm ở TẦNG CHỌN ĐIỀU: oracle chọn tốt nhất trong
# 5 ứng viên đã lấy ra cho 0,6401 so với 0,5630 thực tế — +7,7 điểm bỏ trống. Rót thời
# gian sang reranker, nơi vừa đổi sang công thức Task 1.
FINETUNE_TIME_BUDGET_SEC = (100 * 60) if IS_KAGGLE else (16 * 3600)
DEV_EVAL_SAMPLE_SIZE = 300

# BẢN SỬA (tái lập được kết quả + ablation warmup + sổ thí nghiệm — theo phân tích
# chênh lệch điểm .py-vs-Kaggle, xem PHAN_TICH_KY_THUAT.md): trước đây random.seed(42) chỉ
# đặt ngay trước lúc lấy mẫu dev-eval (Bước 6) — random.sample/random.choice ở Bước 4 (chọn
# tập con để fine-tune, chọn hard-negative dự phòng) chạy TRƯỚC đó với random state KHÔNG
# seed, nên 2 lần chạy cùng code vẫn fine-tune trên 2 tập con khác nhau. SEED được set_all_seeds()
# NGAY SAU khi import xong (Cell 4) — trước Bước 3/4 — để tái lập được. USE_WARMUP cho phép
# ablation có/không warmup.json ở CÙNG seed. EXPERIMENT_LOG_PATH nằm trong OUT_DIR — trên
# Kaggle được giữ lại khi "Save Version" (khác cache/ ở /kaggle/temp, mất khi session kết
# thúc); trên máy cá nhân nằm cạnh script như submission.zip.
# CÂU KẾT — xem docstring render_answer() ở Cell 11 để biết số đo đầy đủ (+4,8 điểm METEOR
# so với không có câu kết, đo trên 501 câu dev, xác nhận bằng split-half).
CONCL = "echo2"          # none | echo | echo2
# v6_2 — OPTION 2: biến thể CÂU DẪN (chỉ đơn vị Điều; câu kết giữ echo2). Đo trước 0 GPU trên
# 3.427 Điều gold của train.json, METEOR thật, Δ so với T0 (hai nửa cùng dấu, SE ~0,0001):
#   T1_theo_qd_tai +0,0007 · T2_title +0,0029 · T3_theo_qd_tai_title +0,0040
# Hiệu ứng THẬT nhưng NHỎ (+0,4 điểm). Cell 12 đo lại trên ứng viên pipeline thật chọn và chỉ
# đổi khỏi T0 khi cả hai nửa dev cùng dương.
ANSWER_TEMPLATES = ["T0_current", "T1_theo_qd_tai", "T2_title", "T3_theo_qd_tai_title"]
ANSWER_TEMPLATE = "T3_theo_qd_tai_title"   # v6_2 chọn T3 (hai nửa cùng dương); v7 đo lại

# NHÃN HUẤN LUYỆN — CHỈ Task 2 (v6: đã xoá hoàn toàn code Task 1, xem Cell 0).
# Nhãn citation của Task 2 (build_train_pairs, Cell 7) phân giải được ~47,8% câu (3.349/
# 7.000) — phần còn lại KHÔNG có nhãn document-level thay thế trong notebook này.
SEED = 42
USE_WARMUP = True   # đặt False để ablation: chỉ dùng train.json, không gộp warmup.json
EXPERIMENT_LOG_PATH = os.path.join(OUT_DIR, "experiment_log.jsonl")

# BẢN SỬA (kết quả thật: dual-encoder 0.5215/0.4829 chỉ nhích rất ít so với single-encoder
# 0.5199/0.4806 dù retrieval mạnh hơn nhiều -> retrieval không còn là nút thắt chính, reranker
# ZERO-SHOT giờ nhiều khả năng là trần chặn điểm. Fine-tune reranker trên chính nhãn citation
# Task 2 -- tái dùng `rows` đã build cho Bước 4, KHÔNG cần dữ liệu thêm.
# =============================================================================
# BẢN SỬA v6_1 — TẮT FINE-TUNE RERANKER. Đây là thay đổi lấy lại NHIỀU THỜI GIAN NHẤT.
# =============================================================================
# Số đo thật, lượt Kaggle 2026-09-09 (v6_log.txt + experiment_log.txt), CÙNG MỘT lượt chạy
# nên so sánh là paired, không phải so giữa hai phiên khác nhau:
#
#     Recall@1 dev (n=143)   zero-shot 58,0%   fine-tuned 55,2%
#     METEOR  dev (n=300)    zero-shot 0,5709  fine-tuned 0,5382      -> âm 3,3 điểm
#     "reranker_source": "zeroshot"   <- dev-eval TỰ CHỌN zero-shot
#
# Chi phí của nhánh thua đó: 110,4 phút fine-tune + ~28 phút chạy thêm một nhánh dev-eval
# = 138 phút (23,5% phiên) cho một checkpoint bị vứt đi ngay sau khi đo.
#
# VÀ nó thua vì một BUG CỤ THỂ, không phải vì phương pháp sai — xem Cell 10:
# `mine_family_negatives()` gọi `load_reranker_on(device, "zero-shot")`, truyền CHUỖI MÔ TẢ
# "zero-shot" vào chỗ đợi TÊN MODEL. HuggingFace không có repo tên "zero-shot" -> trả None ->
# in "[mine] không tải được reranker để lọc -> bỏ chốt chặn, RỦI RO CAO" -> toàn bộ negative
# vào thẳng không qua lọc false-negative. Đúng cái chốt chặn mà result.md §8.3 gọi là BẮT
# BUỘC: không lọc tức là dạy model DÌM đáp án đúng xuống. Hệ quả dây chuyền: reranker-ft
# OOM-backoff tụt batch 8->4->2->1 rồi chạy 7.494 step ở batch 1.
#
# Vì sao v6_1 vẫn TẮT thay vì sửa bug rồi chạy lại: sửa một dòng thì dễ, nhưng để BIẾT bản
# sửa có thắng zero-shot hay không vẫn phải trả đủ 138 phút mỗi lượt, và đó là 138 phút lấy
# thẳng từ chỗ duy nhất còn dư địa lớn đã đo được (§7: +7,7 điểm ở tầng xếp hạng). Bug đã
# được sửa sẵn trong Cell 10 và cờ này để lại nguyên — bật lên là chạy được ngay khi nào có
# một phiên GPU rảnh để dành riêng cho việc đó.
#
# v6_2 — BẬT LẠI, với hai thay đổi so với lượt thua 3,3 điểm:
#   (1) công thức train chép đúng Task 1 (Cell 10: lr 1e-5 từ gốc, tích luỹ 8 nhóm, warmup,
#       gradient checkpointing) — bản cũ tụt về batch 1 và dùng lr của vòng train tiếp;
#   (2) KHÔNG thay thế zero-shot nữa: Cell 12 chấm cả hai trên cùng dev, qua cổng split-half.
USE_RERANKER_FINETUNE = False   # v7: v6_2 đo âm hai nửa (ft2 −0,0096 · ft −0,0177)
RERANKER_BASE = "AITeamVN/Vietnamese_Reranker"
RERANKER_FT_TIME_BUDGET_SEC = (75 * 60) if IS_KAGGLE else (6 * 3600)
RERANKER_FT_ACCUM = 8                   # nhóm (câu hỏi) mỗi bước cập nhật — Task 1 --grad_accum 8
RERANKER_FT_EPOCHS = 2                  # Task 1 --epochs 2; bị cắt sớm hơn nếu hết ngân sách
RERANKER_FT_WARMUP = 0.1
RERANKER_FT_MAX_LEN = 512
# BẢN SỬA (log ver9): 1e-5 -> 3e-6. Đây là lr Task 1 đo được cho ĐÚNG công thức
# listwise + negative cùng họ mà Cell 10 vừa chuyển sang. Giữ 1e-5 với loss listwise là
# trộn hai công thức khác nhau.
# v6_2: 3e-6 là lr Task 1 dùng khi train TIẾP từ checkpoint vòng 2 (eval/job_family.sh);
# train từ model GỐC Task 1 dùng 1e-5 (mặc định eval/train_reranker.py).
RERANKER_FT_LR = 1e-5

# BẢN SỬA (log lỗi thật: torch.AcceleratorError OOM ngay ở lần optimizer.step() ĐẦU TIÊN,
# trước khi kịp chạy batch nào — full fine-tune AdamW cho model ~568M cần khoảng 6-7GB CHỈ
# RIÊNG optimizer state (2 buffer fp32/tham số), không phụ thuộc batch size. Trên GPU 4GB,
# KHÔNG batch nào nhỏ tới đâu cũng không đủ — đây là giới hạn vật lý, không phải cấu hình sai.
# LoRA (chỉ train 1 phần rất nhỏ tham số, đóng băng phần còn lại) không phải "cho nhanh hơn"
# mà là ĐIỀU KIỆN BẮT BUỘC để fine-tune được model cỡ này trên 4GB — đồng thời PHÙ HỢP HƠN
# về phương pháp với lượng dữ liệu nhỏ hiện có (3.500-9.000 câu là rất ít so với 568M tham
# số, full fine-tune có rủi ro overfit/quên kiến thức gốc thật sự). Trên Kaggle (nhiều VRAM),
# GIỮ NGUYÊN full fine-tune như cũ — không đổi kết quả 0.5526/0.4817 đã có kiểm chứng.
_total_vram_gb = 0.0
try:
    import torch as _torch_probe
    if _torch_probe.cuda.is_available():
        _total_vram_gb = _torch_probe.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except Exception:
    pass
LOW_VRAM_MODE = (not IS_KAGGLE) and (_total_vram_gb > 0) and (_total_vram_gb < 10)
USE_LORA = LOW_VRAM_MODE        # LoRA cho CẢ dense encoder LẪN reranker khi VRAM thấp
USE_8BIT_OPTIM = LOW_VRAM_MODE  # optimizer AdamW 8-bit (bitsandbytes) khi VRAM thấp — cộng
                                 # dồn với LoRA, không thay thế; tự lùi về AdamW thường êm ái
                                 # nếu bitsandbytes không cài được (xem Cell 1).
LORA_R = 16          # rank — 16 là điểm cân bằng phổ biến, đủ biểu đạt cho fine-tune domain
LORA_ALPHA = 32      # thường đặt = 2 * LORA_R
LORA_DROPOUT = 0.05

print(f"Môi trường: {'Kaggle' if IS_KAGGLE else 'máy cá nhân (không phải Kaggle)'}")
print(f"DATA_DIR  = {DATA_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")
print(f"CACHE_DIR = {CACHE_DIR}" + ("  (tạm, mất khi session kết thúc)" if IS_KAGGLE else ""))
print(f"LOW_VRAM_MODE={LOW_VRAM_MODE} (VRAM={_total_vram_gb:.1f}GB) -> USE_LORA={USE_LORA}, USE_8BIT_OPTIM={USE_8BIT_OPTIM}")

# =============================================================================
# v6_1 — HARVEST + BỘ CHỌN ĐIỀU HỌC ĐƯỢC (LTR). Đây là toàn bộ nội dung mới của v6_1.
# =============================================================================
# BỐI CẢNH (result.md §7, đo trên 501 câu dev):
#
#     reranker chọn hạng 1 (hiện tại)                    METEOR 0,5630
#     oracle: chọn tốt nhất trong ĐÚNG 5 ứng viên đã có  METEOR 0,6401
#     ------------------------------------------------------------------
#     dư địa nằm sẵn trong danh sách, chỉ bị xếp sai thứ tự       +7,7 điểm
#
# Reranker chọn đúng Điều tốt nhất chỉ 61,3% số lần, và khoảng cách điểm giữa hạng 1 và
# hạng 2 có TRUNG VỊ 0,0023 — ở rất nhiều câu nó gần như tung đồng xu. Bốn quy tắc rẻ tiền
# (dài nhất / ngắn nhất / gần độ dài trung vị) đều TỆ HƠN giữ nguyên hạng 1, và tương quan
# (độ dài, METEOR) chỉ −0,078 -> độ dài không phải tín hiệu. Cần một bộ chọn HỌC ĐƯỢC.
#
# HARVEST — một lượt GPU, ba mục đích. Chạy retrieve_two_tier() trên câu train và CACHE lại
# top-K ứng viên kèm điểm CE + đặc trưng + METEOR của từng ứng viên. Từ đúng cache đó:
#   (1) train LTR          (2) dev-eval chọn cấu hình     (3) error analysis cho đồng đội
# Trước đây (1) và (3) sẽ là hai lượt GPU riêng; gộp lại tiết kiệm nguyên một lượt.
#
# ⏱️ VÌ SAO KHÔNG PHẢI 7.000 CÂU. Đo từ lượt thật: dev-eval 300 câu / 2 GPU song song hết
# 1.479s, Bước 7 1.000 câu hết 4.862s -> ~4,9 giây/câu. 7.000 câu = 34.300s = **572 phút**,
# tức nhiều hơn cả phần fine-tune + encode corpus cộng lại. Con số đó không nằm trong bất kỳ
# phiên nào. HARVEST_TARGET_TOTAL đặt theo thời gian thật còn lại, không theo con số 7.000.
HARVEST_PHASE_A_N = 900       # PHA A (bắt buộc, TRƯỚC bài nộp): đủ để train LTR + gate.
                               # ~74 phút. Gồm cả DEV_EVAL_SAMPLE_SIZE câu dev.
HARVEST_TARGET_TOTAL = 3500   # PHA C (tuỳ chọn, SAU khi submission.zip đã nằm trên đĩa):
                               # harvest thêm tới mốc này HOẶC tới khi hết giờ, tuỳ cái nào
                               # đến trước. Chỉ phục vụ error analysis.
HARVEST_BATCH = 100           # kiểm tra ngân sách sau mỗi lô này -> dừng sạch, không cụt file
PUBLIC_RESERVE_SEC = 100 * 60 # thời gian GIỮ LẠI cho Bước 7 + đóng gói. Pha A tự cắt ngắn
                               # nếu chạm vào phần này. Đo thật: Bước 7 hết 81 phút.

USE_LTR = False               # bộ chọn Điều học được (LightGBM LambdaRank)
LTR_TOP_K_CANDIDATES = 5      # số ứng viên đầu bảng đưa vào LTR — khớp TOP_K_RERANK, và
                               # khớp đúng tập mà oracle +7,7 điểm được đo trên đó
LTR_MIN_GROUPS = 300          # dưới ngưỡng này thì không train LTR (quá ít nhóm để tin)
LTR_NUM_LEAVES = 15           # giữ NHỎ: đặc trưng chỉ ~12 chiều, vài nghìn nhóm — cây to là
LTR_N_ESTIMATORS = 200        # mời overfit. Gate split-half ở Cell 13 vẫn là chốt chặn cuối.
LTR_LEARNING_RATE = 0.05

# Error analysis — ngưỡng phân loại lỗi, xem classify_error() ở Cell 15.
ERR_OK_METEOR = 0.60          # >= ngưỡng này coi như câu đã tốt
ERR_RANKING_GAP = 0.05        # oracle - thực tế >= ngưỡng này -> lỗi XẾP HẠNG (ứng viên tốt
                               # đã nằm trong tay mà không chọn) — đây là loại lỗi LTR nhắm tới
ERR_RETRIEVAL_CEIL = 0.40     # oracle_best < ngưỡng này -> lỗi TRUY XUẤT (không ứng viên nào
                               # cứu được câu này, LTR bó tay)

print(f"v6_1: harvest pha A={HARVEST_PHASE_A_N} câu, mục tiêu tổng={HARVEST_TARGET_TOTAL}, "
      f"USE_LTR={USE_LTR}, USE_RERANKER_FINETUNE={USE_RERANKER_FINETUNE}, AGG_MODE={AGG_MODE}")
print(f"     ngân sách {TIME_BUDGET_SEC/3600:.0f}h · mốc cứng cho submission "
      f"{SUBMISSION_DEADLINE_SEC/3600:.0f}h · giữ lại cho Bước 7 {PUBLIC_RESERVE_SEC/60:.0f} phút")

# =============================================================================
# v6_2 — CỜ BA CỔNG (Cell 12) + CHẶN MODEL NGOÀI DANH SÁCH
# =============================================================================
RUN_TEMPLATE_GATE = True        # Option 2
RUN_CHANNEL_ABLATION = True     # Option 3: đo từng kênh encoder đang đóng góp gì, 0 encode thêm
REGISTERED_MODELS = {
    "AITeamVN/Vietnamese_Embedding_v2", "mainguyen9/vietlegal-harrier-0.6b",
    "BAAI/bge-m3", "intfloat/multilingual-e5-large",
    "AITeamVN/Vietnamese_Reranker", "Qwen/Qwen3-Reranker-0.6B",
}
for _m in (BASE_DENSE_MODEL_A, BASE_DENSE_MODEL_B, RERANKER_BASE):
    if _m not in REGISTERED_MODELS:
        raise SystemExit(f"⛔ {_m} không có trong danh sách model đã đăng ký (rule.md §2.3).")
print(f"v6_2: reranker-ft={USE_RERANKER_FINETUNE} (lr {RERANKER_FT_LR}, accum {RERANKER_FT_ACCUM}, "
      f"{RERANKER_FT_TIME_BUDGET_SEC/60:.0f} phút) · template gate={RUN_TEMPLATE_GATE} "
      f"· channel ablation={RUN_CHANNEL_ABLATION} · encode fp16={ENCODE_FP16}")


# =============================================================================
# v7 — reranker zero-shot + LLM + nhánh tầng 1 mới. Mọi nhánh qua cổng split-half ở Cell 12.
# =============================================================================
USE_LLM = True
LLM_MODEL = "Qwen/Qwen3-1.7B"
LLM_GEN_BATCH = 8
LLM_ARTICLE_WORDS = 700       # độ dài Điều đưa vào prompt sinh
LLM_LIST_WORDS = 250          # mỗi ứng viên trong prompt listwise (5 ứng viên/prompt)
LLM_YN_WORDS = 400            # mỗi ứng viên trong prompt có/không
LLM_MAX_NEW_REWRITE = 64
LLM_MAX_NEW_CONCL = 96
LLM_MAX_NEW_FREE = 512
PARAM_BUDGET = 4_000_000_000  # rule.md §2.1

RRF_WEIGHT_VARIANTS = {"bm25_0.5": {"bm25": 0.5}, "bm25_1.5": {"bm25": 1.5},
                       "A_0.5": {"A": 0.5}, "B_0.5": {"B": 0.5}}
BM25_KB_GRID = [(1.2, 0.75), (0.9, 0.4), (2.0, 0.5)]
PRF_N_DOCS = 10
PRF_N_TERMS = 10
T2_ALPHAS = (0.25, 0.5, 1.0)
RUN_R_ROUND2 = True
RUN_GEN_GATE = True

HARVEST_TARGET_TOTAL = 0          # phân tích lỗi chỉ trên dev (Cell 15 không harvest thêm)
PUBLIC_RESERVE_SEC = 150 * 60     # public 2 cấu hình + LLM sinh

if USE_LLM and LLM_MODEL not in REGISTERED_MODELS:
    print(f"⚠️ v7: {LLM_MODEL} KHÔNG có trong REGISTERED_MODELS (rule.md §2.3, hạn đề xuất 13/09).\n"
          f"   Notebook xuất HAI bài: submission_nollm.zip (chỉ model đã đăng ký) và "
          f"submission.zip (có LLM nếu LLM qua cổng).")
print(f"v7: USE_LLM={USE_LLM} ({LLM_MODEL}) · reranker-ft={USE_RERANKER_FINETUNE} · LTR={USE_LTR} "
      f"· câu dẫn mốc={ANSWER_TEMPLATE}")


# =============================================================================
# v8 — xem Cell 0. Mọi thay đổi là NHÁNH MỚI qua cổng split-half ở Cell 12; nhánh thua thì
# bài nộp giữ hành vi v7.
# =============================================================================
# Cổng R (tầng 1): v7 đo 12 nhánh + vòng 2, không nhánh nào qua (v7_decisions.json) -> tắt,
# lấy lại ~60 phút cho huấn luyện ở Cell 11c. Đặt True để đo lại.
V8_RUN_R_GATE = False

# (1) ỨNG VIÊN ĐA ĐỘ HẠT — ngoài Điều đầy đủ, thêm "cụm 1..SUBSPAN_MAX_WIN khoản liên tiếp" của
#     top-5 Điều. Đo 0 GPU trên 3.427 Điều gold (METEOR thật, câu dẫn T3 + echo2):
#       oracle Điều đầy đủ 0,6596 -> max(Điều, cụm ≤3 khoản) 0,6955 (+0,036)
#       Điều <200 từ +0,002 · 200-400 +0,013 · 400-700 +0,048 · ≥700 +0,111
#     Chunk gốc KHÔNG đổi; Điều đầy đủ vẫn là một ứng viên; bộ chọn ở Cell 12 quyết định.
USE_SUBSPAN = True
SUBSPAN_MAX_WIN = 3
SUBSPAN_MIN_WORDS = 200
SUBSPAN_MAX_PER_Q = 60        # trần số cụm khoản thêm mỗi câu (chi phí CE)

# (2) CE-METEOR — fine-tune reranker tầng 2 với nhãn mềm softmax(METEOR/τ) trên nhóm ứng viên
#     (Điều + cụm khoản) của câu train ĐÃ LOẠI dev. v6_2 fine-tune bằng nhãn citation: âm hai nửa.
USE_CE_METEOR_FT = True
LABEL_MAX_Q = 2000            # số câu train dựng nhãn METEOR (cắt thêm theo LABEL_TIME_BUDGET_SEC)
LABEL_DOC_K = 3               # văn bản BM25 tầng 1 mở ra cho mỗi câu (+ văn bản gold nếu có)
LABEL_MAX_DIEU = 15           # Điều / câu sau lọc theo số từ trùng câu hỏi
LABEL_SUB_TOP = 5             # tách cụm khoản cho ngần này Điều đầu
LABEL_MAX_SUB = 25
LABEL_MIN_BEST = 0.30         # ứng viên tốt nhất dưới ngưỡng -> nhóm không có đáp án, bỏ
LABEL_MIN_SPREAD = 0.05       # các ứng viên gần như bằng điểm -> không có tín hiệu xếp hạng, bỏ
LABEL_TIME_BUDGET_SEC = (15 * 60) if IS_KAGGLE else (3 * 3600)
CE_FT_GROUP = 8               # ứng viên / nhóm (Task 1 + v6_2: 1 dương + 7 âm)
CE_FT_TAU = 0.05
CE_FT_LR = 1e-5
CE_FT_ACCUM = 8
CE_FT_EPOCHS = 2
CE_FT_WARMUP = 0.1
CE_FT_MAX_LEN = 512
CE_FT_MIN_GROUPS = 200
CE_FT_8BIT = False            # AdamW 8-bit (bitsandbytes) — chỉ cần khi thẻ < 16GB
CE_FT_TIME_BUDGET_SEC = (35 * 60) if IS_KAGGLE else (4 * 3600)

# (3) LoRA cho Qwen3-1.7B — SFT đa nhiệm từ nhãn Task 2: (a) viết câu trả lời theo văn phong
#     train.json từ ứng viên METEOR cao nhất; (b) chọn ứng viên listwise, nhãn = ứng viên METEOR
#     cao nhất. Adapter bật/tắt được -> cổng so LLM gốc (v7) với LLM-LoRA trên cùng câu.
USE_LLM_LORA = True
LORA_FT_R = 16
LORA_FT_ALPHA = 32
LORA_FT_LR = 2e-4
LORA_FT_ACCUM = 8
LORA_FT_MAX_LEN = 2048
LORA_FT_MAX_ANS_WORDS = 450   # gold dài hơn -> không làm đích (không dạy model viết cụt)
LORA_FT_MIN_CTX_METEOR = 0.45 # ngữ cảnh phải thật sự chứa đáp án
LORA_FT_MIN_EX = 200
LORA_FT_TIME_BUDGET_SEC = (40 * 60) if IS_KAGGLE else (4 * 3600)

# (4) Hậu xử lý có bằng chứng (v7_dev_arms.jsonl): câu kết dừng giữa câu ở 96 token và lẫn phần
#     nhắc lại prompt. Nâng trần + cắt câu dở dang khi chạm trần (Cell 10b).
LLM_MAX_NEW_CONCL = 160
LLM_MAX_NEW_FREE = 768
PUBLIC_RESERVE_SEC = 170 * 60     # public 2 cấu hình + LLM sinh 768 token

print(f"v8: cổng R={V8_RUN_R_GATE} · cụm khoản={USE_SUBSPAN} (≤{SUBSPAN_MAX_WIN} khoản, Điều ≥"
      f"{SUBSPAN_MIN_WORDS} từ) · CE-METEOR={USE_CE_METEOR_FT} · LLM-LoRA={USE_LLM_LORA}")


In [ ]:
# Cell 3: Kiểm tra GPU (mong đợi 2x Tesla T4) + tiện ích thời gian + theo dõi tài nguyên
import time
import torch

N_GPU = torch.cuda.device_count()
print(f"Số GPU thấy được: {N_GPU}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} — {p.name}, {p.total_memory/1024**3:.1f} GB")

if N_GPU == 0:
    DEVICES = ["cpu"]
    print("[CẢNH BÁO] Không thấy GPU nào — kiểm tra Settings > Accelerator = GPU T4 x2. "
          "Sẽ chạy CPU, RẤT chậm cho Bước 4/5/6/7.")
elif N_GPU == 1:
    DEVICES = ["cuda:0"]
    print("[CẢNH BÁO] Chỉ thấy 1 GPU — vẫn chạy được nhưng KHÔNG tận dụng song song 2 thẻ "
          "ở Bước 5/6/7. Kiểm tra Settings > Accelerator = GPU T4 x2 nếu muốn đủ 2 thẻ.")
else:
    DEVICES = [f"cuda:{i}" for i in range(N_GPU)]
    print(f"OK — sẽ dùng song song {DEVICES} ở Bước 5 (encode corpus) và Bước 6/7 (rerank).")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

_START_TIME = time.time()

def elapsed() -> float:
    return time.time() - _START_TIME

def remaining() -> float:
    return TIME_BUDGET_SEC - elapsed()


# =============================================================================
# v6_1 — THEO DÕI TÀI NGUYÊN. Gắn vào checkpoint() nên KHÔNG tốn gì thêm.
# =============================================================================
# Chỉ chạy ~12 lần mỗi phiên (mỗi checkpoint một lần), đọc con số torch đã giữ sẵn trong bộ
# nhớ — không gọi nvidia-smi, không tạo tiến trình con, không thread nền. Chi phí thật sự
# bằng 0, khác hẳn phương án lấy mẫu định kỳ vốn phải giành GIL với luồng GPU.
#
# Vì sao đáng ghi: hai sự cố tốn nhiều giờ nhất của lượt trước đều là sự cố BỘ NHỚ mà log
# không hề ghi lại — encoder B tụt mini-batch 32->4 sau OOM-backoff (99 phút cho 149 step),
# và reranker-ft tụt batch 8->1. Khi đọc log sau đó, không có cách nào biết VRAM đã ở đâu
# lúc chuyện xảy ra. `max_memory_allocated` là con số đúng cho việc đó: nó nhớ ĐỈNH, không
# phải giá trị tức thời — nên vẫn bắt được cú vọt dù ta chỉ đọc mỗi checkpoint một lần.
_RESOURCE_LOG = []

def resource_snapshot() -> dict:
    snap = {"t_min": round(elapsed() / 60, 1), "gpu": []}
    for i in range(N_GPU):
        try:
            snap["gpu"].append({
                "dev": f"cuda:{i}",
                "alloc_gb": round(torch.cuda.memory_allocated(i) / 1024**3, 2),
                "peak_gb": round(torch.cuda.max_memory_allocated(i) / 1024**3, 2),
                "reserved_gb": round(torch.cuda.memory_reserved(i) / 1024**3, 2),
            })
        except Exception:
            pass
    try:
        import resource as _res
        # ru_maxrss: KB trên Linux. Đây là ĐỈNH RSS của tiến trình, không phải hiện tại —
        # đúng thứ cần để biết có suýt chạm trần RAM 30GB của Kaggle hay không.
        snap["host_peak_rss_gb"] = round(
            _res.getrusage(_res.RUSAGE_SELF).ru_maxrss / 1024**2, 2)
    except Exception:
        pass
    return snap


def checkpoint(label: str) -> None:
    snap = resource_snapshot()
    snap["label"] = label
    _RESOURCE_LOG.append(snap)
    gpu_txt = " · ".join(f"{g['dev']} {g['alloc_gb']:.1f}/{g['peak_gb']:.1f}GB"
                          for g in snap["gpu"])
    ram_txt = (f" · RAM đỉnh {snap['host_peak_rss_gb']:.1f}GB"
               if "host_peak_rss_gb" in snap else "")
    print(f"[{elapsed()/60:6.1f} phút] {label}  (còn ~{remaining()/60:.1f} phút)"
          + (f"\n           VRAM (đang dùng/đỉnh): {gpu_txt}{ram_txt}" if gpu_txt else ram_txt))


checkpoint("Bắt đầu")


In [ ]:
# Cell 4: Import chung + hằng số regex cho chunk theo Điều
import re
import json
import math
import random
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

DIEU_RE = re.compile(r"^[ \t]*Điều\s+(\d+)[a-zđA-ZĐ]?[\.\s]", re.MULTILINE)
# BẢN SỬA v5 (Batch 1 A1+A2, xem EDA_ANALYSIS_REPORT.md §4-5): tầng bậc regex mở rộng
# cho 15,5% document không có Điều (Mục/Phần dùng ở Quyết định hành chính, tiết số thập
# phân dùng ở QCVN/TCVN, Phụ lục dùng ở biểu mẫu) + chặn trên cho đơn vị bất thường dài.
MUC_RE = re.compile(r"^[ \t]*Mục\s+([0-9]+|[IVXLCDM]+)\s*[\.\s:]", re.MULTILINE)
PHU_LUC_RE = re.compile(r"^[ \t]*Phụ\s+lục\s+([0-9IVXLCDM]+)\b", re.MULTILINE | re.IGNORECASE)
TIET_RE = re.compile(r"^[ \t]*(\d+\.\d+(?:\.\d+)?)\s*[\.\s]", re.MULTILINE)
MAX_UNIT_WORDS = 2000  # A2: chặn trên 1 đơn vị (Điều/Mục/...) — EDA đo outlier tới 189.366
                        # từ/đơn vị khi không chặn (regex bỏ sót heading do lỗi OCR/định dạng)
_MUC_PREFIX_STRIP_RE = re.compile(r"^\s*Mục\s+[0-9IVXLCDM]+\s*[\.\s:]*\s*", re.IGNORECASE)
_PHU_LUC_PREFIX_STRIP_RE = re.compile(r"^\s*Phụ\s+lục\s+[0-9IVXLCDM]+\s*[\.\s:]*\s*", re.IGNORECASE)
_TIET_PREFIX_STRIP_RE = re.compile(r"^\s*\d+\.\d+(?:\.\d+)?\s*[\.\s]*\s*")
SO_HEADER_RE = re.compile(r"Số\s*[:：]\s*([0-9A-Za-zĐđ/\-]+)")
SO_HIEU_RE = re.compile(r"\d{1,6}[A-Za-z]{0,3}/(?:\d{4}/)?[A-Za-zĐđ]{2,10}(?:-[A-Za-zĐđ]{2,10})?")
LOAI_VB_CANON = ["Thông tư liên tịch", "Nghị định", "Luật", "Thông tư", "Quyết định",
                 "Pháp lệnh", "Nghị quyết", "Bộ luật", "Chỉ thị"]
LOAI_PATTERN = re.compile("(" + "|".join(re.escape(x) for x in LOAI_VB_CANON) + ")", re.IGNORECASE)
DIEU_CITATION_RE = re.compile(r"Điều\s+(\d+)\s*[a-zđA-ZĐ]?\b")
_TOKEN_RE = re.compile(r"[^\W\d_]+|\d+", re.UNICODE)
_DIEU_PREFIX_STRIP_RE = re.compile(r"^\s*Điều\s+\d+[a-zđA-ZĐ]?\.?\s*", re.IGNORECASE)


def extract_vb_info(passage: str):
    m = SO_HEADER_RE.search(passage[:1500])
    so_hieu = m.group(1).strip("., ") if m else ""
    if not (so_hieu and SO_HIEU_RE.fullmatch(so_hieu)):
        m2 = SO_HIEU_RE.search(passage[:1500])
        so_hieu = m2.group(0) if m2 else ""
    m3 = LOAI_PATTERN.search(passage[:200]) or LOAI_PATTERN.search(passage[:800])
    loai_vb = ""
    if m3:
        low = m3.group(1).lower()
        for canon in LOAI_VB_CANON:
            if canon.lower() == low:
                loai_vb = canon
                break
    return loai_vb, so_hieu


def tokenize_simple(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())


def norm_so_hieu(s: str) -> str:
    return s.strip().upper()

# BẢN SỬA: seed toàn cục NGAY SAU khi import xong (trước Bước 3 ở Cell 7, trước Bước 4 ở
# Cell 8 — hai nơi duy nhất gọi random.sample/random.choice) — xem giải thích đầy đủ ở Cell 2.
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)
print(f"  Đã seed toàn cục với SEED={SEED} (random/numpy/torch) — trước mọi lời gọi random "
      f"ở Bước 3/4, để nhiều lần chạy cùng code fine-tune trên cùng 1 tập con, tái lập được.")

In [ ]:
# Cell 5: Bước 1 — Chunk corpus, HAI TẦNG (BẢN SỬA v4, xem Cell 2/result.md §3):
#   chunk_passage()       tầng 2 — theo Điều (neo đầu dòng, tránh rách nội dung khi 1 Điều
#                          trích dẫn Điều khác trong thân bài) — CHỈ dùng cho sinh nhãn
#                          (Cell 7), fine-tune (Cell 8/10), và làm ứng viên cuối trong
#                          retrieve_two_tier() (Cell 11)
#   chunk_passage_words()  tầng 1 (MỚI) — 450 từ liên tục, dùng cho BM25+dense retrieval
#                          để CHỌN VĂN BẢN trước — biến `all_chunks` (Điều) và
#                          `all_chunks_t1` (450 từ) TÁCH BIỆT hoàn toàn từ đây, không
#                          trộn lẫn ở bất kỳ cell nào phía sau.
# Cả hai đều trích so_hieu/loai_vb từ NỘI DUNG (không phải tên file).
def _split_words_raw(text: str, n: int) -> list:
    """Cắt PHẲNG theo từ, không quan tâm ranh giới câu/cấu trúc — dùng làm fallback cuối
    (không khớp bất kỳ tầng bậc nào) hoặc chặn trên (A2) cho chunk_passage()."""
    words = text.split()
    if not words:
        return []
    return [" ".join(words[i:i + n]) for i in range(0, len(words), n)]


def chunk_passage(passage: str, doc_id) -> list:
    """Tầng 2 — TẦNG BẬC (BẢN SỬA v5, Batch 1 A1+A2 — xem EDA_ANALYSIS_REPORT.md §4-5):
    thử Điều -> Mục -> Phụ lục -> tiết (TCVN dạng "2.2.1.2") theo thứ tự, dùng tầng bậc ĐẦU
    TIÊN khớp được ít nhất 1 lần. Nếu không tầng nào khớp -> cắt 450 từ (KHÔNG còn trả
    nguyên văn bản như v4 — oracle đo trả nguyên văn bản chỉ 0,195 METEOR, result.md §3).

    A2: mỗi đơn vị tách được bị chặn trên MAX_UNIT_WORDS — EDA đo outlier tới 189.366
    từ/đơn vị (11,5% document có >=1 đơn vị >5.000 từ) do lỗi OCR/định dạng khiến heading
    nằm giữa dòng, regex (neo ^ đầu dòng) bỏ sót. Đơn vị vượt trần bị cắt lại bằng
    _split_words_raw() thay vì giữ nguyên — tránh chunk khổng lồ làm hại precision của
    render_answer() hoặc bị cross-encoder cắt cụt ở max_length.

    `dieu_so` giữ nguyên ý nghĩa cũ (số Điều, "0" nếu không phải đơn vị Điều) để KHÔNG phá
    logic đếm/citation hiện có ở load_corpus()/build_train_pairs(). `unit_type`/`unit_no` là
    trường MỚI, dùng ở render_answer() (Cell 11) để dựng câu dẫn đúng loại đơn vị."""
    for regex, unit_type in ((DIEU_RE, "dieu"), (MUC_RE, "muc"),
                              (PHU_LUC_RE, "phu_luc"), (TIET_RE, "tiet")):
        matches = list(regex.finditer(passage))
        if matches:
            break
    else:
        matches, unit_type = [], None

    if not matches:
        words_chunks = _split_words_raw(passage, 450)
        if not words_chunks:
            return [{"id": f"{doc_id}_0", "dieu_so": "0", "unit_type": "raw", "unit_no": "",
                     "loai_vb": "", "so_hieu": "", "text": passage.strip()}]
        return [{"id": f"{doc_id}_w{i}", "dieu_so": "0", "unit_type": "raw450", "unit_no": "",
                 "loai_vb": "", "so_hieu": "", "text": t}
                for i, t in enumerate(words_chunks)]

    chunks = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(passage)
        unit_no = m.group(1)
        unit_text = passage[start:end].strip()
        if len(unit_text.split()) > MAX_UNIT_WORDS:
            for j, sub in enumerate(_split_words_raw(unit_text, 450)):
                chunks.append({"id": f"{doc_id}_{unit_type}{unit_no}_{i}_{j}",
                               "dieu_so": unit_no if unit_type == "dieu" else "0",
                               "unit_type": f"{unit_type}_capped", "unit_no": unit_no,
                               "loai_vb": "", "so_hieu": "", "text": sub})
        else:
            chunks.append({"id": f"{doc_id}_{unit_type}{unit_no}_{i}",
                           "dieu_so": unit_no if unit_type == "dieu" else "0",
                           "unit_type": unit_type, "unit_no": unit_no,
                           "loai_vb": "", "so_hieu": "", "text": unit_text})
    return chunks


def chunk_passage_words(passage: str, doc_id, n: int = 450) -> list:
    """Tầng 1: cắt 450 từ liên tục, KHÔNG quan tâm ranh giới Điều. Task 1 đo trực tiếp:
    450 từ cho document recall TỐT HƠN cắt theo Điều 1,19 điểm (p=0,024, result.md §3) —
    lý do là chunk cắt theo cấu trúc pháp lý làm mất ngữ cảnh liên Điều mà câu hỏi cần."""
    words = passage.split()
    if not words:
        return [{"id": f"{doc_id}_w0", "text": passage.strip()}]
    return [{"id": f"{doc_id}_w{i // n}", "text": " ".join(words[i:i + n])}
            for i in range(0, len(words), n)]


def load_corpus(contexts_dir) -> list:
    contexts_dir = Path(contexts_dir)
    if not contexts_dir.exists():
        raise FileNotFoundError(f"Không tìm thấy {contexts_dir} — kiểm tra lại CONTEXT_DIR ở Cell 2.")
    files = sorted(contexts_dir.glob("context_*.json"))
    if not files:
        nested = contexts_dir / "selected-contexts"
        if nested.exists():
            files = sorted(nested.glob("context_*.json"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy context_*.json trong {contexts_dir}")

    all_chunks, all_chunks_t1, n_no_dieu = [], [], 0
    dieu_by_doc = {}
    for fp in files:
        try:
            with fp.open(encoding="utf-8") as f:
                doc = json.load(f)
        except Exception:
            continue
        passage = doc.get("passage")
        if not passage:
            continue
        doc_id = doc["id"]
        loai_vb, so_hieu = extract_vb_info(passage)

        # tầng 2 (Điều) — GIỮ NGUYÊN như v3, tên biến `all_chunks` không đổi để Cell 7/8/10
        # (sinh nhãn, fine-tune encoder/reranker, đào negative cùng họ) không cần sửa gì.
        chunks = chunk_passage(passage, doc_id)
        if len(chunks) == 1 and chunks[0]["dieu_so"] == "0":
            n_no_dieu += 1
        for c in chunks:
            c["loai_vb"], c["so_hieu"] = loai_vb, so_hieu
        all_chunks.extend(chunks)
        dieu_by_doc[str(doc_id)] = chunks   # tra cứu O(1) ở tầng 2 của retrieve_two_tier()

        # tầng 1 (450 từ, MỚI) — dùng cho BM25/dense retrieval, xem Cell 6/9/11.
        for c in chunk_passage_words(passage, doc_id):
            c["loai_vb"], c["so_hieu"] = loai_vb, so_hieu
            all_chunks_t1.append(c)

    pct = round(100 * (1 - n_no_dieu / len(files)), 2) if files else 0.0
    print(f"  {len(files)} văn bản -> {len(all_chunks)} chunk tầng 2 · "
          f"{len(all_chunks_t1)} chunk 450 từ (tầng 1). {pct}% văn bản có cấu trúc Điều.")
    # BẢN SỬA v5: phân bố unit_type — soi Batch 1 (A1+A2) có hiệu lực thật hay không, đặc
    # biệt bao nhiêu document rơi vào muc/phu_luc/tiet (A1) và *_capped (A2, outlier).
    unit_type_counts = Counter(c.get("unit_type", "dieu") for c in all_chunks)
    print(f"  Phân bố unit_type (tầng 2): "
          + ", ".join(f"{k}={v}" for k, v in unit_type_counts.most_common()))
    n_capped = sum(v for k, v in unit_type_counts.items() if k.endswith("_capped"))
    if n_capped:
        print(f"  [A2] {n_capped} chunk bị chặn trên MAX_UNIT_WORDS={MAX_UNIT_WORDS} rồi cắt lại 450 từ "
              f"(outlier catastrophic — xem EDA_ANALYSIS_REPORT.md §5.1).")
    n_fallback_raw = unit_type_counts.get("raw450", 0) + unit_type_counts.get("raw", 0)
    if n_fallback_raw:
        print(f"  [A1] {n_fallback_raw} chunk fallback cắt 450 từ (không khớp Điều/Mục/Phụ lục/tiết).")
    return all_chunks, all_chunks_t1, dieu_by_doc


print("=== Bước 1: Chunk corpus (2 tầng — BẢN SỬA v4, xem Cell 2/result.md §3) ===")
all_chunks, all_chunks_t1, dieu_by_doc = load_corpus(CONTEXT_DIR)
checkpoint("Xong chunking (2 tầng)")

In [ ]:
# Cell 6: Bước 2 — BM25 tự viết bằng numpy (inverted index vector hoá — nhanh trên corpus lớn)
class BM25:
    def __init__(self, tokenized_docs, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.N = len(tokenized_docs)
        self.doc_len = np.array([len(d) for d in tokenized_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0

        raw_postings = defaultdict(list)
        for i, doc in enumerate(tokenized_docs):
            for term, f in Counter(doc).items():
                raw_postings[term].append((i, f))

        self.inverted: dict = {}
        for term, postings in raw_postings.items():
            idxs = np.fromiter((p[0] for p in postings), dtype=np.int32, count=len(postings))
            freqs = np.fromiter((p[1] for p in postings), dtype=np.float64, count=len(postings))
            self.inverted[term] = (idxs, freqs)

        df = {t: len(idxs) for t, (idxs, _f) in self.inverted.items()}
        idf_raw = {t: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}
        avg_idf = sum(idf_raw.values()) / len(idf_raw) if idf_raw else 0.0
        eps = 0.25 * avg_idf
        self.idf = {t: (v if v > 0 else eps) for t, v in idf_raw.items()}

    def get_scores(self, query_tokens) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float64)
        for term in set(query_tokens):
            posting = self.inverted.get(term)
            if posting is None:
                continue
            idxs, freqs = posting
            idf = self.idf[term]
            denom = freqs + self.k1 * (1 - self.b + self.b * self.doc_len[idxs] / self.avgdl)
            contrib = idf * freqs * (self.k1 + 1) / denom
            scores[idxs] += contrib
        return scores

    def top_k(self, query_tokens, k: int) -> list:
        scores = self.get_scores(query_tokens)
        return list(np.argsort(-scores)[:k])


print("=== Bước 2: BM25 index (2 tầng — BẢN SỬA v4) ===")
# tầng 2 (Điều) — GIỮ NGUYÊN như v3: dùng cho negative sampling ở Cell 8 (_build_training_rows)
# và Cell 10 (mine_family_negatives), CẢ HAI đều lấy positional index vào `all_chunks` nên
# `bm25` và `all_chunks` PHẢI luôn là một cặp cùng corpus — không được lẫn với bm25_t1.
tokenized = [tokenize_simple(f"{c.get('loai_vb','')} {c['text']}") for c in all_chunks]
bm25 = BM25(tokenized)
# tầng 1 (450 từ, MỚI) — dùng cho retrieve_two_tier() ở Cell 11, cặp với all_chunks_t1.
tokenized_t1 = [tokenize_simple(f"{c.get('loai_vb','')} {c['text']}") for c in all_chunks_t1]
bm25_t1 = BM25(tokenized_t1)
checkpoint("Xong BM25 index (2 tầng)")

In [ ]:
# Cell 7: Bước 3 — Sinh nhãn (question -> chunk) từ citation trong train.json (+ warmup.json
# nếu USE_WARMUP=True, cùng schema — gộp thêm dữ liệu train, KHÔNG gộp vào mẫu dev-eval để
# tránh lẫn chất lượng nhãn chưa kiểm chứng vào lúc CHỌN cấu hình cuối cùng)
def extract_citations(answer) -> list:
    # answer có thể KHÔNG phải string (đã gặp thật: warmup.json có answer kiểu list ở một số
    # câu, khác train.json toàn string) — bỏ qua câu đó thay vì crash cả pipeline.
    if not isinstance(answer, str):
        return []
    out = []
    for m in DIEU_CITATION_RE.finditer(answer):
        window = answer[m.end(): m.end() + 60]
        so_m = SO_HIEU_RE.search(window)
        if so_m and so_m.start() <= 40:
            out.append((m.group(1), so_m.group(0)))
    return out


def build_train_pairs(train_data: dict, all_chunks: list):
    so_hieu_index = {}
    for c in all_chunks:
        if c["so_hieu"] and c["dieu_so"] != "0":
            so_hieu_index.setdefault((c["dieu_so"], norm_so_hieu(c["so_hieu"])), c["id"])

    positive = {}
    n_skipped_type = 0
    for qid, item in train_data.items():
        if not isinstance(item.get("answer"), str):
            n_skipped_type += 1
            continue
        for dieu, so_hieu in extract_citations(item["answer"]):
            key = (dieu, norm_so_hieu(so_hieu))
            if key in so_hieu_index:
                positive[qid] = so_hieu_index[key]
                break
    if n_skipped_type:
        print(f"  [CẢNH BÁO] {n_skipped_type} câu có answer KHÔNG phải string -> bỏ qua khi "
              f"sinh nhãn, không tính vào positive pairs.")
    chunk_by_id = {c["id"]: c for c in all_chunks}
    return positive, chunk_by_id


print("=== Bước 3: Sinh nhãn từ train.json" + (" + warmup.json" if USE_WARMUP else "") + " ===")
with open(TRAIN_PATH, encoding="utf-8") as f:
    train_data = json.load(f)
print(f"  train.json: {len(train_data)} câu")

# warmup.json: gộp thêm CHỈ KHI USE_WARMUP=True VÀ file tồn tại VÀ đúng schema
# {qid: {"question": str, "answer": str}} — lọc chặt cả kiểu dữ liệu. KHÔNG gộp vào mẫu
# dev-eval ở Bước 6 — dev-eval chỉ dùng train.json gốc để giữ tín hiệu chọn cấu hình đáng tin
# (xem phần 2.2 trong PHAN_TICH_KY_THUAT.md). USE_WARMUP=False cho phép ablation có kiểm soát.
train_data_for_pairs = dict(train_data)
n_warmup_used = 0
if USE_WARMUP and os.path.exists(WARMUP_PATH):
    try:
        with open(WARMUP_PATH, encoding="utf-8") as f:
            warmup_data = json.load(f)
        n_bad_type = 0
        for qid, item in warmup_data.items():
            if not isinstance(item, dict):
                n_bad_type += 1
                continue
            q, a = item.get("question"), item.get("answer")
            if isinstance(q, str) and isinstance(a, str):
                train_data_for_pairs[f"warmup_{qid}"] = {"question": q, "answer": a}
                n_warmup_used += 1
            else:
                n_bad_type += 1
        print(f"  warmup.json: {len(warmup_data)} câu, {n_warmup_used} câu đúng schema -> gộp thêm"
              + (f", {n_bad_type} câu sai kiểu -> bỏ qua." if n_bad_type else ".")
              + " KHÔNG gộp vào mẫu dev-eval/Recall@k ở Bước 6.")
    except Exception as e:
        print(f"  [CẢNH BÁO] Có WARMUP_PATH nhưng đọc lỗi ({e}) -> bỏ qua, chỉ dùng train.json.")
elif USE_WARMUP:
    print(f"  USE_WARMUP=True nhưng không thấy {WARMUP_PATH} -> chỉ dùng train.json.")
else:
    print(f"  USE_WARMUP=False -> chỉ dùng train.json (bỏ qua warmup.json dù có tồn tại).")

train_positive, chunk_by_id = build_train_pairs(train_data_for_pairs, all_chunks)
print(f"  Positive pairs: {len(train_positive)}/{len(train_data_for_pairs)}")
# v6: KHÔNG còn nhãn Task 1 — chỉ nhãn citation của chính Task 2 (build_train_pairs trên).
# =============================================================================
# BẢN SỬA v4 — RÒ RỈ TRAIN/DEV-EVAL phát hiện khi rà lại pipeline (result.md, mục lỗi mới)
# =============================================================================
# Trước v4: Bước 4/5b lấy TOÀN BỘ train_positive làm dữ liệu fine-tune, còn Bước 6 (dev-eval,
# Cell 12) lấy 300 câu NGẪU NHIÊN từ TOÀN BỘ train_data — KHÔNG có bước nào loại các câu
# dev-eval khỏi tập train trước đó. Với ~3.579 câu có nhãn citation / 7.000 câu train_data,
# MỘT CÂU DEV-EVAL BẤT KỲ CÓ ~51% XÁC SUẤT ĐÃ ĐƯỢC DÙNG ĐỂ FINE-TUNE reranker/encoder — tức
# METEOR ở Bước 6 (và mọi lựa chọn suy ra từ nó: TOP_N_ANSWER, reranker nào thắng, T của hàm
# gộp) MỘT PHẦN đang đo trí nhớ, không phải khả năng tổng quát hoá. Cố định dev_ids Ở ĐÂY,
# TRƯỚC khi Bước 4 build training rows, rồi loại khỏi train_positive.
#
# train_positive_all GIỮ NGUYÊN (không loại gì) — Cell 12 dùng nó để đo Recall@k, vì đó là
# ĐÁNH GIÁ bằng nhãn thật, không phải huấn luyện, nên không có gì phải loại trừ ở đó.
random.seed(SEED)
dev_ids = random.sample(list(train_data.keys()), min(DEV_EVAL_SAMPLE_SIZE, len(train_data)))
dev_ids_set = set(dev_ids)
train_positive_all = dict(train_positive)
n_dev_had_label = sum(1 for q in dev_ids if q in train_positive)
train_positive = {qid: cid for qid, cid in train_positive.items() if qid not in dev_ids_set}
print(f"  Cố định {len(dev_ids)} câu dev-eval NGAY TỪ ĐÂY (trước Bước 4). {n_dev_had_label} "
      f"câu trong số đó có nhãn citation -> LOẠI khỏi tập fine-tune "
      f"({len(train_positive_all)} -> {len(train_positive)} positive pair khả dụng để "
      f"train), nhưng VẪN dùng nhãn gốc (train_positive_all) để đo Recall@k ở Bước 6.")

# =============================================================================
# v6_1 — CHỐT TẬP HARVEST NGAY TẠI ĐÂY, CÙNG CHỖ VÀ CÙNG SEED VỚI dev_ids
# =============================================================================
# Tập harvest dùng cho: (1) train bộ chọn LTR (Cell 13), (2) error analysis (Cell 15).
# Nó PHẢI sạch với fine-tune encoder, nếu không mọi số đo trên đó là đo trí nhớ — đúng lỗi
# mà v4 đã phải sửa một lần (khối ngay trên: 51% câu dev từng nằm trong tập fine-tune).
#
# Chỗ khéo: nhãn của LTR là **METEOR của từng ứng viên so với gold answer**, KHÔNG phải nhãn
# citation. Nên LTR train được trên chính những câu KHÔNG phân giải được citation — mà đó
# đúng là nhóm câu chưa từng đi vào fine-tune encoder (fine-tune chỉ ăn train_positive, tức
# chỉ câu CÓ citation). Hai ràng buộc gặp nhau ở đúng một chỗ:
#
#     ltr_pool = (câu KHÔNG có nhãn citation) \ dev_ids
#              = câu chưa từng vào fine-tune, và cũng không phải câu dùng để chấm
#
# Nhờ vậy điểm đo trên toàn bộ harvest KHÔNG bị thổi phồng bởi contamination encoder — khác
# hẳn con số dev 0,5709 của lượt trước, vốn lẫn cả câu model đã học. Phần lệch còn lại chỉ
# là lệch phân phối train-vs-public, không phải rò rỉ.
_no_citation = [q for q in train_data.keys()
                 if q not in train_positive_all and q not in dev_ids_set]
random.shuffle(_no_citation)          # đã seed ở Cell 4 -> tái lập được
ltr_pool = _no_citation
# dev_ids đi TRƯỚC: pha A phải chấm xong dev thì Cell 13 mới có gì để gate.
harvest_ids_all = list(dev_ids) + ltr_pool
print(f"  [v6_1] Tập harvest: {len(dev_ids)} câu dev (gate) + {len(ltr_pool)} câu không có "
      f"nhãn citation = {len(harvest_ids_all)} câu khả dụng.")
print(f"         Cả hai nhóm đều CHƯA từng vào fine-tune encoder -> METEOR đo trên đây "
      f"không bị thổi phồng bởi contamination.")
if len(ltr_pool) < LTR_MIN_GROUPS:
    print(f"  [CẢNH BÁO] chỉ {len(ltr_pool)} câu sạch cho LTR (< LTR_MIN_GROUPS="
          f"{LTR_MIN_GROUPS}) -> Cell 13 nhiều khả năng bỏ qua LTR.")

checkpoint("Xong sinh nhãn")


In [ ]:
# Cell 8: Bước 4 — Fine-tune 2 dense encoder SONG SONG THẬT trên 2 GPU riêng (subprocess)
#
# CHỦ Ý dùng subprocess (không phải threading/multiprocessing.Process kiểu fork): Cell 3 đã
# init CUDA context trong tiến trình notebook (gọi torch.cuda.get_device_properties) — fork
# SAU khi CUDA đã init là lỗi kinh điển ("Cannot re-initialize CUDA in forked subprocess").
# subprocess.Popen luôn khởi động tiến trình Python HOÀN TOÀN MỚI (tương đương spawn), mỗi
# tiến trình con tự import torch riêng, tự nhận CUDA_VISIBLE_DEVICES riêng — an toàn tuyệt
# đối, đúng pattern `run_shards` của bản gốc `run_qa.py` đầu dự án.
#
# Checkpoint ĐƯỢC LƯU lần này (khác các bản trước) — bạn cần dùng lại qua nhiều phiên Kaggle,
# và vì tiến trình con/cha là 2 process riêng, cách DUY NHẤT đưa model đã train về tiến trình
# cha là qua đĩa (`model.save_pretrained()` rồi `SentenceTransformer(path)` load lại).
import subprocess
import sys  # SỬA: sys.executable dùng để gọi WORKER_SCRIPT bên dưới — thiếu import này\n# là bug thật (Python vẫn cho phép dùng module chưa import NẾU nó tình cờ đã có trong\n# builtins/đã import ở cell khác cùng kernel session — dễ chạy "trót lọt" trong notebook\n# rồi lỗi khó hiểu khi chạy .py độc lập; luôn import tường minh module mình dùng).

print("=== Bước 4: Fine-tune 2 dense encoder song song (bge-m3 @ cuda:0, e5-large @ cuda:1) ===")

WORKER_SCRIPT = os.path.join(CACHE_DIR, "_train_encoder_worker.py")
# Worker con — fine-tune MOT SentenceTransformer tren MOT GPU, chay qua subprocess.Popen,
# nhan tham so qua argv, khong phu thuoc bien toan cuc cua notebook.
worker_code = '''
import argparse, json, os, sys, time


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--base-model", required=True)
    p.add_argument("--gpu-index", required=True)
    p.add_argument("--rows-path", required=True)
    p.add_argument("--output-dir", required=True)
    p.add_argument("--max-seq-len", type=int, default=256)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--mini-batch-size", type=int, default=16)
    p.add_argument("--time-budget-sec", type=float, required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--query-prefix", default="")
    p.add_argument("--passage-prefix", default="")
    p.add_argument("--use-lora", action="store_true")
    p.add_argument("--use-8bit-optim", action="store_true")
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    args = p.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu_index
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import random
    import numpy as np
    import torch
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        from datasets import Dataset
        from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                            SentenceTransformerTrainingArguments)
        from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
        import sentence_transformers as _st
        import accelerate as _acc
    except ImportError as e:
        print(f"[LOI IMPORT] {e}", flush=True)
        print(f"[LOI IMPORT] Ban co the dang dung sentence-transformers/accelerate qua cu -- "
              f"CachedMultipleNegativesRankingLoss can sentence-transformers >= 3.0, "
              f"accelerate >= 1.1.0. Chay:", flush=True)
        print(f'    pip install -U "sentence-transformers>=3.0" "accelerate>=1.1.0"', flush=True)
        sys.exit(1)
    st_ver = tuple(int(x) for x in _st.__version__.split(".")[:2] if x.isdigit())
    if st_ver < (3, 0):
        print(f"[PHIEN BAN CU] sentence-transformers={_st.__version__} (can >= 3.0). Chay: "
              f'pip install -U "sentence-transformers>=3.0"', flush=True)
        sys.exit(1)

    # BAN SUA (log loi that: torch.AcceleratorError OOM ngay o optimizer.step() DAU TIEN --
    # AdamW full fine-tune cho model ~568M can ~6-7GB CHI RIENG optimizer state, khong phu
    # thuoc batch size -- khong batch nao du tren GPU 4GB). LoRA: chi train 1 phan rat nho
    # tham so (dong bang phan con lai) -> optimizer state nho lai theo dung ty le do, GIAI
    # QUYET DUOC loai OOM nay ma batch-backoff khong the giai quyet.
    optim_name = "adamw_torch"
    if args.use_8bit_optim:
        try:
            import bitsandbytes  # noqa: F401
            optim_name = "adamw_bnb_8bit"
            print(f"[{args.base_model}] Dung optimizer AdamW 8-bit (bitsandbytes).", flush=True)
        except ImportError:
            print(f"[{args.base_model}] bitsandbytes khong cai duoc -> dung AdamW thuong.", flush=True)

    with open(args.rows_path, encoding="utf-8") as f:
        rows = json.load(f)
    if args.query_prefix or args.passage_prefix:
        fixed = []
        for r in rows:
            r2 = dict(r)
            r2["anchor"] = args.query_prefix + r["anchor"]
            for k in r:
                if k.startswith("positive") or k.startswith("negative"):
                    r2[k] = args.passage_prefix + r[k]
            fixed.append(r2)
        rows = fixed
    dataset = Dataset.from_list(rows)

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(args.base_model, device=device)
    model.max_seq_length = args.max_seq_len

    lora_used = False
    if args.use_lora:
        try:
            from peft import LoraConfig, get_peft_model
            lora_cfg = LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha,
                                   lora_dropout=args.lora_dropout, bias="none",
                                   target_modules="all-linear")
            model[0].auto_model = get_peft_model(model[0].auto_model, lora_cfg)
            lora_used = True
            n_trainable = sum(pp.numel() for pp in model[0].auto_model.parameters() if pp.requires_grad)
            n_total = sum(pp.numel() for pp in model[0].auto_model.parameters())
            print(f"[{args.base_model}] LoRA bat: {n_trainable}/{n_total} tham so co the train "
                  f"({100*n_trainable/max(n_total,1):.2f}%)", flush=True)
        except Exception as e:
            print(f"[{args.base_model}] LoRA loi ({e}) -> full fine-tune (can nhieu VRAM hon).", flush=True)

    batch_size, mini_batch_size = args.batch_size, args.mini_batch_size
    max_steps, calib_time = 0, None
    t0 = time.time()
    for attempt in range(4):
        try:
            loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=mini_batch_size)
            calib_steps = min(10, max(1, len(dataset) // batch_size))
            calib_args = SentenceTransformerTrainingArguments(
                output_dir=args.output_dir + "_tmp", max_steps=calib_steps,
                per_device_train_batch_size=batch_size, logging_steps=calib_steps + 1,
                save_strategy="no", report_to=[], disable_tqdm=True, fp16=(device == "cuda:0"),
                optim=optim_name)
            c0 = time.time()
            print(f"[{args.base_model}] calib training (batch={batch_size}, mini_batch={mini_batch_size})...", flush=True)
            SentenceTransformerTrainer(model=model, args=calib_args, train_dataset=dataset, loss=loss).train()
            calib_time = (time.time() - c0) / calib_steps

            budget_left = args.time_budget_sec - (time.time() - t0) - 60
            max_steps = max(0, int(budget_left / max(calib_time, 1e-6)))
            max_steps = min(max_steps, (len(dataset) // batch_size) * 8)
            print(f"[{args.base_model}] calib {calib_time:.2f}s/step, ngan sach con "
                  f"{budget_left/60:.1f} phut -> {max_steps} step", flush=True)

            if max_steps > 0:
                targs = SentenceTransformerTrainingArguments(
                    output_dir=args.output_dir + "_tmp", max_steps=max_steps,
                    per_device_train_batch_size=batch_size, learning_rate=2e-5,
                    warmup_steps=0.05, lr_scheduler_type="cosine",
                    logging_steps=max(1, max_steps // 20), save_strategy="no", report_to=[],
                    fp16=(device == "cuda:0"), optim=optim_name)
                SentenceTransformerTrainer(model=model, args=targs, train_dataset=dataset, loss=loss).train()
            break
        except Exception as e:
            if "out of memory" in str(e).lower() and mini_batch_size > 1:
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass  # cache da can kiet toi muc khong con gi de don -- bo qua, cu lui batch
                mini_batch_size = max(1, mini_batch_size // 2)
                print(f"[{args.base_model}] OOM -> mini_batch_size={mini_batch_size}", flush=True)
                continue
            raise

    if lora_used:
        try:
            model[0].auto_model = model[0].auto_model.merge_and_unload()
            print(f"[{args.base_model}] Da merge LoRA vao model goc.", flush=True)
        except Exception as e:
            print(f"[{args.base_model}] Merge LoRA loi ({e}) -> luu adapter rieng.", flush=True)

    model.save_pretrained(args.output_dir)
    meta = {"max_steps": max_steps, "mini_batch_final": mini_batch_size,
            "calib_time_s": calib_time, "elapsed_s": time.time() - t0,
            "lora_used": lora_used, "optim": optim_name}
    with open(args.output_dir + "_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f)
    print(f"[{args.base_model}] DONE -> {args.output_dir}", flush=True)


if __name__ == "__main__":
    main()
'''
with open(WORKER_SCRIPT, "w", encoding="utf-8") as f:
    f.write(worker_code)

# ---- Tạo training rows 1 LẦN trong tiến trình cha (dùng chung cho cả 2 encoder) ----
def _build_training_rows(train_positive, train_data, chunk_by_id, all_chunks, bm25, n_neg=N_NEG_PER_ROW):
    rows = []
    n = len(train_positive)
    for i, (qid, pos_id) in enumerate(train_positive.items()):
        question = train_data[qid]["question"]
        pos_text = chunk_by_id[pos_id]["text"]
        token_q = tokenize_simple(question)
        ranked = bm25.top_k(token_q, 60)
        neg_ids = [all_chunks[i2]["id"] for i2 in ranked[5:60] if all_chunks[i2]["id"] != pos_id][:n_neg]
        if len(neg_ids) < n_neg:
            pool = [c["id"] for c in all_chunks if c["id"] != pos_id]
            while len(neg_ids) < n_neg and pool:
                neg_ids.append(random.choice(pool))
        # _pos_id/_qid: metadata cho tầng đào negative "cùng họ" ở Cell 10. Tiền tố "_"
        # để phân biệt với các khoá anchor/positive/negative_* mà trainer thật sự đọc.
        row = {"anchor": question, "positive": pos_text, "_pos_id": pos_id, "_qid": qid}
        for j, nid in enumerate(neg_ids[:n_neg]):
            row[f"negative_{j+1}"] = chunk_by_id[nid]["text"]
        rows.append(row)
        if (i + 1) % 500 == 0 or (i + 1) == n:
            print(f"    _build_training_rows: {i+1}/{n}  ({elapsed()/60:.1f} phút)")
    return rows


finetune_info = {"used_finetune": False, "reason": None, "n_pairs_available": len(train_positive),
                  "n_pairs_used": 0, "models": {}}
DENSE_CHANNELS = []  # điền ở cuối cell; embeddings điền ở Cell 9

# BẢN SỬA: build `rows` (anchor/positive/negative) MỘT LẦN, KHÔNG PHỤ THUỘC USE_FINETUNE của
# dense encoder — Cell 10 (fine-tune reranker) cần dùng lại đúng `rows` này. Trước đây rows chỉ
# được build bên trong nhánh "if use_finetune" của dense encoder, nên nếu USE_FINETUNE=False thì
# Cell 10 không có gì để fine-tune reranker dù USE_RERANKER_FINETUNE=True.
#
# BẢN SỬA THÊM: tách riêng `rows_clean` (CHỈ nhãn citation, chính xác tới từng Điều) khỏi
# `rows` (citation + Task 1) — nhãn Task 1 là GIÁM SÁT YẾU ở mức Điều (chọn Điều điểm BM25 cao
# nhất TRONG đúng văn bản gold — có thể sai Điều dù đúng văn bản, xem docstring
# build_task1_pairs() ở Cell 7). Dense encoder train bằng contrastive loss với nhiều negative,
# chịu nhiễu nhãn tốt — dùng cả 2 nguồn (`rows`) là hợp lý. Reranker train bằng margin ranking
# loss trên 1 cặp positive/negative mỗi lần, KHÔNG có gì làm mềm nhiễu — lỡ học "Điều sai nhưng
# đúng văn bản" thành positive thật sẽ kéo NGƯỢC độ chính xác, đúng kiểu lỗi khớp với quan sát
# thật: thêm nhãn Task 1 làm METEOR tăng mạnh (tìm đúng NHIỀU câu hỏi hơn) nhưng ROUGE-L gần
# như đứng yên (không chính xác hơn ở mức từng chữ) — nghi vấn hợp lý là do một phần nhãn Điều
# không hoàn toàn đúng đang lẫn vào huấn luyện. Reranker vì vậy CHỈ train trên `rows_clean`.
rows_needed = (USE_FINETUNE or USE_RERANKER_FINETUNE) and len(train_positive) >= MIN_TRAIN_PAIRS \
              and remaining() > 10 * 60
rows, rows_path = [], None
rows_clean = []
if rows_needed:
    train_positive_used = train_positive
    if len(train_positive) > MAX_TRAIN_EXAMPLES:
        sampled_qids = random.sample(list(train_positive.keys()), MAX_TRAIN_EXAMPLES)
        train_positive_used = {qid: train_positive[qid] for qid in sampled_qids}
        print(f"  Có {len(train_positive)} positive pairs, lấy mẫu {MAX_TRAIN_EXAMPLES} "
              f"(tái lập được nhờ SEED={SEED}).")
    finetune_info["n_pairs_used"] = len(train_positive_used)

    print(f"  Đang tạo training rows (dense encoder — citation + Task 1)...")
    rows = _build_training_rows(train_positive_used, train_data_for_pairs, chunk_by_id, all_chunks, bm25)
    rows_path = os.path.join(CACHE_DIR, "train_rows.json")
    with open(rows_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    print(f"  {len(rows)} rows -> {rows_path}")

    # BẢN SỬA v6_1: `rows_clean` CHỈ phục vụ fine-tune reranker, mà v6_1 đã tắt hẳn nhánh đó
    # (xem khối lý do ở Cell 2). Lượt build thứ hai này tốn 3,4 phút trong lượt trước và cho
    # ra một mảng không ai đọc. Chỉ build khi cờ thật sự bật.
    if USE_RERANKER_FINETUNE:
        clean_positive = {qid: cid for qid, cid in train_positive_used.items()
                           if not str(qid).startswith("task1_")}
        if clean_positive:
            print(f"  Đang tạo training rows (reranker — CHỈ citation, {len(clean_positive)} câu)...")
            rows_clean = _build_training_rows(clean_positive, train_data_for_pairs, chunk_by_id,
                                               all_chunks, bm25)
            print(f"  {len(rows_clean)} rows_clean (reranker)")
    else:
        print(f"  USE_RERANKER_FINETUNE=False -> BỎ QUA lượt build rows_clean thứ hai "
              f"(tiết kiệm ~3,4 phút, đo từ lượt trước).")

use_finetune = USE_FINETUNE and bool(rows)
if not use_finetune:
    reason = ("USE_FINETUNE=False" if not USE_FINETUNE else
               ("chưa có rows (xem lý do rows_needed=False ở trên)" if not rows else "?"))
    print(f"  {reason} -> dùng zero-shot cho cả 2 dense encoder, không fine-tune.")
    finetune_info["reason"] = reason
    from sentence_transformers import SentenceTransformer
    # Nạp có ĐƯỜNG LÙI. Cặp encoder mới (vnembv2 + harrier) là chuyển giao từ Task 1 và
    # CHƯA từng chạy trên Kaggle: có thể thiếu quyền tải, thiếu trust_remote_code, hoặc
    # kiến trúc sentence-transformers không nhận. Hỏng ở đây mà không có đường lùi là mất
    # trắng một phiên GPU 6 tiếng, nên thử -> lỗi thì quay về cặp cũ đã cho 0,5528 và NÓI TO.
    def _load(name, dev):
        return SentenceTransformer(name, device=dev, trust_remote_code=True)

    try:
        m_a = _load(BASE_DENSE_MODEL_A, DEVICES[0])
        m_b = _load(BASE_DENSE_MODEL_B, DEVICES[-1])
    except Exception as e:
        print(f"  ⛔ Không nạp được cặp encoder mới ({type(e).__name__}: {e})")
        print(f"     -> LÙI VỀ bge-m3 + e5-large (cấu hình đã cho 0,5528). "
              f"Mọi so sánh với bản mới KHÔNG còn hiệu lực.")
        BASE_DENSE_MODEL_A, BASE_DENSE_MODEL_B = "BAAI/bge-m3", "intfloat/multilingual-e5-large"
        QUERY_PREFIX_A, QUERY_PREFIX_B, PASSAGE_PREFIX_B = "", "query: ", "passage: "
        m_a = _load(BASE_DENSE_MODEL_A, DEVICES[0])
        m_b = _load(BASE_DENSE_MODEL_B, DEVICES[-1])
    m_a.max_seq_length = DENSE_MAX_SEQ_LEN
    m_b.max_seq_length = DENSE_MAX_SEQ_LEN
    DENSE_CHANNELS = [
        {"name": "A", "model": m_a, "embeddings": None,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": "B", "model": m_b, "embeddings": None,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
else:
    specs = [
        # BẢN SỬA: tên slot suy TỪ MODEL THẬT, không viết cứng. Bản cũ luôn ghi
        # "bge-m3"/"e5-large" vào log dù USE_NEW_ENCODERS=True đang nạp model khác —
        # khiến log ver9 không thể quy kết, và chính tôi đã đọc sai log vì lý do đó.
        {"name": BASE_DENSE_MODEL_A.split("/")[-1], "base_model": BASE_DENSE_MODEL_A,
         "gpu": DEVICES[0].split(":")[-1],
         "out": os.path.join(CHECKPOINT_DIR, "A-ft"), "mini_batch": TRAIN_MINI_BATCH_SIZE,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": BASE_DENSE_MODEL_B.split("/")[-1], "base_model": BASE_DENSE_MODEL_B,
         "gpu": DEVICES[-1].split(":")[-1],
         "out": os.path.join(CHECKPOINT_DIR, "B-ft"), "mini_batch": TRAIN_MINI_BATCH_SIZE_B,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
    # Chỉ chạy THẬT SỰ song song nếu có >= 2 GPU riêng biệt cho 2 spec — nếu chỉ 1 GPU, cả 2
    # subprocess sẽ tranh cùng 1 thẻ nếu phóng cùng lúc (dễ OOM cả hai) -> chạy TUẦN TỰ.
    run_parallel = len(DEVICES) > 1 and specs[0]["gpu"] != specs[1]["gpu"]
    time_budget_each = max(600.0, min(remaining() - 5 * 60, FINETUNE_TIME_BUDGET_SEC)
                            / (1.0 if run_parallel else 2.0))
    print(f"  Chạy {'SONG SONG (2 GPU riêng)' if run_parallel else 'TUẦN TỰ (chỉ 1 GPU khả dụng)'} "
          f"— ngân sách mỗi encoder ~{time_budget_each/60:.0f} phút.")

    def _launch(spec):
        log_path = os.path.join(CACHE_DIR, f"train_{spec['name']}.log")
        cmd = [sys.executable, WORKER_SCRIPT,
               "--base-model", spec["base_model"], "--gpu-index", spec["gpu"],
               "--rows-path", rows_path, "--output-dir", spec["out"],
               "--max-seq-len", str(DENSE_MAX_SEQ_LEN), "--batch-size", str(TRAIN_BATCH_SIZE),
               "--mini-batch-size", str(spec.get("mini_batch", TRAIN_MINI_BATCH_SIZE)), "--time-budget-sec", str(time_budget_each),
               "--seed", str(SEED), "--query-prefix", spec["query_prefix"], "--passage-prefix", spec["passage_prefix"],
               "--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA), "--lora-dropout", str(LORA_DROPOUT)]
        if USE_LORA:
            cmd.append("--use-lora")
        if USE_8BIT_OPTIM:
            cmd.append("--use-8bit-optim")
        lf = open(log_path, "w")
        print(f"  Khởi động fine-tune {spec['name']} trên GPU {spec['gpu']} -> log: {log_path}")
        return subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT), lf

    failed = []
    if run_parallel:
        procs = [(spec, *_launch(spec)) for spec in specs]
        print(f"  Đang chờ {len(procs)} tiến trình fine-tune song song...", flush=True)
        for spec, proc, lf in procs:
            rc = proc.wait()
            print(f"  {spec['name']}: xong, mã thoát {rc}")
            if rc != 0:
                failed.append(spec["name"])
            lf.close()
    else:
        for spec in specs:
            proc, lf = _launch(spec)
            rc = proc.wait()
            print(f"  {spec['name']}: xong, mã thoát {rc}")
            if rc != 0:
                failed.append(spec["name"])
            lf.close()
    if failed:
        raise SystemExit(f"Fine-tune lỗi: {failed} — xem log trong {CACHE_DIR}/train_<tên>.log")

    from sentence_transformers import SentenceTransformer
    for spec in specs:
        meta_path = spec["out"] + "_meta.json"
        with open(meta_path, encoding="utf-8") as f:
            m = json.load(f)
        finetune_info["models"][spec["name"]] = m
        print(f"  {spec['name']}: {m['max_steps']} step, mini_batch cuối={m['mini_batch_final']}, "
              f"{m['elapsed_s']/60:.1f} phút")

    m_a = SentenceTransformer(specs[0]["out"], device=DEVICES[0])
    m_b = SentenceTransformer(specs[1]["out"], device=DEVICES[-1])
    DENSE_CHANNELS = [
        {"name": "A", "model": m_a, "embeddings": None,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": "B", "model": m_b, "embeddings": None,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
    finetune_info["used_finetune"] = True
    print(f"  Checkpoint đã lưu trong {CHECKPOINT_DIR} — tự tải về nếu muốn dùng lại phiên sau.")

checkpoint("Xong Bước 4 (2 dense encoder)")


In [ ]:
# Cell 9 [v6_2]: Bước 5 — Encode toàn bộ corpus CHO CẢ 2 ENCODER
#
# v6_2 — ENCODE FP16 CÓ CỔNG KIỂM TRA. Lượt v6_1 encode fp32 mất 185 phút (29% phiên), trong
# khi nhánh 1-GPU của chính cell này vốn đã .half(). Nhánh 2-GPU (multi-process pool) chưa
# bao giờ ép fp16. Tiết kiệm thời gian ở đây là thứ trả cho fine-tune reranker ở Cell 10.
#
# Rủi ro có thật: harrier là họ Qwen3, mà Qwen dễ tràn số (inf/NaN) ở fp16. Nên KHÔNG ép mù:
# encode một mẫu cả fp32 lẫn fp16, chỉ dùng fp16 khi không có NaN VÀ cosine nhỏ nhất giữa hai
# bản >= FP16_MIN_COS. Trượt cổng -> kênh đó quay về fp32 như v6_1, không có gì hỏng.
print(f"=== Bước 5 [v6_2]: Encode toàn bộ corpus (tầng 1, 450 từ) cho {len(DENSE_CHANNELS)} encoder ===")
texts_raw = [c["text"] for c in all_chunks_t1]
encode_info = {}


def _fp16_guard(model, sample_texts, device):
    """-> (dùng_fp16, thông_tin). Để model ở fp16 nếu qua cổng, fp32 nếu trượt."""
    dtype0 = next(model.parameters()).dtype
    if dtype0 == torch.float16:
        return True, {"note": "checkpoint đã là fp16 sẵn"}
    model.to(device)
    ref = model.encode(sample_texts, batch_size=32, convert_to_numpy=True,
                       normalize_embeddings=True, device=device)
    model.half()
    try:
        emb = model.encode(sample_texts, batch_size=32, convert_to_numpy=True,
                           normalize_embeddings=True, device=device)
    except Exception as e:
        model.float()
        return False, {"error": f"{type(e).__name__}: {e}"}
    finite = bool(np.isfinite(emb).all())
    cos = (ref.astype(np.float64) * emb.astype(np.float64)).sum(1) if finite else np.array([0.0])
    info = {"finite": finite, "min_cos": round(float(cos.min()), 5),
            "mean_cos": round(float(cos.mean()), 5), "n_sample": len(sample_texts)}
    ok = finite and float(cos.min()) >= FP16_MIN_COS
    if not ok:
        model.float()
    return ok, info


_rng = random.Random(SEED)
_by_len = sorted(range(len(texts_raw)), key=lambda i: -len(texts_raw[i]))
# Nửa mẫu là chunk DÀI nhất (tràn số hay xảy ra ở chuỗi dài), nửa ngẫu nhiên.
_guard_idx = _by_len[: FP16_GUARD_N // 2] + _rng.sample(range(len(texts_raw)), FP16_GUARD_N // 2)

for ch in DENSE_CHANNELS:
    t0 = time.time()
    texts = [ch["passage_prefix"] + t for t in texts_raw] if ch["passage_prefix"] else texts_raw
    model = ch["model"]
    dev0 = str(next(model.parameters()).device)
    use_fp16, ginfo = False, {"skipped": "ENCODE_FP16=False hoặc không có CUDA"}
    if ENCODE_FP16 and dev0.startswith("cuda"):
        use_fp16, ginfo = _fp16_guard(model, [texts[i] for i in _guard_idx], dev0)
    encode_info[ch["name"]] = {"fp16": use_fp16, **ginfo}
    print(f"  [{ch['name']}] cổng fp16: {'QUA' if use_fp16 else 'TRƯỢT -> fp32'} {ginfo}")
    print(f"  [{ch['name']}] encode {len(texts)} chunk ({'fp16' if use_fp16 else 'fp32'})"
          + (f' (tiền tố "{ch["passage_prefix"]}")' if ch["passage_prefix"] else "") + " ...")
    if len(DEVICES) > 1 and DEVICES[0].startswith("cuda"):
        # Pool spawn nhận bản pickle của model -> dtype hiện tại (fp16 nếu qua cổng) đi theo.
        pool = model.start_multi_process_pool(target_devices=DEVICES)
        try:
            emb = model.encode_multi_process(texts, pool, batch_size=ENCODE_BATCH_SIZE,
                                              normalize_embeddings=True)
        finally:
            model.stop_multi_process_pool(pool)
    else:
        device = DEVICES[0]
        model = model.to(device)
        if device.startswith("cuda") and use_fp16:
            model = model.half()
        batch_size = ENCODE_BATCH_SIZE
        while True:
            try:
                emb = model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                                    show_progress_bar=True, normalize_embeddings=True, device=device)
                break
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and batch_size > 1:
                    print(f"    [CUDA OOM] batch_size={batch_size} -> thử {batch_size // 2}")
                    torch.cuda.empty_cache()
                    batch_size = max(1, batch_size // 2)
                    continue
                raise
        ch["model"] = model
    if not np.isfinite(emb).all():
        raise SystemExit(f"[{ch['name']}] embedding corpus có NaN/inf — dừng thay vì retrieval rác.")
    ch["embeddings"] = emb
    encode_info[ch["name"]]["encode_min"] = round((time.time() - t0) / 60, 1)
    print(f"    -> {emb.shape}, {time.time()-t0:.0f}s")

checkpoint("Xong encode corpus (2 encoder)")


In [ ]:
# Cell 10: Bước 5b — Fine-tune reranker (nếu USE_RERANKER_FINETUNE, tái dùng `rows_clean`
# CHỈ nhãn citation của Bước 4 — KHÔNG dùng nhãn Task 1, xem giải thích ở Cell 8) RỒI tải
# 1 bản MỖI GPU để rerank song song thật ở Bước 6/7 (xem Cell 11)
#
# LÝ DO: kết quả dual-encoder thật (0.5215/0.4829) chỉ nhích rất ít so với single-encoder
# (0.5199/0.4806) dù retrieval mạnh hơn nhiều -> retrieval không còn là nút thắt chính,
# reranker ZERO-SHOT (AITeamVN/Vietnamese_Reranker, train trên Legal Zalo 2021 — KHÔNG phải
# đúng format "Điều X" của bài này) nhiều khả năng đang là trần chặn điểm tiếp theo.
#
# Huấn luyện bằng vòng lặp PyTorch thuần (KHÔNG dùng CrossEncoderTrainer của
# sentence-transformers) — tránh phụ thuộc API cross-encoder mới có thể không có ở mọi
# phiên bản cài qua Cell 1; margin ranking loss (điểm(positive) phải lớn hơn điểm(negative)
# ít nhất RERANKER_FT_MARGIN) — không cần thang điểm chuẩn hoá, chỉ cần đúng THỨ TỰ, ổn
# định hơn BCE/MSE cho một đầu hồi quy logit thô như model này.
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_reranker_on(device: str, source: str):
    for attempt in range(2):
        try:
            print(f"  Đang tải reranker {source} lên {device}"
                  f"{' — thử lại lần 2' if attempt else ''}...")
            tok = AutoTokenizer.from_pretrained(source)
            mdl = AutoModelForSequenceClassification.from_pretrained(source)
            mdl = mdl.to(device)
            if device.startswith("cuda"):
                mdl = mdl.half()
            mdl.eval()
            return mdl, tok
        except Exception as e:
            if attempt == 0:
                print(f"  [Lần 1 lỗi: {e}] thử lại sau 5s...")
                time.sleep(5)
                continue
            print(f"  [CẢNH BÁO] Không tải được reranker trên {device} ({e}) -> bỏ qua "
                  f"reranker trên thẻ này.")
            return None, None


# finetune_reranker [v6_2] — CÔNG THỨC TASK 1 (eval/train_reranker.py + eval/job_family.sh).
#
# Soi code v6_1 thấy vòng lặp cũ LỆCH công thức Task 1 ở bốn chỗ, độc lập với bug chốt chặn
# false-negative đã sửa:
#   · lr 3e-6 áp thẳng lên model GỐC — Task 1 dùng 3e-6 để train TIẾP từ checkpoint vòng 2,
#     còn train từ gốc là 1e-5;
#   · không warmup, không lịch lr;
#   · batch 8 câu/forward -> OOM -> tụt về 1 câu/bước (log v6: 7.494 bước ở batch 1), tức
#     batch hiệu dụng 1 thay vì 8;
#   · không gradient checkpointing, không đóng băng embedding, không clip gradient.
# Bản dưới đây chép đúng các lựa chọn đó: 1 nhóm/forward, tích luỹ 8 nhóm/bước cập nhật,
# warmup 10% + giảm tuyến tính, clip 1,0, weight decay 0,01, max_length 512, listwise CE.
# Chỉ PHƯƠNG PHÁP được chuyển giao; dữ liệu vẫn là nhãn citation của Task 2 (rows_clean).
def finetune_reranker(rows, base_model: str, device: str, time_budget_sec: float, lr: float,
                       seed: int, n_neg: int, accum: int = 8, epochs: int = 2,
                       warmup: float = 0.1, max_length: int = 512, use_8bit_optim: bool = False,
                       log_every: int = 25):
    import torch.nn.functional as F
    cuda = device.startswith("cuda")
    tok = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=1).to(device)
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    for p_ in model.get_input_embeddings().parameters():
        p_.requires_grad = False
    params = [p_ for p_ in model.parameters() if p_.requires_grad]
    opt = None
    if use_8bit_optim:
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, lr=lr, weight_decay=0.01)
        except Exception as e:
            print(f"  [reranker-ft] AdamW 8-bit không dùng được ({e}) -> AdamW thường")
    if opt is None:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=cuda)

    g = random.Random(seed)
    total_groups = len(rows) * epochs
    planned_updates = max(1, total_groups // accum)
    CALIB_GROUPS = 40

    def _lr_at(u):
        w = max(1, int(planned_updates * warmup))
        if u < w:
            return lr * (u + 1) / w
        return lr * max(0.0, (planned_updates - u) / max(1, planned_updates - w))

    model.train()
    t0, it, updates, run_loss, run_acc, seen = time.time(), 0, 0, 0.0, 0.0, 0
    stop_reason = "hết số epoch dự kiến"
    loss_log = []
    for ep in range(epochs):
        order = list(range(len(rows)))
        g.shuffle(order)
        for j in order:
            r = rows[j]
            texts = [r["positive"]] + [r[f"negative_{k+1}"] for k in range(n_neg)]
            enc = tok([[r["anchor"], t] for t in texts], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
                logits = model(**enc).logits.view(1, -1)
            logits = logits.float()
            loss = F.cross_entropy(logits, torch.zeros(1, dtype=torch.long, device=device))
            scaler.scale(loss / accum).backward()
            run_loss += loss.item()
            run_acc += float(int(torch.argmax(logits, dim=1).item() == 0))
            seen += 1
            it += 1
            if it == CALIB_GROUPS:
                # Hiệu chỉnh số bước theo tốc độ ĐO THẬT (cùng ý với worker encoder ở Cell 8):
                # lịch lr phải kết thúc đúng lúc hết ngân sách, không phải bị cắt giữa chừng
                # khi lr còn cao.
                rate = (time.time() - t0) / it
                fit_groups = int(0.95 * time_budget_sec / max(rate, 1e-6))
                planned_updates = max(1, min(total_groups, fit_groups) // accum)
                print(f"    [reranker-ft] {rate:.2f} giây/nhóm -> {planned_updates} bước cập nhật "
                      f"(tối đa {total_groups // accum} nếu đủ {epochs} epoch)", flush=True)
            if it % accum == 0:
                for pg in opt.param_groups:
                    pg["lr"] = _lr_at(updates)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                updates += 1
                if updates % log_every == 0:
                    loss_log.append({"update": updates, "loss": round(run_loss / seen, 4),
                                     "acc_in_group": round(run_acc / seen, 4),
                                     "min": round((time.time() - t0) / 60, 1)})
                    print(f"    reranker-ft ep {ep} bước {updates}/{planned_updates} · "
                          f"loss {run_loss/seen:.4f} · acc@1 trong nhóm {run_acc/seen:.3f} · "
                          f"lr {_lr_at(updates):.2e} · {(time.time()-t0)/60:.1f} phút", flush=True)
                    run_loss, run_acc, seen = 0.0, 0.0, 0
                if updates >= planned_updates:
                    stop_reason = "đủ số bước theo ngân sách"
                    break
                if time.time() - t0 > time_budget_sec:
                    stop_reason = "chạm trần thời gian"
                    break
        else:
            continue
        break
    opt.zero_grad(set_to_none=True)
    model.eval()
    meta = {"updates": updates, "groups_seen": it, "planned_updates": planned_updates,
            "elapsed_s": round(time.time() - t0, 1), "stop_reason": stop_reason,
            "lr": lr, "accum": accum, "max_length": max_length, "n_neg": n_neg,
            "n_rows": len(rows), "loss_log": loss_log}
    return model, tok, meta


# =============================================================================
# ĐÀO NEGATIVE "CÙNG HỌ" — chuyển giao PHƯƠNG PHÁP từ Task 1 (LegalIR)
# =============================================================================
# ⚖️ Vì sao được phép: rule.md §7c(b) cấm dùng **dữ liệu** (cặp câu hỏi–gold) của Task 1
#    để huấn luyện Task 2, và cấm cả checkpoint fine-tune bằng nhãn Task 1. Nó KHÔNG cấm
#    thuật toán. Ở đây ta chỉ mượn CÁCH ĐÀO NEGATIVE, còn nhãn thì lấy từ citation trong
#    answer của chính Task 2, model xuất phát là bản pretrain công khai. Không có một bit
#    dữ liệu Task 1 nào đi vào.
#
# Task 1 đo được (result.md §4): negative "cùng họ" 0,9428 · negative ngữ nghĩa 0,9360 ·
# negative hợp nhất 0,9357. Càng nhiều negative ngữ nghĩa điểm càng thấp. "Cùng họ" =
# văn bản cùng chủ đề/cùng series nhưng KHÔNG phải gold — negative khó nhất có thể có.
#
# Bản dịch sang Task 2: đơn vị ứng viên là ĐIỀU, nên "cùng họ" tự nhiên nhất là
# **các Điều ANH EM trong CÙNG văn bản gold**. Đó đúng là chỗ pipeline đang thua: đo trên
# 501 câu dev, reranker chọn đúng Điều tốt nhất trong 5 ứng viên chỉ 61,3% số lần, và
# khoảng cách điểm giữa hạng 1 và hạng 2 có trung vị 0,0023 — nó gần như không phân biệt
# nổi hai ứng viên đầu. Dạy đúng vào chỗ đó là dạy đúng chỗ đau.
#
# CHỐT CHẶN BẮT BUỘC (Task 1 gọi là "bỏ ứng viên CE chấm cao hơn gold"): văn bản/Điều cùng
# họ RẤT dễ là false negative — nó cũng trả lời được câu hỏi mà nhãn không ghi. Nhãn của
# Task 2 lại lấy citation ĐẦU TIÊN gặp trong answer, càng dễ trượt. Nếu không lọc, ta đang
# dạy model DÌM đáp án đúng xuống. Đây nhiều khả năng là nguyên nhân recall@1 sập từ 0,6503
# (không rerank) xuống 0,4266 (rerank fine-tune) trong log ver9.
def mine_family_negatives(rows, all_chunks, chunk_by_id, bm25, base_model, device,
                           n_neg=7, n_sibling=12, gate=True):
    """rows -> rows mới với negative_1..n_neg là Điều 'cùng họ' đã qua chốt chặn."""
    from collections import defaultdict
    import torch
    by_doc = defaultdict(list)
    for c in all_chunks:
        by_doc[str(c["id"]).split("_")[0]].append(c["id"])

    cand_of = {}
    for idx, r in enumerate(rows):
        pos_id = r.get("_pos_id")
        if not pos_id:
            continue
        doc = str(pos_id).split("_")[0]
        sib = [cid for cid in by_doc.get(doc, []) if cid != pos_id][:n_sibling]
        # Đổ thêm bằng BM25 để đủ số lượng khi văn bản gold chỉ có 1-2 Điều.
        if len(sib) < n_neg * 2:
            ranked = bm25.top_k(tokenize_simple(r["anchor"]), 60)
            extra = [all_chunks[i]["id"] for i in ranked
                      if all_chunks[i]["id"] != pos_id and all_chunks[i]["id"] not in sib]
            sib = sib + extra[: n_neg * 2 - len(sib)]
        if sib:
            cand_of[idx] = sib

    if not cand_of:
        print("  [mine] không đào được ứng viên nào -> giữ nguyên negative cũ")
        return rows

    kept = 0
    if gate:
        # Chấm bằng reranker ZERO-SHOT (chưa fine-tune) — đúng tinh thần Task 1: dùng một
        # bộ chấm ĐỘC LẬP với model sắp train để phát hiện positive chưa gán nhãn.
        # ⛔ BUG v6 ĐÃ SỬA Ở ĐÂY (v6_log.txt dòng 82-92). Bản v6 viết:
        #       load_reranker_on(device, "zero-shot")
        # tức truyền CHUỖI MÔ TẢ "zero-shot" vào tham số đợi TÊN MODEL. HuggingFace không có
        # repo nào tên "zero-shot" -> load_reranker_on trả (None, None) -> nhánh dưới in
        # "bỏ chốt chặn, RỦI RO CAO" -> toàn bộ negative đi thẳng vào train KHÔNG qua lọc
        # false-negative. Tham số `base_model` đã nằm sẵn trong chữ ký hàm và chính là thứ
        # cần truyền — lỗi chỉ là không dùng nó.
        #
        # Hậu quả đã đo được: reranker fine-tune trên negative bẩn thua zero-shot 3,3 điểm
        # METEOR (0,5382 vs 0,5709). Đúng cơ chế result.md §8.3 cảnh báo: không lọc tức là
        # dạy model dìm chính đáp án đúng xuống.
        model, tok = load_reranker_on(device, base_model)
        if model is None:
            print("  [mine] không tải được reranker để lọc -> BỎ HẲN fine-tune thay vì train "
                  "trên negative chưa lọc (bản v6 đi tiếp và mất 110 phút cho một checkpoint "
                  "thua zero-shot).")
            raise RuntimeError("không tải được reranker cho chốt chặn false-negative")
        else:
            pairs, index = [], []
            for idx, sib in cand_of.items():
                pairs.append([rows[idx]["anchor"], rows[idx]["positive"]])
                index.append((idx, None))
                for cid in sib:
                    pairs.append([rows[idx]["anchor"], chunk_by_id[cid]["text"]])
                    index.append((idx, cid))
            print(f"  [mine] chấm {len(pairs)} cặp để lọc false negative ...")
            scores = []
            with torch.no_grad():
                for i in range(0, len(pairs), RERANK_SUBBATCH):
                    enc = tok(pairs[i:i + RERANK_SUBBATCH], padding=True, truncation=True,
                              max_length=512, return_tensors="pt").to(device)
                    scores.extend(model(**enc).logits.view(-1).float().cpu().tolist())
            pos_score, cand_score = {}, defaultdict(list)
            for (idx, cid), sc in zip(index, scores):
                if cid is None:
                    pos_score[idx] = sc
                else:
                    cand_score[idx].append((sc, cid))
            for idx in list(cand_of):
                survive = [(sc, cid) for sc, cid in cand_score.get(idx, [])
                            if sc < pos_score.get(idx, float("inf"))]
                survive.sort(reverse=True)          # khó nhất trước
                cand_of[idx] = [cid for _sc, cid in survive]
                kept += len(survive)
            del model
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

    out, n_drop = [], 0
    for idx, r in enumerate(rows):
        sib = cand_of.get(idx, [])
        if len(sib) < n_neg:
            # Không đủ negative sau khi lọc -> giữ row nhưng đệm bằng negative cũ nếu có.
            old = [r[k] for k in r if k.startswith("negative_")]
            texts = [chunk_by_id[c]["text"] for c in sib] + old
        else:
            texts = [chunk_by_id[c]["text"] for c in sib[:n_neg]]
        if len(texts) < n_neg:
            n_drop += 1
            continue
        r2 = {"anchor": r["anchor"], "positive": r["positive"]}
        for k, t in enumerate(texts[:n_neg]):
            r2[f"negative_{k+1}"] = t
        out.append(r2)
    print(f"  [mine] {len(out)}/{len(rows)} nhóm · {n_neg} negative/nhóm"
          + (f" · giữ lại {kept} ứng viên sau chốt chặn" if gate else " · KHÔNG lọc")
          + (f" · bỏ {n_drop} nhóm thiếu negative" if n_drop else ""))
    return out



# =============================================================================
# v6_2 — OPTION 1: FINE-TUNE RERANKER, NHƯNG GIỮ CẢ HAI BẢN ĐỂ SO TRONG CÙNG PHIÊN
# =============================================================================
# v6_1 ghi: "bản fine-tune THAY THẾ zero-shot, muốn so thì chạy hai lượt". Hai lượt Kaggle
# khác nhau không phải phép so paired — khác cả encoder fine-tune, khác cả máy. v6_2 nạp CẢ
# HAI bản lên mỗi GPU (568M fp16 ≈ 1,1GB/bản, T4 dư chỗ) rồi để Cell 12 chấm cả hai trên
# CÙNG câu dev, CÙNG ứng viên tầng 1, qua cổng split-half. Bản thắng mới đi vào bài nộp; bản
# thua bị giải phóng, nên pipeline nộp chỉ có một reranker (hoặc hai nếu nhánh "ft2" thắng —
# vẫn 2,3B < 4B, xem rule.md §2.1).
#
# Thứ tự trong cell: fine-tune TRƯỚC (cần gần trọn một thẻ T4), rồi mới nạp các bản suy luận.
# Encoder dense đang nằm trên thẻ fine-tune được dời tạm sang CPU để nhường VRAM.
print("=== Bước 5b [v6_2]: Fine-tune reranker + nạp zero-shot VÀ fine-tune ===")
reranker_finetune_info = {"used": False, "reason": None, "meta": None, "ckpt": None,
                          "mine_elapsed_s": None}
reranker_models_ft, reranker_tokenizers_ft = {}, {}
_ft_ckpt = os.path.join(CHECKPOINT_DIR, "reranker-ft")
# Ngân sách tối thiểu còn lại SAU fine-tune: chấm nhiều nhánh trên dev + harvest LTR + Bước 7.
_need_after_ft = PUBLIC_RESERVE_SEC + 90 * 60

if USE_RERANKER_FINETUNE and rows_clean and remaining() > RERANKER_FT_TIME_BUDGET_SEC + 15 * 60 + _need_after_ft:
    ft_dev = DEVICES[0]
    moved = []
    try:
        for ch in DENSE_CHANNELS:
            if str(next(ch["model"].parameters()).device) == ft_dev:
                ch["model"].to("cpu")
                moved.append(ch)
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        t_mine = time.time()
        rerank_rows = mine_family_negatives(
            rows_clean, all_chunks, chunk_by_id, bm25, RERANKER_BASE, ft_dev,
            n_neg=N_NEG_RERANK, gate=RERANK_FALSE_NEG_GATE)
        reranker_finetune_info["mine_elapsed_s"] = round(time.time() - t_mine, 1)
        budget = min(RERANKER_FT_TIME_BUDGET_SEC, remaining() - _need_after_ft - 5 * 60)
        print(f"  [reranker-ft] {len(rerank_rows)} nhóm · ngân sách {budget/60:.0f} phút")
        m_ft, t_ft, meta = finetune_reranker(
            rerank_rows, RERANKER_BASE, ft_dev, budget, RERANKER_FT_LR, SEED, N_NEG_RERANK,
            accum=RERANKER_FT_ACCUM, epochs=RERANKER_FT_EPOCHS, warmup=RERANKER_FT_WARMUP,
            max_length=RERANKER_FT_MAX_LEN, use_8bit_optim=USE_8BIT_OPTIM)
        m_ft.half().save_pretrained(_ft_ckpt)
        t_ft.save_pretrained(_ft_ckpt)
        del m_ft
        reranker_finetune_info.update({"used": True, "meta": meta, "ckpt": _ft_ckpt})
        print(f"  [reranker-ft] xong: {meta['updates']} bước, {meta['elapsed_s']/60:.1f} phút, "
              f"dừng vì {meta['stop_reason']} -> {_ft_ckpt}")
    except Exception as e:
        print(f"  [reranker-ft] BỎ QUA ({type(e).__name__}: {e}) -> chỉ còn nhánh zero-shot.")
        reranker_finetune_info["reason"] = f"{type(e).__name__}: {e}"
    finally:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        for ch in moved:
            ch["model"].to(ft_dev)
else:
    reranker_finetune_info["reason"] = (
        "USE_RERANKER_FINETUNE=False" if not USE_RERANKER_FINETUNE
        else ("không có rows_clean" if not rows_clean else
              f"không đủ thời gian (còn {remaining()/60:.0f} phút)"))
    print(f"  {reranker_finetune_info['reason']} -> chỉ nhánh zero-shot.")

reranker_models, reranker_tokenizers = {}, {}          # ZERO-SHOT — luôn có
for dev in DEVICES:
    m, t = load_reranker_on(dev, RERANKER_BASE)
    if m is not None:
        reranker_models[dev] = m
        reranker_tokenizers[dev] = t

if reranker_finetune_info["used"]:
    for dev in reranker_models:
        m, t = load_reranker_on(dev, _ft_ckpt)
        if m is None:
            print(f"  [reranker-ft] không nạp được bản fine-tune lên {dev} -> bỏ nhánh fine-tune")
            reranker_models_ft, reranker_tokenizers_ft = {}, {}
            reranker_finetune_info["reason"] = f"không nạp lại được {_ft_ckpt} trên {dev}"
            break
        reranker_models_ft[dev] = m
        reranker_tokenizers_ft[dev] = t

HAS_RERANKER = len(reranker_models) > 0
HAS_FT = bool(reranker_models_ft) and set(reranker_models_ft) == set(reranker_models)
RERANK_DEVICES = list(reranker_models.keys())
RERANKER_SOURCE = "zeroshot"        # Cell 12 cập nhật sau cổng so nhánh
print(f"  Zero-shot trên: {RERANK_DEVICES or '(không tải được)'} · "
      f"fine-tune: {list(reranker_models_ft) if HAS_FT else '(không có)'}")
if not HAS_RERANKER:
    print("  ⛔ KHÔNG có reranker nào -> pipeline sẽ chạy bằng thứ hạng RRF thuần. "
          "Điểm sẽ thấp hơn hẳn (Recall@1 0,154 so với 0,580 — đo từ lượt trước).")
checkpoint("Xong fine-tune + nạp reranker")


In [ ]:
# Cell 10b [v8]: LLM — HÀM + PROMPT, CHƯA NẠP MODEL. v8 dời việc nạp sang Cell 11d vì LoRA phải
# train ở Cell 11c TRƯỚC khi nạp bản suy luận (VRAM). Bốn việc như v7: viết lại truy vấn (tầng 1)
# · chấm có/không + xếp hạng listwise (tầng 2) · sinh câu kết / câu trả lời (tầng 3).
# v8: mọi hàm gọi LLM nhận `ft` — True dùng adapter LoRA, False tắt adapter (= LLM gốc, hành vi
# v7). Mỗi GPU giữ một bản riêng và mỗi luồng chỉ đụng bản của thẻ mình -> bật/tắt an toàn.
import contextlib
import re as _re
from transformers import AutoModelForCausalLM

llm_models, llm_tokenizers = {}, {}
llm_info = {"requested": USE_LLM, "model": LLM_MODEL, "devices": [], "dtype": {}, "error": None,
            "registered": LLM_MODEL in REGISTERED_MODELS, "adapter": None}
_THINK_RE = _re.compile(r"<think>.*?</think>", _re.DOTALL)
YES_ID = NO_ID = None
DIGIT_IDS = []
HAS_LLM = False
HAS_LLM_FT = False


def _chat(tok, user, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return (tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                + "<think>\n\n</think>\n\n")


def _last_logits(mdl, enc):
    try:
        out = mdl(**enc, logits_to_keep=1)
    except TypeError:
        out = mdl(**enc)
    return out.logits[:, -1, :].float()


def _llm_ctx(mdl, ft):
    """ft=False trên model có adapter -> tắt adapter trong khối lệnh (LLM gốc)."""
    if ft or not hasattr(mdl, "disable_adapter"):
        return contextlib.nullcontext()
    return mdl.disable_adapter()


def _load_llm_on(device, adapter_dir=None):
    tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Nạp fp32 rồi .half(): tên tham số dtype khác nhau giữa transformers 4.x và 5.x.
    mdl = AutoModelForCausalLM.from_pretrained(LLM_MODEL)
    if adapter_dir:
        from peft import PeftModel
        mdl = PeftModel.from_pretrained(mdl, adapter_dir)
    if device.startswith("cuda"):
        mdl = mdl.half()
    mdl = mdl.to(device).eval()
    # Qwen dễ tràn số ở fp16: thử một lượt (cả có lẫn không adapter), không hữu hạn thì lùi fp32.
    enc = tok([_chat(tok, "Điều 1. Phạm vi điều chỉnh. Luật này quy định về hợp đồng lao động.")],
              return_tensors="pt").to(device)
    with torch.no_grad():
        finite = bool(torch.isfinite(_last_logits(mdl, enc)).all())
        if adapter_dir:
            with _llm_ctx(mdl, False):
                finite = finite and bool(torch.isfinite(_last_logits(mdl, enc)).all())
    if not finite:
        mdl = mdl.float()
    return mdl, tok, str(next(mdl.parameters()).dtype)


def _words(text, n):
    w = str(text).split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def unit_ref(c):
    loai, so = c.get("loai_vb") or "văn bản", c.get("so_hieu") or ""
    ut = c.get("unit_type", "dieu")
    if ut == "dieu_sub":
        return f"{c.get('khoan_ref', '')} Điều {c.get('dieu_so')} {loai} {so}".strip()
    ut = ut[:-len("_capped")] if ut.endswith("_capped") else ut
    names = {"dieu": "Điều", "muc": "Mục", "phu_luc": "Phụ lục", "tiet": "tiết"}
    no = c.get("dieu_so") if ut == "dieu" else c.get("unit_no", "")
    return (f"{names[ut]} {no} {loai} {so}" if ut in names and no else f"{loai} {so}").strip()


def article_text(c):
    return _words(c["text"], LLM_ARTICLE_WORDS)


def trim_incomplete(text):
    """Văn bản dừng vì chạm max_new_tokens -> bỏ câu dở dang cuối (v7: câu kết cụt giữa câu).
    Chỉ cắt khi còn giữ được ít nhất nửa văn bản; không thì để nguyên."""
    t = str(text).rstrip()
    if not t or t[-1] in ".!?:;)\"”»":
        return t
    ends = [m.end() for m in _re.finditer(r"[.!?;](?=\s|$)", t)]
    if ends and ends[-1] >= 0.5 * len(t):
        return t[:ends[-1]]
    nl = t.rfind("\n")
    if nl >= 0.5 * len(t):
        return t[:nl].rstrip()
    return t


def llm_generate(prompts, device, max_new_tokens, ft=False):
    mdl, tok = llm_models[device], llm_tokenizers[device]
    stop_ids = {i for i in (tok.eos_token_id, tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>"))
                if isinstance(i, int)}
    out, i, bs = [], 0, LLM_GEN_BATCH
    while i < len(prompts):
        chunk = prompts[i:i + bs]
        try:
            enc = tok(chunk, return_tensors="pt", padding=True).to(device)
            with torch.no_grad(), _llm_ctx(mdl, ft):
                gen = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                   temperature=None, top_p=None, top_k=None,
                                   repetition_penalty=1.05, pad_token_id=tok.pad_token_id)
            new = gen[:, enc["input_ids"].shape[1]:]
            texts = tok.batch_decode(new, skip_special_tokens=True)
            for row, t in zip(new.tolist(), texts):
                t = _THINK_RE.sub("", t).strip()
                cut = len(row) >= max_new_tokens and row[-1] not in stop_ids
                out.append(trim_incomplete(t) if cut else t)
            i += len(chunk)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 1:
                torch.cuda.empty_cache()
                bs = max(1, bs // 2)
                continue
            raise
    return out


def llm_map(items, make_user, max_new_tokens, label, system=None, ft=False):
    """items {khoá: payload} -> {khoá: văn bản sinh}; chia đều các GPU có LLM, gom lô theo độ dài."""
    if not items:
        return {}
    step = LLM_GEN_BATCH * 4

    def _worker(chunk, dev, widx):
        tok = llm_tokenizers[dev]
        chunk = sorted(chunk, key=lambda k: len(make_user(items[k])))
        res = {}
        for b in range(0, len(chunk), step):
            part = chunk[b:b + step]
            outs = llm_generate([_chat(tok, make_user(items[k]), system) for k in part], dev,
                                max_new_tokens, ft=ft)
            res.update(zip(part, outs))
            with _print_lock:
                print(f"    [{label} · luồng {widx}] {b + len(part)}/{len(chunk)}")
        return res
    return parallel_process(list(items), _worker, list(llm_models), label=label)


YN_SYSTEM = ('Judge whether the Document meets the requirements based on the Query and the Instruct '
             'provided. Note that the answer can only be "yes" or "no".')
YN_INSTRUCT = "Given a Vietnamese legal question, judge whether the legal article answers the question"


def llm_yes_no(question, texts, device, ft=False):
    mdl, tok = llm_models[device], llm_tokenizers[device]
    prompts = [_chat(tok, f"<Instruct>: {YN_INSTRUCT}\n<Query>: {question}\n"
                          f"<Document>: {_words(t, LLM_YN_WORDS)}", YN_SYSTEM) for t in texts]
    enc = tok(prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad(), _llm_ctx(mdl, ft):
        lg = _last_logits(mdl, enc)
    s = (lg[:, YES_ID] - lg[:, NO_ID]).cpu().numpy().astype(np.float64)
    return s if np.isfinite(s).all() else np.zeros(len(texts))


def list_user(question, texts, order):
    """Prompt listwise — Cell 11c train LoRA trên ĐÚNG chuỗi này (thứ tự xuôi)."""
    n = len(order)
    body = "\n\n".join(f"[{k + 1}] {_words(texts[j], LLM_LIST_WORDS)}" for k, j in enumerate(order))
    return (f"Câu hỏi: {question}\n\nCác điều luật ứng viên:\n{body}\n\n"
            f"Điều luật nào trả lời trực tiếp và đầy đủ nhất cho câu hỏi? "
            f"Chỉ trả lời bằng một chữ số từ 1 đến {n}.")


def llm_listwise(question, texts, device, ft=False):
    """Logit chữ số của câu trả lời "ứng viên nào đúng nhất", trung bình hai hoán vị (xuôi,
    ngược) để khử thiên lệch vị trí. Trả log-xác suất theo đúng thứ tự `texts`."""
    n = len(texts)
    if n < 2:
        return np.zeros(n)
    mdl, tok = llm_models[device], llm_tokenizers[device]

    def one(order):
        enc = tok([_chat(tok, list_user(question, texts, order))], return_tensors="pt").to(device)
        with torch.no_grad(), _llm_ctx(mdl, ft):
            lg = _last_logits(mdl, enc)[0, DIGIT_IDS[:n]]
        return torch.log_softmax(lg, dim=0).cpu().numpy().astype(np.float64)

    fwd = list(range(n))
    rev = fwd[::-1]
    s = np.zeros(n)
    s[fwd] += one(fwd)
    s[rev] += one(rev)
    s /= 2
    return s if np.isfinite(s).all() else np.zeros(n)


REWRITE_USER = (
    "Viết lại câu hỏi pháp luật dưới đây thành MỘT truy vấn tìm kiếm văn bản quy phạm pháp luật "
    "Việt Nam.\n- Giữ nguyên các thuật ngữ quan trọng của câu hỏi.\n- Bổ sung thuật ngữ pháp lý "
    "đồng nghĩa hoặc liên quan, và loại văn bản có thể điều chỉnh vấn đề (luật, nghị định, thông "
    "tư...).\n- Không trả lời câu hỏi, không giải thích.\nChỉ in ra truy vấn trên một dòng.\n\n"
    "Câu hỏi: {q}")
CONCL_USER = (
    "Câu hỏi: {q}\n\nĐiều luật liên quan ({ref}):\n{article}\n\n"
    "Dựa hoàn toàn vào điều luật trên, viết 1-2 câu kết luận trả lời trực tiếp câu hỏi. Bắt đầu "
    "bằng \"Như vậy,\". Dùng lại đúng từ ngữ của câu hỏi và điều luật, không thêm thông tin ngoài "
    "điều luật.")
FREE_USER = (
    "Câu hỏi: {q}\n\nĐiều luật liên quan ({ref}):\n{article}\n\n"
    "Viết câu trả lời cho câu hỏi, dựa hoàn toàn vào điều luật trên, theo đúng khuôn:\n"
    "1. Mở đầu: \"Căn cứ {ref} quy định như sau:\"\n"
    "2. Trích nguyên văn các khoản, điểm liên quan đến câu hỏi.\n"
    "3. Kết luận bắt đầu bằng \"Như vậy,\" trả lời trực tiếp câu hỏi.\n"
    "Không thêm thông tin ngoài điều luật.")
_SENT_RE = _re.compile(r"(?<=[.!?])\s+")


def parse_rewrite(text, question):
    line = next((ln.strip() for ln in str(text).splitlines() if ln.strip()), "")
    if line.lower().startswith(("truy vấn", "query")) and ":" in line:
        line = line.split(":", 1)[1]
    words = line.strip().strip('"“”\'').split()
    return " ".join(words[:60]) if len(words) >= 3 else question


def clean_concl(text):
    t = " ".join(str(text).split())
    i = t.find("Như vậy")
    if i < 0:
        # v8: v7 giữ nguyên văn bản khi không thấy "Như vậy" -> lọt phần LLM nhắc lại prompt
        # ("Câu hỏi: ... Điều luật liên quan ...") vào câu trả lời. Không có kết luận thì lùi template.
        return ""
    t = " ".join(_SENT_RE.split(t[i:])[:2]).strip()
    return t if 3 <= len(t.split()) <= 120 else ""


def valid_free(text, c):
    t = str(text)
    return len(t.split()) >= 20 and ("Điều" in t or bool(c.get("so_hieu") and c["so_hieu"] in t))


print("=== Bước 5c [v8]: hàm LLM sẵn sàng (nạp model ở Cell 11d, sau khi train LoRA ở Cell 11c) ===")


In [ ]:
# Cell 11 [v6_2]: Hàm retrieval (RRF fusion N kênh — BM25 + N encoder) + rerank theo lô
# + hạ tầng chạy song song 2 GPU
#
# v6_2 tách rrf_retrieve thành hai nửa (xếp hạng từng kênh / trộn RRF) và thêm bộ nhớ đệm
# điểm CE theo id ứng viên. Mục đích: Cell 12 so NHIỀU cấu hình trên cùng câu dev (reranker
# zero-shot vs fine-tune, bỏ bớt kênh encoder) mà không phải chấm lại những cặp
# (câu hỏi, ứng viên) đã chấm. Khi không truyền mask/memo/rank_lists, kết quả TRÙNG v6_1:
# cùng thứ tự dựng tập, cùng công thức RRF, cùng thứ tự sort.
from concurrent.futures import ThreadPoolExecutor

_print_lock = __import__("threading").Lock()

CHANNEL_NAMES = ["bm25"] + [ch["name"] for ch in DENSE_CHANNELS]


def rrf_rank_lists(question: str, bm25, dense_channels, top_k: int = TOP_K_RETRIEVE):
    """-> [xếp hạng BM25, xếp hạng kênh dense 1, ...] — mỗi phần tử là list index chunk."""
    lists = [list(bm25.top_k(tokenize_simple(question), top_k))]
    for ch in dense_channels:
        q_text = ch["query_prefix"] + question if ch["query_prefix"] else question
        q_emb = ch["model"].encode([q_text], convert_to_numpy=True, normalize_embeddings=True)[0]
        scores = ch["embeddings"] @ q_emb
        lists.append(list(np.argsort(-scores)[:top_k]))
    return lists


def rrf_fuse(rank_lists, all_chunks, mask=None, top_k: int = TOP_K_RETRIEVE):
    """Trộn RRF các kênh có mask=True (None = mọi kênh)."""
    used = [rl for i, rl in enumerate(rank_lists) if mask is None or mask[i]]
    if not used:
        return []
    rank_maps = [{idx: r for r, idx in enumerate(rl)} for rl in used]
    all_idx = set(used[0])
    for rl in used[1:]:
        all_idx |= set(rl)
    rrf = {i: sum(1 / (60 + rm.get(i, top_k + 1)) for rm in rank_maps) for i in all_idx}
    ranked = sorted(rrf, key=rrf.get, reverse=True)
    return [all_chunks[i] for i in ranked]


def rrf_retrieve(question: str, bm25, dense_channels, all_chunks, top_k: int = TOP_K_RETRIEVE):
    return rrf_fuse(rrf_rank_lists(question, bm25, dense_channels, top_k), all_chunks, None, top_k)


def rerank(question: str, candidates: list, reranker_model, reranker_tokenizer,
           max_candidates: int = TOP_K_RETRIEVE, max_length: int = 1024,
           sub_batch: int = RERANK_SUBBATCH, memo: dict = None):
    """Chấm lại top `max_candidates` bằng cross-encoder theo lô nhỏ. `memo` = {id: điểm} của
    ĐÚNG model này cho ĐÚNG câu hỏi này — id đã có điểm thì không chấm lại."""
    if reranker_model is None or not candidates:
        return candidates, None
    subset = candidates[:max_candidates]
    scores = np.empty(len(subset), dtype=np.float32)
    todo = []
    for j, c in enumerate(subset):
        if memo is not None and c["id"] in memo:
            scores[j] = memo[c["id"]]
        else:
            todo.append(j)
    if todo:
        device = next(reranker_model.parameters()).device
        pairs = [[question, subset[j]["text"]] for j in todo]
        out_all = np.empty(len(pairs), dtype=np.float32)
        bs, i = max(1, sub_batch), 0
        while i < len(pairs):
            batch = pairs[i:i + bs]
            try:
                with torch.no_grad():
                    inputs = reranker_tokenizer(batch, padding=True, truncation=True,
                                                 return_tensors="pt", max_length=max_length).to(device)
                    out = reranker_model(**inputs, return_dict=True).logits.view(-1).float().cpu().numpy()
                out_all[i:i + len(batch)] = out
                i += bs
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and bs > 1:
                    torch.cuda.empty_cache()
                    bs = max(1, bs // 2)
                    continue
                if "out of memory" in str(e).lower():
                    return candidates, None
                raise
        for j, s in zip(todo, out_all):
            scores[j] = s
            if memo is not None:
                memo[subset[j]["id"]] = float(s)
    order = np.argsort(-scores)
    reranked = [subset[i2] for i2 in order]
    sorted_scores = scores[order]
    return reranked + candidates[max_candidates:], sorted_scores


def adaptive_k_cutoff(scores, min_k: int = 1, max_k: int = TOP_K_RERANK, search_window: int = 15) -> int:
    """Adaptive-k (Taguchi et al. 2025, arXiv:2506.08479): tìm điểm "gãy" tự nhiên trong
    phân phối điểm reranker đã sort giảm dần thay vì luôn cắt ở top_n cố định."""
    if scores is None or len(scores) == 0:
        return min_k
    n = min(len(scores), search_window)
    if n <= 1:
        return min_k
    gaps = [scores[i] - scores[i + 1] for i in range(n - 1)]
    k_star = int(np.argmax(gaps)) + 1
    return max(min_k, min(k_star, max_k))


_TITLE_MAX_WORDS = 30


def _dieu_title(text: str) -> str:
    """Dòng tiêu đề của Điều ('Điều 17. Vi phạm quy định chung về ...' -> 'vi phạm quy định
    chung về ...'). Rỗng nếu dòng đầu quá dài (thân Điều dính liền, không có tiêu đề riêng)."""
    body = _DIEU_PREFIX_STRIP_RE.sub("", text, count=1)
    first = body.split("\n", 1)[0].strip().rstrip(".:;").strip()
    if not first or len(first.split()) > _TITLE_MAX_WORDS:
        return ""
    return first[0].lower() + first[1:]


def _dieu_lead(ref: str, text: str, template: str) -> str:
    """Câu dẫn cho đơn vị Điều theo biến thể template (Option 2, v6_2).
    Đo trước 0 GPU trên 3.427 Điều gold (METEOR thật, Δ so với T0, hai nửa cùng dấu):
      T1_theo_qd_tai +0,0007 · T2_title +0,0029 · T3_theo_qd_tai_title +0,0040.
    Cell 12 đo lại trên ứng viên pipeline THẬT chọn (không phải Điều gold) trước khi dùng."""
    if template == "T0_current":
        return f"Căn cứ {ref} quy định như sau:"
    title = _dieu_title(text) if template in ("T2_title", "T3_theo_qd_tai_title") else ""
    if template == "T1_theo_qd_tai":
        return f"Căn cứ theo quy định tại {ref} như sau:"
    if template == "T2_title":
        return f"Căn cứ {ref} quy định về {title} như sau:" if title else f"Căn cứ {ref} quy định như sau:"
    if template == "T3_theo_qd_tai_title":
        return (f"Căn cứ theo quy định tại {ref} quy định về {title} như sau:" if title
                else f"Căn cứ theo quy định tại {ref} như sau:")
    raise ValueError(f"template không hợp lệ: {template}")


def render_answer(selected_chunks: list, top_n: int, question: str = "",
                   concl: str = CONCL, template: str = None) -> str:
    """Câu dẫn "Căn cứ Điều X <loại VB> <số hiệu> quy định như sau:" — khuôn phổ biến nhất
    đo được trên answer thật (57.4% mở đầu "Căn cứ", 24.6% có "quy định như sau"). Cắt bỏ
    "Điều X." lặp lại ở đầu thân bài (98.8% answer thật không lặp).

    CÂU KẾT (`concl`) — thay đổi ĐÁNG GIÁ NHẤT và rẻ nhất trong cả pipeline. Đo trên
    501 câu dev, cùng retrieval, chỉ đổi một biến:

        concl=none    METEOR 0,5151
        concl=echo    METEOR 0,5499    Δ +0,0348 ± 0,0017 · 416 thắng / 85 thua · t = 20,4
        concl=echo2   METEOR 0,5630    Δ +0,0131 ± 0,0010 · 371 thắng / 130 thua · t = 12,8

    Split-half (chia đôi dev, chọn trên nửa này đo nửa kia): CẢ HAI nửa độc lập đều chọn
    echo2 (A 0,5675 · B 0,5589) — nên đây không phải ảo giác đỉnh-trên-toàn-dev.

    Vì sao ăn điểm: METEOR có alpha = 0,9 nên nặng recall, và 36,2% đáp án thật chứa
    "Như vậy", 27,5% chứa "Theo đó" — chúng nhắc lại nội dung câu hỏi ở phần kết. Lặp
    lại câu hỏi làm khớp đúng nhóm token đó.

    Đã dò tiếp số lần lặp: 1× 0,5499 · 2× 0,5630 · 3× 0,5674 · 4× 0,5682 · 6× 0,5661.
    Có đỉnh thật quanh 4, NHƯNG dừng ở 2: từ 2 lên 4 chỉ được +0,5 điểm (đúng vùng mà
    "chọn đỉnh trên toàn dev" đã lừa project này ba lần), còn đáp án lặp câu hỏi bốn lần
    thì nhìn bằng mắt là hỏng rõ ràng. Đây là tối ưu HÌNH DẠNG ĐỘ ĐO, hợp lệ theo luật
    nhưng không làm câu trả lời tốt hơn cho người đọc — biết để không đi xa hơn một cách
    mù quáng."""
    # BẢN SỬA v5 (Batch 1 A1): câu dẫn dựng theo `unit_type` thay vì chỉ nhánh "Điều/khác" —
    # unit_type mới ("muc"/"phu_luc"/"tiet" và biến thể "*_capped") đến từ chunk_passage()
    # (Cell 5) khi văn bản không có Điều nhưng có cấu trúc thay thế. "raw"/"raw450" (không
    # khớp tầng bậc nào) vẫn dùng câu dẫn cũ "Căn cứ {loai_vb} {so_hieu}".
    parts, seen = [], set()
    for c in selected_chunks:
        if c["id"] in seen or len(parts) >= top_n:
            continue
        seen.add(c["id"])
        loai_vb = c["loai_vb"] or "văn bản"
        so_hieu = c["so_hieu"] or ""
        dieu = c["dieu_so"]
        unit_type = c.get("unit_type", "dieu" if dieu != "0" else "raw")
        unit_no = c.get("unit_no", "")
        base_unit_type = unit_type[:-len("_capped")] if unit_type.endswith("_capped") else unit_type
        if base_unit_type in ("dieu", "dieu_sub"):
            _kref = (c.get("khoan_ref", "") + " ") if base_unit_type == "dieu_sub" else ""
            lead = _dieu_lead(f"{_kref}Điều {dieu} {loai_vb} {so_hieu}", c["text"],
                              template or ANSWER_TEMPLATE)
            body = _DIEU_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "muc" and unit_no:
            lead = f"Căn cứ Mục {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _MUC_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "phu_luc" and unit_no:
            lead = f"Căn cứ Phụ lục {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _PHU_LUC_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "tiet" and unit_no:
            lead = f"Căn cứ tiết {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _TIET_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        else:
            lead = f"Căn cứ {loai_vb} {so_hieu} quy định như sau:"
            body = c["text"]
        parts.append(f"{lead}\n{body}")
    ans = "\n\n".join(parts)
    if concl != "none" and question:
        q = question.strip().rstrip("?").strip()
        if q:
            ql = q[0].lower() + q[1:]
            if concl == "echo":
                ans += f"\nNhư vậy, theo quy định nêu trên thì {ql}."
            elif concl == "echo2":
                ans += f"\nTheo đó, {ql}.\nNhư vậy, theo quy định nêu trên thì {ql}."
    return ans



def _score_docs(candidates: list, scores, mode: str = "max", T: float = 1.0) -> dict:
    """{doc_id: điểm gộp}. Tách riêng khỏi aggregate_docs (BẢN SỬA v4) để dùng lại được ở
    TẦNG 1 (chọn top-K văn bản trong retrieve_two_tier) — trước v4 phép gộp này chỉ áp
    dụng được sau khi ĐÃ có danh sách Điều phẳng của một tầng duy nhất."""
    by_doc = {}
    for c, sc in zip(candidates, scores):
        by_doc.setdefault(str(c["id"]).split("_")[0], []).append(float(sc))
    if mode == "max":
        return {d: max(v) for d, v in by_doc.items()}
    doc_score = {}
    for d, v in by_doc.items():
        arr = np.array(v, dtype=np.float64) / max(T, 1e-6)
        doc_score[d] = float(T * (arr.max() + np.log(np.exp(arr - arr.max()).sum())))
    return doc_score


# v6_1 — ĐÃ XOÁ aggregate_docs(). Nó chỉ tồn tại để phục vụ AGG_MODE="lse", mà LSE đã đo âm
# hai lần độc lập và bị ép về "max" ở Cell 2 — tức hàm này chỉ còn là một no-op có 30 dòng
# docstring giải thích một hướng đã đóng. _score_docs() bên trên GIỮ LẠI vì retrieve_two_tier
# vẫn dùng nó để gộp điểm chọn top-K văn bản ở tầng 1 (luôn mode="max").

def retrieve_two_tier(question: str, bm25_t1, dense_channels, all_chunks_t1, dieu_by_doc,
                       reranker_model=None, reranker_tokenizer=None, doc_k: int = DOC_K,
                       t2_model=None, t2_tokenizer=None, mask=None, rank_lists=None,
                       memo_t1: dict = None, memo_t2: dict = None, diag: dict = None):
    """HAI TẦNG (result.md §3): tầng 1 truy xuất trên corpus 450 TỪ để chọn doc_k VĂN BẢN,
    tầng 2 gom Điều CHỈ TRONG các văn bản đó và rerank lại.

    v6_2 — tham số mới, đều tuỳ chọn (bỏ trống = hành vi v6_1):
      t2_model/t2_tokenizer  reranker riêng cho tầng 2 (nhánh "ft2": zero-shot chọn văn bản,
                             fine-tune chọn Điều — bản fine-tune chỉ học trên đơn vị Điều)
      mask                   kênh nào tham gia RRF tầng 1, theo CHANNEL_NAMES (Option 3)
      rank_lists             xếp hạng từng kênh đã tính sẵn (khỏi encode lại câu hỏi)
      memo_t1/memo_t2        {id: điểm CE} đã chấm — tránh chấm lại
      diag                   dict nhận top_docs để đo recall văn bản tầng 1
    """
    if rank_lists is None:
        rank_lists = rrf_rank_lists(question, bm25_t1, dense_channels)
    t1_ranked = rrf_fuse(rank_lists, all_chunks_t1, mask)
    if not t1_ranked:
        return [], None
    t1_scores = None
    if reranker_model is not None:
        t1_ranked, t1_scores = rerank(question, t1_ranked, reranker_model, reranker_tokenizer,
                                      memo=memo_t1)
    if t1_scores is None:
        top_docs = list(dict.fromkeys(str(c["id"]).split("_")[0] for c in t1_ranked))[:doc_k]
    else:
        doc_score = _score_docs(t1_ranked, t1_scores, "max", 1.0)
        top_docs = sorted(doc_score, key=doc_score.get, reverse=True)[:doc_k]
    if diag is not None:
        diag["top_docs"] = list(top_docs)

    cand = []
    for d in top_docs:
        cand.extend(dieu_by_doc.get(d, []))
    if len(cand) > MAX_DIEU_CANDIDATES:
        qt = set(tokenize_simple(question))
        cand = sorted(cand, key=lambda c: -len(qt & set(tokenize_simple(c["text"]))))[:MAX_DIEU_CANDIDATES]
    if not cand:
        return [], None
    m2, k2 = (t2_model, t2_tokenizer) if t2_model is not None else (reranker_model, reranker_tokenizer)
    if m2 is None:
        return cand, None
    return rerank(question, cand, m2, k2, max_candidates=len(cand), memo=memo_t2)


# v6_1 — ĐÃ XOÁ answer_question(). Bước 7 dùng answer_question_v61() ở Cell 14, vốn phải
# thêm bước LTR resort. Giữ cả hai là để hai đường dẫn sinh đáp án gần-giống-nhau cùng tồn
# tại trong một notebook — đúng loại bẫy đã làm result.md §4.3 đọc sai log một lần.

def split_evenly(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


def parallel_process(ids, worker_fn, devices, label: str = "", progress_every: int = 50):
    """Chia `ids` đều cho từng thiết bị trong `devices`, chạy worker_fn(ids_chunk, device)
    ĐỒNG THỜI trên các luồng riêng. PyTorch giải phóng GIL trong lúc chờ CUDA hoàn thành nên
    2 luồng ghim vào 2 GPU vật lý khác nhau chạy song song THẬT (không phải giả song song do
    GIL) — đây là chỗ mang lại tốc độ x~2 cho phần rerank ở Bước 6/7.
    worker_fn(ids_chunk, device, worker_idx) -> dict {qid: kết quả}."""
    ids = list(ids)
    devices = list(devices) if devices else ["cpu"]
    chunks = split_evenly(ids, len(devices))
    results = {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(devices)) as ex:
        futures = [ex.submit(worker_fn, chunk, dev, i) for i, (chunk, dev) in enumerate(zip(chunks, devices))]
        for f in futures:
            results.update(f.result())
    print(f"    [{label}] {len(ids)} câu / {len(devices)} thiết bị song song -> {time.time()-t0:.0f}s")
    return results


def _progress_print(label, worker_idx, i, n):
    if (i + 1) % 50 == 0 or (i + 1) == n:
        with _print_lock:
            print(f"    [{label} · luồng {worker_idx}] {i+1}/{n}")

# =============================================================================
# v6_1 — ĐẶC TRƯNG CHO BỘ CHỌN ĐIỀU HỌC ĐƯỢC (LTR)
# =============================================================================
# Mục tiêu: bắt phần dư địa +7,7 điểm ở result.md §7 — Điều tốt nhất ĐÃ nằm trong 5 ứng viên
# reranker đưa ra, chỉ bị xếp sai thứ tự (reranker chọn đúng 61,3%, khoảng cách điểm hạng 1
# vs hạng 2 trung vị 0,0023).
#
# NGUYÊN TẮC CHỌN ĐẶC TRƯNG — không đặc trưng nào được nhìn thấy gold:
#   · mọi thứ tính từ (câu hỏi, văn bản ứng viên) hoặc từ chính phân phối điểm CE;
#   · KHÔNG dùng độ dài đáp án thật, KHÔNG dùng nhãn citation, KHÔNG dùng METEOR.
# METEOR chỉ xuất hiện ở phía NHÃN lúc train (Cell 13), không bao giờ ở phía đặc trưng.
#
# Vì sao các đặc trưng "hình dạng phân phối điểm" (z-score, gap) quan trọng hơn điểm thô:
# result.md §7 đo được điểm CE tuyệt đối gần như không phân biệt nổi hạng 1 với hạng 2. Cái
# còn mang tin là điểm đó nằm ở đâu TRONG NHÓM — một điểm 8,0 giữa nhóm toàn 7,9 nói lên
# điều khác hẳn một điểm 8,0 giữa nhóm toàn 2,0.
_Q_SO_HIEU_RE = re.compile(r"\d+\s*/\s*\d{4}\s*/\s*[A-ZĐ\-]+", re.IGNORECASE)
_Q_DIEU_RE = re.compile(r"[Đđ]iều\s+(\d+)")

LTR_FEATURE_NAMES = [
    "ce_score", "ce_rank", "ce_gap_to_top", "ce_gap_to_next", "ce_z_in_group",
    "n_words", "q_overlap", "q_overlap_ratio", "so_hieu_match", "dieu_no_match",
    "doc_rank", "is_first_in_doc", "unit_is_dieu",
]


def extract_ltr_features(question: str, ranked: list, scores, top_k: int = LTR_TOP_K_CANDIDATES):
    """-> (X, cands) với X là list vector đặc trưng song song với cands (top_k ứng viên đầu).

    Trả về list rỗng nếu không có điểm CE — LTR chỉ có nghĩa khi đã có một thứ hạng để sửa.
    """
    if scores is None or not ranked:
        return [], []
    cands = ranked[:top_k]
    sc = np.asarray(scores[:top_k], dtype=np.float64)
    if len(sc) == 0:
        return [], []
    top, mean, std = float(sc[0]), float(sc.mean()), float(sc.std())
    q_tokens = set(tokenize_simple(question))
    q_so_hieu = {norm_so_hieu(m.group(0)) for m in _Q_SO_HIEU_RE.finditer(question)}
    q_dieu = {m.group(1) for m in _Q_DIEU_RE.finditer(question)}
    # Thứ hạng VĂN BẢN theo thứ tự xuất hiện đầu tiên trong danh sách đã rerank.
    doc_order, seen_doc = {}, 0
    for c in cands:
        d = str(c["id"]).split("_")[0]
        if d not in doc_order:
            doc_order[d] = seen_doc
            seen_doc += 1
    X = []
    for i, c in enumerate(cands):
        c_tokens = set(tokenize_simple(c["text"]))
        overlap = len(q_tokens & c_tokens)
        n_words = len(c["text"].split())
        doc = str(c["id"]).split("_")[0]
        unit_type = c.get("unit_type", "dieu" if c.get("dieu_so", "0") != "0" else "raw")
        X.append([
            float(sc[i]),
            float(i),
            top - float(sc[i]),
            float(sc[i] - sc[i + 1]) if i + 1 < len(sc) else 0.0,
            (float(sc[i]) - mean) / std if std > 1e-9 else 0.0,
            float(n_words),
            float(overlap),
            overlap / max(len(q_tokens), 1),
            1.0 if (q_so_hieu and norm_so_hieu(c.get("so_hieu") or "") in q_so_hieu) else 0.0,
            1.0 if (q_dieu and str(c.get("dieu_so", "0")) in q_dieu) else 0.0,
            float(doc_order[doc]),
            1.0 if (i == 0 or str(cands[i - 1]["id"]).split("_")[0] != doc) else 0.0,
            1.0 if unit_type.startswith("dieu") else 0.0,
        ])
    return X, cands


def apply_ltr_resort(question: str, ranked: list, scores, ltr_model,
                      top_k: int = LTR_TOP_K_CANDIDATES):
    """Xếp lại top_k ứng viên đầu bằng LTR, GIỮ NGUYÊN phần đuôi.

    Chỉ đổi THỨ TỰ trong top_k — không thêm/bớt ứng viên, không đụng tầng 1. Nếu LTR lỗi
    (model None, đặc trưng rỗng, predict ném lỗi) thì trả về đúng đầu vào: hỏng ở đây phải
    im lặng lùi về hành vi đã proven, không được làm sập một phiên GPU nhiều giờ.
    """
    if ltr_model is None or scores is None or not ranked:
        return ranked, scores
    try:
        X, cands = extract_ltr_features(question, ranked, scores, top_k)
        if not X:
            return ranked, scores
        pred = ltr_model.predict(np.asarray(X, dtype=np.float64))
        order = list(np.argsort(-np.asarray(pred)))
        new_head = [cands[i] for i in order]
        head_scores = [float(scores[i]) for i in order]
        tail = list(ranked[len(cands):])
        tail_scores = [float(s) for s in np.asarray(scores[len(cands):], dtype=np.float64)]
        return new_head + tail, np.asarray(head_scores + tail_scores, dtype=np.float64)
    except Exception as e:
        print(f"    [LTR] lỗi lúc resort ({type(e).__name__}: {e}) -> giữ thứ hạng reranker")
        return ranked, scores


In [ ]:
# Cell 11b [v7]: tầng 1 nhiều cấu hình + breadcrumb + các bộ chấm tầng 2 + dựng câu trả lời.
# Một cấu hình tầng 1 = {"w": trọng số RRF theo tên kênh, "kb": (k1, b) BM25 hoặc None,
#                        "prf": None|"add"|"replace", "rw": None|"all"|"bm25"}.
# Cấu hình rỗng (BASE_CFG) cho ra ĐÚNG thứ hạng của retrieve_two_tier() v6_2.
import copy

BASE_CFG = {"w": {}, "kb": None, "prf": None, "rw": None}


def rrf_idx(rank_lists, weights, top_k=TOP_K_RETRIEVE):
    # Cùng thứ tự dựng tập và cùng công thức với rrf_fuse (Cell 11) -> trọng số toàn 1 trùng v6_2.
    rank_maps = [{idx: r for r, idx in enumerate(rl)} for rl in rank_lists]
    all_idx = set(rank_lists[0])
    for rl in rank_lists[1:]:
        all_idx |= set(rl)
    rrf = {i: sum(w / (60 + rm.get(i, top_k + 1)) for rm, w in zip(rank_maps, weights))
           for i in all_idx}
    return sorted(rrf, key=rrf.get, reverse=True)


_BM25_VARIANTS = {}


def bm25_kb(k1, b):
    # k1/b chỉ vào công thức lúc chấm (idf không phụ thuộc) -> bản sao nông, dùng chung index.
    key = (k1, b)
    if key not in _BM25_VARIANTS:
        o = copy.copy(bm25_t1)
        o.k1, o.b = k1, b
        _BM25_VARIANTS[key] = o
    return _BM25_VARIANTS[key]


def prf_query_tokens(question, ranked_idx, n_docs=PRF_N_DOCS, n_terms=PRF_N_TERMS):
    q = tokenize_simple(question)
    qs = set(q)
    tf = Counter()
    for i in ranked_idx[:n_docs]:
        toks = tokenized_t1[i]
        for t, f in Counter(toks).items():
            tf[t] += f / max(len(toks), 1)
    scored = sorted(((w * bm25_t1.idf.get(t, 0.0), t) for t, w in tf.items()
                     if t not in qs and len(t) > 1 and not t.isdigit()), reverse=True)
    return q + [t for _w, t in scored[:n_terms]]


def build_list_cache(question, rw_query=None, cfgs=None):
    """Mọi xếp hạng từng kênh mà các cấu hình `cfgs` cần (None = mọi thứ)."""
    need_kb = ({tuple(cf["kb"]) for cf in cfgs if cf.get("kb")} if cfgs is not None
               else {tuple(kb) for kb in BM25_KB_GRID})
    need_prf = cfgs is None or any(cf.get("prf") for cf in cfgs)
    need_rw = bool(rw_query) and (cfgs is None or any(cf.get("rw") for cf in cfgs))
    base = rrf_rank_lists(question, bm25_t1, DENSE_CHANNELS)
    toks = tokenize_simple(question)
    c = {"bm25": base[0], "dense": base[1:], "kb": {}, "prf": None, "rw": None}
    for kb in need_kb:
        c["kb"][kb] = list(bm25_kb(*kb).top_k(toks, TOP_K_RETRIEVE))
    if need_prf:
        c["prf"] = list(bm25_t1.top_k(prf_query_tokens(question, rrf_idx(base, [1.0] * len(base))),
                                      TOP_K_RETRIEVE))
    if need_rw:
        c["rw"] = rrf_rank_lists(rw_query, bm25_t1, DENSE_CHANNELS)
    return c


def lists_for(cfg, c):
    lists = [c["kb"][tuple(cfg["kb"])] if cfg.get("kb") else c["bm25"]] + list(c["dense"])
    w = [float(cfg["w"].get(n, 1.0)) for n in CHANNEL_NAMES]
    if cfg.get("prf") == "replace" and c["prf"] is not None:
        lists[0] = c["prf"]
    elif cfg.get("prf") == "add" and c["prf"] is not None:
        lists.append(c["prf"])
        w.append(1.0)
    if cfg.get("rw") and c["rw"] is not None:
        extra = c["rw"] if cfg["rw"] == "all" else c["rw"][:1]
        lists += extra
        w += [1.0] * len(extra)
    return lists, w


def t2_from_lists(question, lists, weights, m, k, memo_t1, memo_t2, diag=None, doc_k=DOC_K):
    """retrieve_two_tier() (Cell 11) với tầng 1 trộn từ `lists`/`weights`; phần còn lại y hệt."""
    t1_ranked = [all_chunks_t1[i] for i in rrf_idx(lists, weights)]
    if not t1_ranked:
        return [], None
    t1_scores = None
    if m is not None:
        t1_ranked, t1_scores = rerank(question, t1_ranked, m, k, memo=memo_t1)
    if t1_scores is None:
        top_docs = list(dict.fromkeys(str(c["id"]).split("_")[0] for c in t1_ranked))[:doc_k]
    else:
        doc_score = _score_docs(t1_ranked, t1_scores, "max", 1.0)
        top_docs = sorted(doc_score, key=doc_score.get, reverse=True)[:doc_k]
    if diag is not None:
        diag["top_docs"] = list(top_docs)
    cand = []
    for d in top_docs:
        cand.extend(dieu_by_doc.get(d, []))
    if len(cand) > MAX_DIEU_CANDIDATES:
        qt = set(tokenize_simple(question))
        cand = sorted(cand, key=lambda c: -len(qt & set(tokenize_simple(c["text"]))))[:MAX_DIEU_CANDIDATES]
    if not cand:
        return [], None
    if m is None:
        return cand, None
    return rerank(question, cand, m, k, max_candidates=len(cand), memo=memo_t2)


# ---------------------------------------------------------------------------
# Breadcrumb Chương/Mục cho đơn vị Điều (Task 1 đo +0,72 nhưng p=0,136 — thử lại qua cổng T2).
# Chỉ đổi văn bản CE nhìn thấy khi chấm lại top-5, không đổi index.
# ---------------------------------------------------------------------------
CHUONG_RE = re.compile(r"^[ \t]*Chương\s+([IVXLCDM]+|\d+)\b", re.MULTILINE)


def _heading_line(passage, m):
    end = passage.find("\n", m.start())
    line = passage[m.start(): end if end >= 0 else len(passage)].strip()
    if len(line.split()) <= 3 and end >= 0:
        nxt = passage[end + 1: end + 300].strip().split("\n", 1)[0].strip()
        if nxt and len(nxt.split()) <= 25 and not DIEU_RE.match(nxt):
            line = f"{line}. {nxt}"
    return " ".join(line.split()[:30])


def build_breadcrumbs(contexts_dir):
    """{id Điều (Cell 5): "Chương ... > Mục ..."} — cùng DIEU_RE và cùng chỉ số i với chunk_passage."""
    contexts_dir = Path(contexts_dir)
    files = sorted(contexts_dir.glob("context_*.json"))
    if not files and (contexts_dir / "selected-contexts").exists():
        files = sorted((contexts_dir / "selected-contexts").glob("context_*.json"))
    out = {}
    for fp in files:
        try:
            with fp.open(encoding="utf-8") as f:
                doc = json.load(f)
        except Exception:
            continue
        passage = doc.get("passage")
        if not passage:
            continue
        dm = list(DIEU_RE.finditer(passage))
        if not dm:
            continue
        heads = sorted([(m.start(), 0, _heading_line(passage, m)) for m in CHUONG_RE.finditer(passage)]
                       + [(m.start(), 1, _heading_line(passage, m)) for m in MUC_RE.finditer(passage)])
        h, ch, mu = 0, "", ""
        for i, m in enumerate(dm):
            while h < len(heads) and heads[h][0] < m.start():
                if heads[h][1] == 0:
                    ch, mu = heads[h][2], ""
                else:
                    mu = heads[h][2]
                h += 1
            if ch or mu:
                out[f"{doc['id']}_dieu{m.group(1)}_{i}"] = " > ".join(x for x in (ch, mu) if x)
    return out


def bc_text(c):
    cid = str(c["id"])
    crumb = BREADCRUMBS.get(cid) or BREADCRUMBS.get(cid.rsplit("_", 1)[0], "")
    head = " ".join(x for x in (c.get("loai_vb") or "", c.get("so_hieu") or "") if x)
    pre = " > ".join(x for x in (head, crumb) if x)
    return f"[{pre}]\n{c['text']}" if pre else c["text"]


def ce_score_texts(question, texts, dev, max_length=1024):
    m, t = reranker_models.get(dev), reranker_tokenizers.get(dev)
    if m is None or not texts:
        return np.zeros(len(texts))
    with torch.no_grad():
        enc = t([[question, x] for x in texts], padding=True, truncation=True, return_tensors="pt",
                max_length=max_length).to(next(m.parameters()).device)
        return m(**enc, return_dict=True).logits.view(-1).float().cpu().numpy().astype(np.float64)


# ---------------------------------------------------------------------------
# Bộ chấm tầng 2 trên top-5 CE. Mỗi nhánh = (hàm điểm, tập tín hiệu cần tính).
# ---------------------------------------------------------------------------
def _z(a):
    a = np.asarray(a, dtype=np.float64)
    s = a.std()
    return (a - a.mean()) / s if s > 1e-9 else np.zeros_like(a)


T2_ARMS = {
    "base": (lambda ex: ex["ce"], set()),
    "bc_only": (lambda ex: ex["bc"], {"bc"}),
    "bc_ce_0.5": (lambda ex: _z(ex["ce"]) + 0.5 * _z(ex["bc"]), {"bc"}),
}
for _a in T2_ALPHAS:
    T2_ARMS[f"yn_a{_a}"] = (lambda ex, a=_a: _z(ex["ce"]) + a * _z(ex["yn"]), {"yn"})
    T2_ARMS[f"list_a{_a}"] = (lambda ex, a=_a: _z(ex["ce"]) + a * _z(ex["list"]), {"list"})
T2_ARMS["yn_only"] = (lambda ex: ex["yn"], {"yn"})
T2_ARMS["list_only"] = (lambda ex: ex["list"], {"list"})
T2_ARMS["yn_list_0.5"] = (lambda ex: _z(ex["ce"]) + 0.5 * _z(ex["yn"]) + 0.5 * _z(ex["list"]),
                          {"yn", "list"})
T2_ARMS["bc_list_0.5"] = (lambda ex: _z(ex["bc"]) + 0.5 * _z(ex["list"]), {"bc", "list"})
T2_LLM_ARMS = {n for n, (_f, need) in T2_ARMS.items() if need & {"yn", "list"}}


def t2_extras(question, top, ce, dev, needs):
    ex = {"ce": np.asarray(ce, dtype=np.float64)}
    if "bc" in needs:
        ex["bc"] = ce_score_texts(question, [bc_text(c) for c in top], dev)
    if "yn" in needs:
        ex["yn"] = llm_yes_no(question, [c["text"] for c in top], dev)
    if "list" in needs:
        ex["list"] = llm_listwise(question, [c["text"] for c in top], dev)
    return ex


def _t2_order(ex, arm, n):
    if not ex or arm == "base":
        return list(range(n))
    s = np.asarray(T2_ARMS[arm][0](ex), dtype=np.float64)
    return [int(i) for i in np.argsort(-s, kind="stable")]


# ---------------------------------------------------------------------------
# Dựng câu trả lời (cổng G). "base" = render_answer của v6_2 (câu dẫn + Điều + echo2).
# ---------------------------------------------------------------------------
G_ARMS = ["base", "concl_replace", "concl_plus_echo2", "free", "free_plus_verbatim"]


def compose_answer(c, question, arm, gen):
    base = render_answer([c], 1, question)
    if arm == "base" or not gen:
        return base
    if arm in ("concl_replace", "concl_plus_echo2"):
        g = clean_concl(gen)
        if not g:
            return base
        if arm == "concl_replace":
            return render_answer([c], 1, question, concl="none") + "\n" + g
        return base + "\n" + g
    if not valid_free(gen, c):
        return base
    return gen if arm == "free" else gen + "\n\n" + base


print("=== Bước 5d [v7]: Breadcrumb Chương/Mục ===")
BREADCRUMBS = build_breadcrumbs(CONTEXT_DIR)
_n_dieu = sum(1 for c in all_chunks if c.get("unit_type", "").startswith("dieu"))
_n_bc = sum(1 for c in all_chunks if c.get("unit_type", "").startswith("dieu")
            and (str(c["id"]) in BREADCRUMBS or str(c["id"]).rsplit("_", 1)[0] in BREADCRUMBS))
print(f"  {len(BREADCRUMBS)} Điều có Chương/Mục · phủ {_n_bc}/{_n_dieu} chunk Điều")
for _k in list(BREADCRUMBS)[:3]:
    print(f"    {_k}: {BREADCRUMBS[_k]}")
checkpoint("Xong hàm v7 + breadcrumb")


# =============================================================================
# v8 — ỨNG VIÊN ĐA ĐỘ HẠT: cụm 1..SUBSPAN_MAX_WIN khoản liên tiếp của một Điều dài.
# =============================================================================
# Đo 0 GPU trên 3.427 Điều gold (Cell 2): Điều đầy đủ 0,6596, oracle thêm cụm ≤3 khoản 0,6955.
# 55% đáp án gold trích "khoản X ... Điều Y" — người viết đáp án thường chỉ chép vài khoản.
# Cụm khoản là ứng viên THÊM VÀO; Điều đầy đủ vẫn nằm trong danh sách. Id xác định (tái lập được):
# "<id Điều>#k<đầu>-<cuối>", đăng ký vào chunk_by_id để mọi chỗ tra cứu cũ dùng được.
from functools import lru_cache

KHOAN_RE = re.compile(r"^[ \t]*(\d{1,2})\.[ \t]+", re.MULTILINE)
_ctx_dir = Path(CONTEXT_DIR)
_ctx_files = (sorted(_ctx_dir.glob("context_*.json"))
              or sorted((_ctx_dir / "selected-contexts").glob("context_*.json")))
_CTX_PATH = {p.stem.split("_", 1)[1]: p for p in _ctx_files}


@lru_cache(maxsize=8192)
def _capped_parent_text(parent_id):
    """Nguyên văn (CÒN xuống dòng) của Điều > MAX_UNIT_WORDS mà Cell 5 đã cắt phẳng 450 từ —
    bản cắt phẳng mất hết đầu dòng nên không tách khoản được. Id cha = "<doc>_dieu<số>_<i>",
    i = chỉ số khớp DIEU_RE trong passage (đúng cách chunk_passage đánh số)."""
    doc, _sep, rest = str(parent_id).partition("_")
    p = _CTX_PATH.get(doc)
    if p is None or not rest:
        return ""
    try:
        with p.open(encoding="utf-8") as f:
            passage = json.load(f).get("passage") or ""
        i = int(rest.rsplit("_", 1)[1])
    except Exception:
        return ""
    ms = list(DIEU_RE.finditer(passage))
    if i >= len(ms):
        return ""
    end = ms[i + 1].start() if i + 1 < len(ms) else len(passage)
    return passage[ms[i].start():end].strip()


def _subspans_of(c):
    ut = str(c.get("unit_type", ""))
    if not USE_SUBSPAN or not ut.startswith("dieu") or ut == "dieu_sub":
        return []
    capped = ut.endswith("_capped")
    base_id = str(c["id"]).rsplit("_", 1)[0] if capped else str(c["id"])
    text = _capped_parent_text(base_id) if capped else c["text"]
    if not text or len(text.split()) < SUBSPAN_MIN_WORDS:
        return []
    ms, want = [], 1
    for m in KHOAN_RE.finditer(text):
        if int(m.group(1)) == want:        # chỉ nhận dãy khoản đánh số liên tiếp từ 1
            ms.append(m)
            want += 1
    if len(ms) < 2:
        return []
    head = text[:ms[0].start()].strip()
    parts = [(m.group(1), text[m.start():(ms[j + 1].start() if j + 1 < len(ms) else len(text))].strip())
             for j, m in enumerate(ms)]
    out, n = [], len(parts)
    for a in range(n):
        for b in range(a, min(n, a + SUBSPAN_MAX_WIN)):
            if a == 0 and b == n - 1:
                continue                   # = cả Điều, đã là ứng viên
            ks = [parts[j][0] for j in range(a, b + 1)]
            ref = (f"khoản {ks[0]}" if len(ks) == 1 else f"khoản {ks[0]}, khoản {ks[1]}"
                   if len(ks) == 2 else f"khoản {ks[0]} đến khoản {ks[-1]}")
            sid = f"{base_id}#k{ks[0]}-{ks[-1]}"
            sub = chunk_by_id.get(sid)
            if sub is None:
                sub = {"id": sid, "dieu_so": c.get("dieu_so", "0"), "unit_type": "dieu_sub",
                       "unit_no": c.get("unit_no", ""), "loai_vb": c.get("loai_vb", ""),
                       "so_hieu": c.get("so_hieu", ""), "khoan_ref": ref, "parent": base_id,
                       "text": head + "\n" + "\n".join(parts[j][1] for j in range(a, b + 1))}
                chunk_by_id[sid] = sub
            out.append(sub)
    return out


def expand_pool(top, max_add=None):
    """top-5 Điều + cụm khoản của chúng (ưu tiên Điều xếp trên). Điều đầy đủ luôn đứng đầu."""
    pool, seen = list(top), {c["id"] for c in top}
    if not USE_SUBSPAN:
        return pool
    add = []
    for c in top:
        for s in _subspans_of(c):
            if s["id"] not in seen:
                seen.add(s["id"])
                add.append(s)
    return pool + add[:(SUBSPAN_MAX_PER_Q if max_add is None else max_add)]


def ce_score_many(question, texts, models, toks, dev, max_length=1024, bs=16):
    m, t = models.get(dev), toks.get(dev)
    if m is None or not texts:
        return np.zeros(len(texts))
    out, i = [], 0
    while i < len(texts):
        part = texts[i:i + bs]
        try:
            with torch.no_grad():
                enc = t([[question, x] for x in part], padding=True, truncation=True, return_tensors="pt",
                        max_length=max_length).to(next(m.parameters()).device)
                out.extend(m(**enc, return_dict=True).logits.view(-1).float().cpu().tolist())
            i += len(part)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 1:
                torch.cuda.empty_cache()
                bs = max(1, bs // 2)
                continue
            raise
    return np.asarray(out, dtype=np.float64)


def t2_extras_v8(question, top, ce_top, dev, needs):
    """Như t2_extras (5 Điều) + tập ứng viên mở rộng. ex["pool_ids"][i] là ứng viên thứ i; 5 phần
    tử đầu là top-5 Điều theo CE zero-shot (nên nhánh chỉ-Điều dùng chung chỉ số)."""
    ex = t2_extras(question, top, ce_top, dev, needs & {"bc", "yn", "list"})
    n5 = len(top)
    ex["n5"] = n5
    ex["margin"] = float(ex["ce"][0] - ex["ce"][1]) if n5 > 1 else float("inf")
    if "listft" in needs:
        ex["listft"] = llm_listwise(question, [c["text"] for c in top], dev, ft=True)
    pool = expand_pool(top) if "sub" in needs else list(top)
    ex["pool_ids"] = [c["id"] for c in pool]
    ce_pool = np.asarray(ex["ce"], dtype=np.float64)
    if len(pool) > n5:
        ce_pool = np.concatenate([ce_pool, ce_score_many(question, [c["text"] for c in pool[n5:]],
                                                         reranker_models, reranker_tokenizers, dev)])
    ex["ce_pool"] = ce_pool
    if "ceft" in needs and reranker_models_ft.get(dev) is not None:
        ex["ceft_pool"] = ce_score_many(question, [c["text"] for c in pool], reranker_models_ft,
                                        reranker_tokenizers_ft, dev, max_length=CE_FT_MAX_LEN)
    return ex


# Nhánh T2 mới. Chỉ số trả về đánh vào ex["pool_ids"].
LOWM_THRESH = 0.0     # Cell 12 đặt = trung vị margin CE hạng 1-2 trên dev (không nhìn gold)
T2_ARMS["ceft"] = (lambda ex: ex["ceft_pool"][:ex["n5"]], {"ceft"})
T2_ARMS["ceft_ce_0.5"] = (lambda ex: _z(ex["ceft_pool"][:ex["n5"]]) + 0.5 * _z(ex["ce"]), {"ceft"})
T2_ARMS["sub_ce"] = (lambda ex: ex["ce_pool"], {"sub"})
T2_ARMS["sub_ceft"] = (lambda ex: ex["ceft_pool"], {"sub", "ceft"})
T2_ARMS["sub_ceft_ce_0.5"] = (lambda ex: _z(ex["ceft_pool"]) + 0.5 * _z(ex["ce_pool"]), {"sub", "ceft"})
for _a in T2_ALPHAS:
    T2_ARMS[f"listft_a{_a}"] = (lambda ex, a=_a: _z(ex["ce"]) + a * _z(ex["listft"]), {"listft"})
T2_ARMS["listft_only"] = (lambda ex: ex["listft"], {"listft"})
T2_ARMS["lowm_list_a0.5"] = (lambda ex: (_z(ex["ce"]) + 0.5 * _z(ex["list"]))
                             if ex["margin"] < LOWM_THRESH else ex["ce"], {"list"})
T2_ARMS["lowm_listft_a0.5"] = (lambda ex: (_z(ex["ce"]) + 0.5 * _z(ex["listft"]))
                               if ex["margin"] < LOWM_THRESH else ex["ce"], {"listft"})
LLM_NEEDS = {"yn", "list", "listft"}
T2_LLM_ARMS = {n for n, (_f, need) in T2_ARMS.items() if need & LLM_NEEDS}

# Tầng 3: câu trả lời do LLM-LoRA viết (cùng prompt FREE_USER nó được train).
G_ARMS_V8 = G_ARMS + ["ftfree", "ftfree_plus_verbatim"]


def compose_answer_v8(c, question, arm, gen):
    return compose_answer(c, question, arm[2:] if arm.startswith("ft") else arm, gen)


_n_long = sum(1 for c in all_chunks if str(c.get("unit_type", "")) == "dieu"
              and len(c["text"].split()) >= SUBSPAN_MIN_WORDS)
_demo = next((c for c in all_chunks if str(c.get("unit_type", "")) == "dieu"
              and len(c["text"].split()) >= 400 and len(_subspans_of(c)) >= 3), None)
print(f"  [v8] cụm khoản: {_n_long} Điều ≥{SUBSPAN_MIN_WORDS} từ có thể tách · {len(_CTX_PATH)} file "
      f"context cho Điều bị cắt")
if _demo is not None:
    _s = _subspans_of(_demo)
    print(f"    ví dụ {_demo['id']} ({len(_demo['text'].split())} từ) -> {len(_s)} cụm, vd {_s[0]['id']}: "
          f"{render_answer([_s[0]], 1, 'câu hỏi mẫu?')[:160]!r}")
checkpoint("Xong hàm v8 (cụm khoản + nhánh T2 mới)")


In [ ]:
# Cell 11c [v8]: NHÃN METEOR từ train (ĐÃ LOẠI dev) + HUẤN LUYỆN SONG SONG hai model
#   GPU đầu : CE-METEOR — fine-tune reranker tầng 2, nhãn mềm softmax(METEOR/τ) trên nhóm ứng
#             viên cùng phân phối lúc suy luận (Điều + cụm khoản trong văn bản gold / BM25).
#   GPU cuối: LoRA Qwen3-1.7B — (a) viết câu trả lời theo văn phong train.json từ ứng viên METEOR
#             cao nhất; (b) chọn ứng viên listwise, nhãn = ứng viên METEOR cao nhất.
# Khác v6_2 (reranker-ft âm hai nửa): nhãn không phải citation mà là METEOR thật của từng ứng
# viên — đúng thứ độ đo chấm, phủ cả câu KHÔNG có citation. Hỏng ở đâu thì nhánh tương ứng tắt,
# Cell 12 chạy như v7. rule.md §1.5: chỉ nhãn Task 2, model xuất phát là bản pretrain công khai.
import gc
import sys
import multiprocessing
from concurrent.futures import ThreadPoolExecutor as _TPE
import torch.nn.functional as F
import nltk

if str(NLTK_CACHE_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_CACHE_DIR))
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True, download_dir=str(NLTK_CACHE_DIR))
    nltk.download("omw-1.4", quiet=True, download_dir=str(NLTK_CACHE_DIR))

print("=== Bước 5e [v8]: nhãn METEOR + fine-tune CE-METEOR ∥ LoRA LLM ===")
ce_ft_info = {"used": False, "reason": None, "ckpt": None, "meta": None}
lora_info = {"used": False, "reason": None, "adapter": None, "meta": None}
label_info = {"n_q": 0, "n_pairs": 0, "n_groups": 0, "n_gen": 0, "n_list": 0, "elapsed_s": None,
              "stop": None}

# METEOR nhiều tiến trình: hàm worker phải nằm trong module import được (hàm định nghĩa trong
# notebook không pickle được qua mọi môi trường) -> ghi một module nhỏ vào CACHE_DIR.
_MW_PATH = os.path.join(CACHE_DIR, "v8_meteor_worker.py")
with open(_MW_PATH, "w", encoding="utf-8") as f:
    f.write("import nltk\n"
            "def init(p):\n"
            "    if p not in nltk.data.path:\n"
            "        nltk.data.path.insert(0, p)\n"
            "def score(pair):\n"
            "    from nltk.translate.meteor_score import meteor_score\n"
            "    try:\n"
            "        return float(meteor_score([pair[0].split()], pair[1].split()))\n"
            "    except Exception:\n"
            "        return 0.0\n")
if CACHE_DIR not in sys.path:
    sys.path.insert(0, CACHE_DIR)
import v8_meteor_worker as _mw
_mw.init(str(NLTK_CACHE_DIR))


def meteor_many(pairs):
    """[(gold, hyp)] -> [METEOR]. Fork nhiều tiến trình; gc.freeze() để tiến trình con không chạm
    vào object của cha (tránh copy-on-write nhân đôi ~20GB RAM). Lỗi thì lùi tuần tự."""
    if not pairs:
        return []
    n_proc = max(1, min(8, (os.cpu_count() or 2) - 1))
    if n_proc > 1 and len(pairs) > 200:
        gc.collect()
        gc.freeze()
        try:
            with multiprocessing.get_context("fork").Pool(
                    n_proc, initializer=_mw.init, initargs=(str(NLTK_CACHE_DIR),)) as p:
                return p.map(_mw.score, pairs, chunksize=64)
        except Exception as e:
            print(f"  [meteor] pool lỗi ({type(e).__name__}: {e}) -> tuần tự")
        finally:
            gc.unfreeze()
    return [_mw.score(x) for x in pairs]


_TOKSET = {}


def _tokset(c):
    s = _TOKSET.get(c["id"])
    if s is None:
        s = _TOKSET[c["id"]] = set(tokenize_simple(c["text"]))
    return s


def _label_candidates(q):
    """Ứng viên của một câu train: Điều trong văn bản gold (nếu có citation) + LABEL_DOC_K văn
    bản đầu BM25 tầng 1, lọc theo số từ trùng câu hỏi, cộng cụm khoản của các Điều đầu."""
    question = train_data[q]["question"]
    toks = tokenize_simple(question)
    docs = []
    gcid = train_positive.get(q)          # train_positive đã loại dev (Cell 7)
    if gcid:
        docs.append(str(gcid).split("_")[0])
    for i in bm25_t1.top_k(toks, 40):
        d = str(all_chunks_t1[i]["id"]).split("_")[0]
        if d not in docs:
            docs.append(d)
        if len(docs) >= LABEL_DOC_K + (1 if gcid else 0):
            break
    qt = set(toks)
    cand = [c for d in docs for c in dieu_by_doc.get(d, [])]
    cand = sorted(cand, key=lambda c: -len(qt & _tokset(c)))[:LABEL_MAX_DIEU]
    subs = []
    for c in cand[:LABEL_SUB_TOP]:
        subs.extend(_subspans_of(c))
    seen, out = set(), []
    for c in cand + subs[:LABEL_MAX_SUB]:
        if c["id"] not in seen:
            seen.add(c["id"])
            out.append(c)
    return out


def build_meteor_labels(qids, time_budget_sec, batch=200):
    out, t0, n_pairs = [], time.time(), 0
    stop = "hết câu"
    for b in range(0, len(qids), batch):
        if time.time() - t0 > time_budget_sec:
            stop = "chạm trần thời gian"
            break
        part, pairs, spans = qids[b:b + batch], [], []
        for q in part:
            question, gold = train_data[q]["question"], train_data[q]["answer"]
            cands = _label_candidates(q)
            if len(cands) < 2:
                continue
            spans.append((q, question, gold, cands, len(pairs)))
            pairs.extend((gold, render_answer([c], 1, question)) for c in cands)
        ms = meteor_many(pairs)
        n_pairs += len(pairs)
        for q, question, gold, cands, s in spans:
            out.append((q, question, gold, cands, np.asarray(ms[s:s + len(cands)], dtype=np.float64)))
        print(f"    [nhãn METEOR] {min(b + batch, len(qids))}/{len(qids)} câu · {n_pairs} cặp · "
              f"{(time.time() - t0) / 60:.1f} phút", flush=True)
    return out, n_pairs, stop


def make_training_sets(labeled, seed):
    rng = random.Random(seed)
    groups, gen_ex, list_ex = [], [], []
    for q, question, gold, cands, ms in labeled:
        order = [int(i) for i in np.argsort(-ms, kind="stable")]
        best = float(ms[order[0]])
        if best >= LABEL_MIN_BEST and best - float(ms.min()) >= LABEL_MIN_SPREAD:
            hard, rest = order[1:4], order[4:]
            rng.shuffle(rest)
            idx = [order[0]] + hard + rest[:max(0, CE_FT_GROUP - 1 - len(hard))]
            if len(idx) >= 4:
                groups.append({"q": question, "texts": [cands[i]["text"] for i in idx],
                               "m": [float(ms[i]) for i in idx]})
        if best >= LORA_FT_MIN_CTX_METEOR and len(gold.split()) <= LORA_FT_MAX_ANS_WORDS:
            gen_ex.append({"kind": "gen", "q": question, "c": cands[order[0]], "target": gold})
        if best >= LABEL_MIN_BEST and len(order) >= 5 and best - float(ms[order[1]]) >= 0.02:
            pick = [order[0]] + order[1:3] + rng.sample(order[3:], 2)
            rng.shuffle(pick)
            list_ex.append({"kind": "list", "q": question, "texts": [cands[i]["text"] for i in pick],
                            "target": str(pick.index(order[0]) + 1)})
    rng.shuffle(list_ex)
    return groups, gen_ex, list_ex[:max(len(gen_ex), 1)]


def finetune_reranker_soft(groups, base_model, device, time_budget_sec, lr, seed, tau, accum=8,
                           epochs=2, warmup=0.1, max_length=512, use_8bit_optim=False, log_every=25):
    """Công thức v6_2/Task 1 (1 nhóm/forward, tích luỹ, warmup + giảm tuyến tính, clip 1,0,
    gradient checkpointing, đóng băng embedding) — chỉ đổi đích: phân phối mềm theo METEOR."""
    cuda = device.startswith("cuda")
    tok = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=1).to(device)
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    for p_ in model.get_input_embeddings().parameters():
        p_.requires_grad = False
    params = [p_ for p_ in model.parameters() if p_.requires_grad]
    opt = None
    if use_8bit_optim:
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, lr=lr, weight_decay=0.01)
        except Exception as e:
            print(f"  [CE-METEOR] AdamW 8-bit không dùng được ({e}) -> AdamW thường")
    if opt is None:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=cuda)
    g = random.Random(seed)
    total = len(groups) * epochs
    planned = max(1, total // accum)

    def _lr_at(u):
        w = max(1, int(planned * warmup))
        return lr * (u + 1) / w if u < w else lr * max(0.0, (planned - u) / max(1, planned - w))

    model.train()
    t0, it, upd, run_loss, run_acc, seen, loss_log = time.time(), 0, 0, 0.0, 0.0, 0, []
    stop = "hết số epoch dự kiến"
    for ep in range(epochs):
        order = list(range(len(groups)))
        g.shuffle(order)
        for j in order:
            r = groups[j]
            enc = tok([[r["q"], t] for t in r["texts"]], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(device)
            target = torch.softmax(torch.tensor(r["m"], dtype=torch.float32, device=device) / tau, 0)
            with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
                logits = model(**enc).logits.view(-1)
            logp = torch.log_softmax(logits.float(), 0)
            loss = -(target * logp).sum()
            scaler.scale(loss / accum).backward()
            run_loss += loss.item()
            run_acc += float(int(torch.argmax(logits).item() == int(torch.argmax(target).item())))
            seen += 1
            it += 1
            if it == 40:
                rate = (time.time() - t0) / it
                planned = max(1, min(total, int(0.95 * time_budget_sec / max(rate, 1e-6))) // accum)
                print(f"    [CE-METEOR] {rate:.2f} giây/nhóm -> {planned} bước cập nhật "
                      f"(tối đa {total // accum})", flush=True)
            if it % accum == 0:
                for pg in opt.param_groups:
                    pg["lr"] = _lr_at(upd)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                upd += 1
                if upd % log_every == 0:
                    loss_log.append({"update": upd, "loss": round(run_loss / seen, 4),
                                     "acc_top1": round(run_acc / seen, 4),
                                     "min": round((time.time() - t0) / 60, 1)})
                    print(f"    CE-METEOR ep {ep} bước {upd}/{planned} · loss {run_loss/seen:.4f} · "
                          f"chọn đúng ứng viên METEOR cao nhất {run_acc/seen:.3f} · "
                          f"{(time.time()-t0)/60:.1f} phút", flush=True)
                    run_loss, run_acc, seen = 0.0, 0.0, 0
                if upd >= planned:
                    stop = "đủ số bước theo ngân sách"
                    break
                if time.time() - t0 > time_budget_sec:
                    stop = "chạm trần thời gian"
                    break
        else:
            continue
        break
    opt.zero_grad(set_to_none=True)
    model.eval()
    meta = {"updates": upd, "groups_seen": it, "planned_updates": planned, "n_groups": len(groups),
            "elapsed_s": round(time.time() - t0, 1), "stop_reason": stop, "lr": lr, "tau": tau,
            "accum": accum, "max_length": max_length, "loss_log": loss_log}
    return model, tok, meta


def finetune_llm_lora(examples, device, time_budget_sec, out_dir, seed):
    """SFT LoRA, 1 ví dụ/forward, loss CHỈ trên token đích (logits_to_keep -> không dựng logits cho
    cả prompt). Trọng số gốc fp16 đóng băng, adapter fp32 + autocast + GradScaler."""
    from peft import LoraConfig, get_peft_model
    cuda = device.startswith("cuda")
    tok = AutoTokenizer.from_pretrained(LLM_MODEL)
    end_id = tok.convert_tokens_to_ids("<|im_end|>")
    if not isinstance(end_id, int) or end_id == tok.unk_token_id:
        end_id = tok.eos_token_id
    data = []
    for ex in examples:
        user = (FREE_USER.format(q=ex["q"], ref=unit_ref(ex["c"]), article=article_text(ex["c"]))
                if ex["kind"] == "gen" else list_user(ex["q"], ex["texts"], list(range(len(ex["texts"])))))
        p_ids = tok(_chat(tok, user), add_special_tokens=False)["input_ids"]
        t_ids = tok(ex["target"], add_special_tokens=False)["input_ids"] + [end_id]
        if len(p_ids) + len(t_ids) <= LORA_FT_MAX_LEN:
            data.append((p_ids, t_ids, ex["kind"]))
    if len(data) < 20:
        raise RuntimeError(f"chỉ {len(data)} ví dụ vừa LORA_FT_MAX_LEN={LORA_FT_MAX_LEN}")
    random.Random(seed).shuffle(data)
    mdl = AutoModelForCausalLM.from_pretrained(LLM_MODEL)
    if cuda:
        mdl = mdl.half()
    mdl = mdl.to(device)
    mdl.config.use_cache = False
    try:
        mdl.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        mdl.gradient_checkpointing_enable()
    mdl.enable_input_require_grads()
    mdl = get_peft_model(mdl, LoraConfig(
        r=LORA_FT_R, lora_alpha=LORA_FT_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    for p_ in mdl.parameters():
        if p_.requires_grad and p_.dtype != torch.float32:
            p_.data = p_.data.float()
    params = [p_ for p_ in mdl.parameters() if p_.requires_grad]
    n_trainable = sum(p_.numel() for p_ in params)
    opt = torch.optim.AdamW(params, lr=LORA_FT_LR, weight_decay=0.0)
    scaler = torch.cuda.amp.GradScaler(enabled=cuda)
    planned = max(1, len(data) // LORA_FT_ACCUM)

    def _lr_at(u):
        w = max(1, int(0.05 * planned))
        return LORA_FT_LR * (u + 1) / w if u < w else LORA_FT_LR * max(0.0, (planned - u) / max(1, planned - w))

    mdl.train()
    t0, it, upd, bad, loss_log = time.time(), 0, 0, 0, []
    run = {"gen": [0.0, 0], "list": [0.0, 0]}
    stop = "hết dữ liệu (1 epoch)"
    for p_ids, t_ids, kind in data:
        ids = torch.tensor([p_ids + t_ids], device=device)
        tgt = torch.tensor(t_ids, device=device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            out = mdl(input_ids=ids, logits_to_keep=len(t_ids) + 1)
        loss = F.cross_entropy(out.logits[0, :-1].float(), tgt)
        it += 1
        if not torch.isfinite(loss):
            bad += 1
            if bad > 30 and bad > 0.3 * it:
                raise RuntimeError(f"loss không hữu hạn ở {bad}/{it} ví dụ (fp16 tràn số)")
            continue
        scaler.scale(loss / LORA_FT_ACCUM).backward()
        run[kind][0] += loss.item()
        run[kind][1] += 1
        if it == 20:
            rate = (time.time() - t0) / it
            planned = max(1, min(len(data), int(0.95 * time_budget_sec / max(rate, 1e-6))) // LORA_FT_ACCUM)
            print(f"    [LoRA] {rate:.2f} giây/ví dụ -> {planned} bước cập nhật "
                  f"(tối đa {len(data) // LORA_FT_ACCUM})", flush=True)
        if it % LORA_FT_ACCUM == 0:
            for pg in opt.param_groups:
                pg["lr"] = _lr_at(upd)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            upd += 1
            if upd % 10 == 0:
                rec = {"update": upd, "min": round((time.time() - t0) / 60, 1)}
                for k_, (s_, n_) in run.items():
                    rec[f"loss_{k_}"] = round(s_ / n_, 4) if n_ else None
                loss_log.append(rec)
                print(f"    LoRA bước {upd}/{planned} · loss sinh {rec['loss_gen']} · loss listwise "
                      f"{rec['loss_list']} · {rec['min']} phút", flush=True)
                run = {"gen": [0.0, 0], "list": [0.0, 0]}
            if upd >= planned:
                stop = "đủ số bước theo ngân sách"
                break
            if time.time() - t0 > time_budget_sec:
                stop = "chạm trần thời gian"
                break
    if upd == 0:
        raise RuntimeError("không có bước cập nhật nào")
    mdl.eval()
    mdl.save_pretrained(out_dir)
    meta = {"updates": upd, "examples_seen": it, "planned_updates": planned, "n_examples": len(data),
            "n_gen": sum(1 for d in data if d[2] == "gen"), "n_list": sum(1 for d in data if d[2] == "list"),
            "n_bad_loss": bad, "n_trainable": n_trainable, "elapsed_s": round(time.time() - t0, 1),
            "stop_reason": stop, "loss_log": loss_log}
    del mdl, opt
    gc.collect()
    torch.cuda.empty_cache()
    return meta


# ---------------------------------------------------------------------------
# Chạy
# ---------------------------------------------------------------------------
_want_ce = USE_CE_METEOR_FT and HAS_RERANKER
_want_lora = USE_LLM and USE_LLM_LORA and HAS_RERANKER
_need_after = PUBLIC_RESERVE_SEC + 130 * 60          # cổng dev + sinh public
_train_budget = max(CE_FT_TIME_BUDGET_SEC if _want_ce else 0, LORA_FT_TIME_BUDGET_SEC if _want_lora else 0)
if (_want_ce or _want_lora) and remaining() < _need_after + LABEL_TIME_BUDGET_SEC + 15 * 60:
    _r = f"không đủ giờ (còn {remaining()/60:.0f} phút)"
    ce_ft_info["reason"] = lora_info["reason"] = _r
    _want_ce = _want_lora = False
    print(f"  BỎ QUA nhãn + huấn luyện: {_r}")

labeled = []
if _want_ce or _want_lora:
    _t0 = time.time()
    _lab_qids = [q for q in train_data if q not in dev_ids_set and isinstance(train_data[q].get("answer"), str)
                 and isinstance(train_data[q].get("question"), str)]
    random.Random(SEED).shuffle(_lab_qids)
    _lab_qids = _lab_qids[:LABEL_MAX_Q]
    _lab_budget = min(LABEL_TIME_BUDGET_SEC, max(5 * 60, remaining() - _need_after - _train_budget - 10 * 60))
    labeled, _n_pairs, _stop = build_meteor_labels(_lab_qids, _lab_budget)
    ce_groups, gen_ex, list_ex = make_training_sets(labeled, SEED)
    label_info.update({"n_q": len(labeled), "n_pairs": _n_pairs, "n_groups": len(ce_groups),
                       "n_gen": len(gen_ex), "n_list": len(list_ex), "stop": _stop,
                       "elapsed_s": round(time.time() - _t0, 1)})
    if labeled:
        _bests = [float(x[4].max()) for x in labeled]
        _sub_best = sum(1 for x in labeled if x[3][int(np.argmax(x[4]))].get("unit_type") == "dieu_sub")
        label_info["oracle_mean"] = round(float(np.mean(_bests)), 4)
        label_info["best_is_subspan_pct"] = round(100 * _sub_best / len(labeled), 1)
    print(f"  nhãn: {label_info}")
    if len(ce_groups) < CE_FT_MIN_GROUPS:
        _want_ce = False
        ce_ft_info["reason"] = f"chỉ {len(ce_groups)} nhóm (< CE_FT_MIN_GROUPS={CE_FT_MIN_GROUPS})"
    if len(gen_ex) < LORA_FT_MIN_EX:
        _want_lora = False
        lora_info["reason"] = f"chỉ {len(gen_ex)} ví dụ sinh (< LORA_FT_MIN_EX={LORA_FT_MIN_EX})"


def _train_ce(dev, budget):
    ckpt = os.path.join(CHECKPOINT_DIR, "reranker-meteor-ft")
    m, t, meta = finetune_reranker_soft(ce_groups, RERANKER_BASE, dev, budget, CE_FT_LR, SEED, CE_FT_TAU,
                                        accum=CE_FT_ACCUM, epochs=CE_FT_EPOCHS, warmup=CE_FT_WARMUP,
                                        max_length=CE_FT_MAX_LEN, use_8bit_optim=CE_FT_8BIT)
    m.half().save_pretrained(ckpt)
    t.save_pretrained(ckpt)
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return ckpt, meta


def _train_lora(dev, budget):
    out = os.path.join(CHECKPOINT_DIR, "llm-lora")
    meta = finetune_llm_lora(gen_ex + list_ex, dev, budget, out, SEED)
    return out, meta


if _want_ce or _want_lora:
    _ce_dev, _llm_dev = DEVICES[0], DEVICES[-1]
    _budget_ce = min(CE_FT_TIME_BUDGET_SEC, max(5 * 60, remaining() - _need_after - 5 * 60))
    _budget_lora = min(LORA_FT_TIME_BUDGET_SEC, max(5 * 60, remaining() - _need_after - 5 * 60))
    _parked = []
    for ch in DENSE_CHANNELS:                       # nhường VRAM cho hai lượt train
        _d = str(next(ch["model"].parameters()).device)
        if _d.startswith("cuda"):
            ch["model"].to("cpu")
            _parked.append((ch, _d))
    torch.cuda.empty_cache()
    _jobs = []
    if _want_ce:
        _jobs.append(("ce", _train_ce, _ce_dev, _budget_ce))
    if _want_lora:
        _jobs.append(("lora", _train_lora, _llm_dev, _budget_lora))
    print(f"  huấn luyện: " + " ∥ ".join(f"{n} trên {d} ({b/60:.0f} phút)" for n, _f, d, b in _jobs))

    def _run(job):
        name, fn, dev, budget = job
        try:
            return name, fn(dev, budget), None
        except Exception as e:
            return name, None, f"{type(e).__name__}: {e}"

    _t0 = time.time()
    if _want_ce and _want_lora and _ce_dev != _llm_dev:
        with _TPE(max_workers=2) as _ex:
            _results = list(_ex.map(_run, _jobs))
    else:
        _results = [_run(j) for j in _jobs]
    for name, res, err in _results:
        info = ce_ft_info if name == "ce" else lora_info
        if err:
            info["reason"] = err
            print(f"  [{name}] LỖI -> bỏ nhánh này: {err}")
        else:
            path, meta = res
            info.update({"used": True, "meta": meta, ("ckpt" if name == "ce" else "adapter"): path})
            print(f"  [{name}] xong: {meta['updates']} bước, {meta['elapsed_s']/60:.1f} phút, dừng vì "
                  f"{meta['stop_reason']} -> {path}")
    for ch, _d in _parked:
        ch["model"].to(_d)
    torch.cuda.empty_cache()
    print(f"  tổng thời gian huấn luyện {(time.time()-_t0)/60:.1f} phút")
del labeled
gc.collect()
checkpoint(f"Xong Cell 11c (CE-METEOR={ce_ft_info['used']}, LoRA={lora_info['used']})")


In [ ]:
# Cell 11d [v8]: NẠP model SUY LUẬN sau huấn luyện — CE-METEOR (nếu train được) và LLM gốc +
# adapter LoRA (nếu train được), một bản mỗi GPU có reranker. Nạp lỗi thì tắt đúng nhánh đó.
print("=== Bước 5f [v8]: nạp CE-METEOR + LLM ===")
reranker_models_ft, reranker_tokenizers_ft = {}, {}
if ce_ft_info.get("ckpt"):
    for dev in RERANK_DEVICES:
        m, t = load_reranker_on(dev, ce_ft_info["ckpt"])
        if m is None:
            reranker_models_ft, reranker_tokenizers_ft = {}, {}
            ce_ft_info["reason"] = f"không nạp lại được {ce_ft_info['ckpt']} trên {dev}"
            break
        reranker_models_ft[dev], reranker_tokenizers_ft[dev] = m, t
HAS_CEFT = bool(reranker_models_ft) and set(reranker_models_ft) == set(RERANK_DEVICES)
HAS_FT = HAS_CEFT


def _load_all_llm(adapter):
    ms, ts = {}, {}
    for dev in RERANK_DEVICES:
        m, t, dt = _load_llm_on(dev, adapter)
        ms[dev], ts[dev] = m, t
        llm_info["dtype"][dev] = dt
        print(f"  {LLM_MODEL}{' + LoRA' if adapter else ''} trên {dev} ({dt})")
    return ms, ts


if USE_LLM and HAS_RERANKER:
    _adapter = lora_info.get("adapter")
    for _try in ([_adapter, None] if _adapter else [None]):
        try:
            llm_models, llm_tokenizers = _load_all_llm(_try)
            llm_info["adapter"] = _try
            llm_info["devices"] = list(llm_models)
            llm_info["error"] = None
            break
        except Exception as e:
            llm_info["error"] = f"{type(e).__name__}: {e}"
            print(f"  [LLM] nạp lỗi{' (có adapter)' if _try else ''}: {llm_info['error']}")
            llm_models, llm_tokenizers = {}, {}
            torch.cuda.empty_cache()
            if _try:
                lora_info["reason"] = f"nạp adapter lỗi: {llm_info['error']}"
if llm_tokenizers:
    _t = next(iter(llm_tokenizers.values()))
    YES_ID, NO_ID = _t.convert_tokens_to_ids("yes"), _t.convert_tokens_to_ids("no")
    DIGIT_IDS = [_t.convert_tokens_to_ids(str(d)) for d in range(1, 10)]
    _bad = [i for i in [YES_ID, NO_ID] + DIGIT_IDS[:TOP_K_RERANK] if i is None or i == _t.unk_token_id]
    if _bad:
        llm_info["error"] = f"token yes/no/chữ số không đơn token: {_bad}"
        print(f"  [LLM] {llm_info['error']} -> tắt mọi nhánh LLM")
        llm_models.clear()
        llm_tokenizers.clear()
HAS_LLM = bool(llm_models) and set(llm_models) == set(RERANK_DEVICES)
HAS_LLM_FT = HAS_LLM and bool(llm_info.get("adapter")) and all(
    hasattr(m, "disable_adapter") for m in llm_models.values())
if llm_models:
    llm_info["n_params"] = sum(p.numel() for p in next(iter(llm_models.values())).parameters())
    print(f"  LLM {llm_info['n_params']/1e9:.3f}B tham số (gồm adapter nếu có)")
print(f"  HAS_CEFT={HAS_CEFT} · HAS_LLM={HAS_LLM} · HAS_LLM_FT={HAS_LLM_FT}")
checkpoint(f"Xong nạp model suy luận (CEFT={HAS_CEFT}, LLM={HAS_LLM}, LoRA={HAS_LLM_FT})")


In [ ]:
# Cell 12 [v8]: Bước 6 — CÁC CỔNG TRÊN DEV rồi chốt HAI cấu hình
#   NOLLM : chỉ model trong REGISTERED_MODELS (+ bản fine-tune của chúng)    -> submission_nollm.zip
#   FULL  : được dùng LLM (gốc hoặc LoRA); CHỈ nhận nếu thắng NOLLM cả hai nửa -> submission.zip
# Thứ tự: R (tầng 1; v8 tắt mặc định) -> T2 (chọn ứng viên: Điều hoặc cụm khoản) -> câu dẫn -> G.
# Paired trên cùng 300 câu dev (đã loại khỏi mọi fine-tune ở Cell 7/11c). Chọn trên nửa A, chỉ nhận
# khi CẢ HAI nửa cùng dương so với mốc.
import itertools
import nltk
if str(NLTK_CACHE_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_CACHE_DIR))
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True, download_dir=str(NLTK_CACHE_DIR))
    nltk.download("omw-1.4", quiet=True, download_dir=str(NLTK_CACHE_DIR))
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def _meteor(ref: str, hyp: str) -> float:
    try:
        return float(meteor_score([str(ref).split()], str(hyp).split()))
    except Exception:
        return 0.0


def paired_gate(ids, base_fn, cand_fn):
    """Δ = cand - base theo từng câu, trên nửa A / nửa B (id sort, chẵn/lẻ) và toàn mẫu."""
    s = sorted(ids)

    def _stats(sub):
        d = [cand_fn(q) - base_fn(q) for q in sub]
        n = len(d)
        if n == 0:
            return {"n": 0, "delta": 0.0, "se": 0.0, "win": 0, "loss": 0}
        mean = sum(d) / n
        sd = (sum((x - mean) ** 2 for x in d) / (n - 1)) ** 0.5 if n > 1 else 0.0
        return {"n": n, "delta": round(mean, 5), "se": round(sd / n ** 0.5, 5),
                "win": sum(x > 1e-9 for x in d), "loss": sum(x < -1e-9 for x in d)}
    a, b = _stats(s[0::2]), _stats(s[1::2])
    return {"half_a": a, "half_b": b, "all": _stats(s),
            "pass": a["delta"] > 0 and b["delta"] > 0}


def choose_by_gate(label, ids, base_name, candidates: dict, base_fn):
    """candidates = {tên: cand_fn}. Chọn tên có Δ nửa A lớn nhất, NHẬN nếu hai nửa cùng dương."""
    report = {"base": base_name, "candidates": {}}
    for name, fn in candidates.items():
        report["candidates"][name] = paired_gate(ids, base_fn, fn)
    print(f"  --- Cổng {label} (n={len(ids)}, mốc = {base_name}) ---")
    for name, g_ in report["candidates"].items():
        a_, b_, al = g_["half_a"], g_["half_b"], g_["all"]
        weak = "  ⚠️ |Δ| < 2·SE" if abs(al["delta"]) < 2 * al["se"] else ""
        print(f"    {name:<34s} nửa A {a_['delta']:+.4f} · nửa B {b_['delta']:+.4f} · "
              f"toàn mẫu {al['delta']:+.4f} ± {al['se']:.4f} "
              f"({al['win']} thắng / {al['loss']} thua){weak}")
    winner = base_name
    if report["candidates"]:
        best = max(report["candidates"], key=lambda k: report["candidates"][k]["half_a"]["delta"])
        if report["candidates"][best]["pass"]:
            winner = best
    report["winner"] = winner
    print(f"    => chọn {winner}")
    return winner, report


def _run_on_devices(qids, fn, label):
    """fn(qid, dev) -> kết quả; chia đều cho các GPU có reranker."""
    devs = RERANK_DEVICES or [None]

    def _worker(chunk, dev, widx):
        out = {}
        for i, qid in enumerate(chunk):
            out[qid] = fn(qid, dev)
            _progress_print(label, widx, i, len(chunk))
        return out
    return parallel_process(list(qids), _worker, devs, label=label)


def _gold_doc(qid):
    cid = train_positive_all.get(qid)
    return str(cid).split("_")[0] if cid else None


def _pack(qid, question, gold, ranked, scores):
    """Top-K ứng viên kèm điểm CE và METEOR nếu chọn ứng viên đó (câu dẫn hiện hành + echo2)."""
    rec = {"qid": qid, "question": question, "n_cands": len(ranked) if ranked else 0,
           "cands": [], "feat": []}
    for i, c in enumerate((ranked or [])[:TOP_K_RERANK]):
        rec["cands"].append({
            "id": c["id"],
            "ce_score": float(scores[i]) if scores is not None else None,
            "meteor": _meteor(gold, render_answer([c], 1, question)),
            "n_words": len(c["text"].split()),
            "unit_type": c.get("unit_type", "dieu" if c.get("dieu_so", "0") != "0" else "raw"),
            "dieu_so": c.get("dieu_so", "0"),
            "so_hieu": c.get("so_hieu") or "",
        })
    return rec


def _top1(rec):
    return rec["cands"][0]["meteor"] if rec["cands"] else 0.0


def _rescore_meteor(rec):
    gold = train_data[rec["qid"]].get("answer", "")
    for cand in rec["cands"]:
        if cand["id"] in chunk_by_id:
            cand["meteor"] = _meteor(gold, render_answer([chunk_by_id[cand["id"]]], 1, rec["question"]))


def _elapsed_min(t0):
    return round((time.time() - t0) / 60, 1)


def _n_params(mdl):
    return sum(p.numel() for p in mdl.parameters()) if mdl is not None else 0


print("=== Bước 6 [v8]: Các cổng trên dev ===")
V8_DECISIONS = {"llm": llm_info, "ce_meteor_ft": ce_ft_info, "llm_lora": lora_info, "labels": label_info,
                "gate_retrieval": None, "gate_retrieval_final": None, "gate_retrieval_final_nollm": None,
                "retrieval_arms": None, "gate_t2": None, "gate_t2_nollm": None, "gate_template": None,
                "gate_gen": None, "gate_full_vs_nollm": None, "subspan": {}, "doc_recall_at_doc_k": {},
                "timing_min": {}}
V7_DECISIONS = V8_DECISIONS
dev_log = {q: {"qid": q, "gold_doc": _gold_doc(q)} for q in dev_ids}

# Ngân sách tham số (rule.md §2.1) đếm trên model THẬT; tổ hợp vượt 4B bị loại ngay tại cổng:
# encoder 1,16B + reranker 0,57B + CE-METEOR 0,57B + LLM 1,74B ≈ 4,04B -> CE-METEOR và LLM không
# đi cùng nhau được. Khi đó FULL (có LLM) chỉ xét nhánh T2 không dùng CE-METEOR.
N_PARAMS = {"encoders": sum(_n_params(ch["model"]) for ch in DENSE_CHANNELS),
            "reranker": _n_params(next(iter(reranker_models.values()))) if reranker_models else 0,
            "reranker_meteor_ft": _n_params(next(iter(reranker_models_ft.values()))) if reranker_models_ft else 0,
            "llm": _n_params(next(iter(llm_models.values()))) if llm_models else 0}


def _total_params(uses_ceft, uses_llm):
    return (N_PARAMS["encoders"] + N_PARAMS["reranker"]
            + (N_PARAMS["reranker_meteor_ft"] if uses_ceft else 0) + (N_PARAMS["llm"] if uses_llm else 0))


CEFT_WITH_LLM_OK = _total_params(True, True) < PARAM_BUDGET
V8_DECISIONS["n_params"] = dict(N_PARAMS, ceft_with_llm_ok=CEFT_WITH_LLM_OK)
print(f"  tham số: encoder {N_PARAMS['encoders']/1e9:.3f}B · reranker {N_PARAMS['reranker']/1e9:.3f}B · "
      f"CE-METEOR {N_PARAMS['reranker_meteor_ft']/1e9:.3f}B · LLM {N_PARAMS['llm']/1e9:.3f}B -> "
      f"CE-METEOR + LLM {_total_params(True, True)/1e9:.3f}B "
      f"({'vừa' if CEFT_WITH_LLM_OK else 'VƯỢT'} trần {PARAM_BUDGET/1e9:.0f}B)")

# ---------------------------------------------------------------------------
# LLM viết lại truy vấn cho 300 câu dev — chỉ khi đo lại cổng R.
# ---------------------------------------------------------------------------
dev_rw = {}
if HAS_LLM and V8_RUN_R_GATE:
    _t0 = time.time()
    _raw = llm_map({q: train_data[q]["question"] for q in dev_ids},
                   lambda x: REWRITE_USER.format(q=x), LLM_MAX_NEW_REWRITE, "viết lại truy vấn dev")
    dev_rw = {q: parse_rewrite(_raw.get(q, ""), train_data[q]["question"]) for q in dev_ids}
    V8_DECISIONS["timing_min"]["rewrite_dev"] = _elapsed_min(_t0)
    for q in dev_ids:
        dev_log[q]["rewrite"] = dev_rw[q]

# ---------------------------------------------------------------------------
# CỔNG R — tầng 1. v8 mặc định chỉ chạy cấu hình "base" (= v7 đã chốt), vẫn dựng r_res cho T2.
# ---------------------------------------------------------------------------


def _cfg(**kw):
    d = copy.deepcopy(BASE_CFG)
    d.update(kw)
    return d


R_ARMS = {"base": _cfg()}
if V8_RUN_R_GATE:
    for _n, _wd in RRF_WEIGHT_VARIANTS.items():
        R_ARMS[f"w_{_n}"] = _cfg(w=dict(_wd))
    for _kb in BM25_KB_GRID:
        R_ARMS[f"kb_{_kb[0]}_{_kb[1]}"] = _cfg(kb=tuple(_kb))
    R_ARMS["prf_add"] = _cfg(prf="add")
    R_ARMS["prf_replace"] = _cfg(prf="replace")
    if HAS_LLM:
        R_ARMS["rw_all"] = _cfg(rw="all")
        R_ARMS["rw_bm25"] = _cfg(rw="bm25")
R_GROUP = {"w": [n for n in R_ARMS if n.startswith("w_")],
           "kb": [n for n in R_ARMS if n.startswith("kb_")],
           "prf": [n for n in R_ARMS if n.startswith("prf_")],
           "rw": [n for n in R_ARMS if n.startswith("rw_")]}
dev_state = {}


def _r_eval(qid, dev, arms):
    q, gold = train_data[qid]["question"], train_data[qid].get("answer", "")
    st = dev_state.get(qid)
    if st is None:
        st = {"c": build_list_cache(q, dev_rw.get(qid), list(arms.values())), "memo_t1": {}, "memo_t2": {}}
        dev_state[qid] = st
    m, k = reranker_models.get(dev), reranker_tokenizers.get(dev)
    out = {}
    for name, cfg in arms.items():
        lists, w = lists_for(cfg, st["c"])
        diag = {}
        ranked, scores = t2_from_lists(q, lists, w, m, k, st["memo_t1"], st["memo_t2"], diag)
        out[name] = {"rec": _pack(qid, q, gold, ranked, scores), "top_docs": diag.get("top_docs", [])}
    return out


_t0 = time.time()
r_res = _run_on_devices(dev_ids, lambda q, d: _r_eval(q, d, R_ARMS), f"cổng R · {len(R_ARMS)} cấu hình")
r1_sec = time.time() - _t0
V8_DECISIONS["timing_min"]["gate_r_round1"] = round(r1_sec / 60, 1)
print(f"  [cổng R] {len(dev_ids)} câu × {len(R_ARMS)} cấu hình trong {r1_sec/60:.1f} phút")


def _gate_R(names, label):
    return choose_by_gate(label, dev_ids, "base",
                          {n: (lambda q, n_=n: _top1(r_res[q][n_]["rec"])) for n in names if n != "base"},
                          lambda q: _top1(r_res[q]["base"]["rec"]))


_w1, V8_DECISIONS["gate_retrieval"] = _gate_R(list(R_ARMS), "R · tầng 1 (vòng 1, từng nhánh)")
all_cfgs = dict(R_ARMS)
if RUN_R_ROUND2 and V8_RUN_R_GATE:
    rep = V8_DECISIONS["gate_retrieval"]["candidates"]
    best = {}
    for g, names in R_GROUP.items():
        pos = [n for n in names if rep.get(n, {}).get("all", {}).get("delta", 0.0) > 0]
        if pos:
            best[g] = max(pos, key=lambda n: rep[n]["all"]["delta"])
    combos = {}
    keys = list(best)
    for r in range(2, len(keys) + 1):
        for sub in itertools.combinations(keys, r):
            cfg = _cfg()
            for g in sub:
                cfg[g] = copy.deepcopy(R_ARMS[best[g]][g])
            combos["+".join(best[g] for g in sub)] = cfg
    est = 0.6 * r1_sec * len(combos) / max(len(R_ARMS), 1)
    if combos and est < remaining() - PUBLIC_RESERVE_SEC - 90 * 60:
        _t0 = time.time()
        dev_state.clear()
        r2 = _run_on_devices(dev_ids, lambda q, d: _r_eval(q, d, combos), f"cổng R · vòng 2 ({len(combos)} tổ hợp)")
        for q in dev_ids:
            r_res[q].update(r2[q])
        all_cfgs.update(combos)
        V8_DECISIONS["timing_min"]["gate_r_round2"] = _elapsed_min(_t0)

R_CFG_NAME_FULL, V8_DECISIONS["gate_retrieval_final"] = _gate_R(list(all_cfgs), "R · chốt FULL")
_nollm_names = [n for n in all_cfgs if not all_cfgs[n].get("rw")]
R_CFG_NAME_NOLLM, V8_DECISIONS["gate_retrieval_final_nollm"] = _gate_R(_nollm_names, "R · chốt NOLLM")
R_CFG_FULL, R_CFG_NOLLM = all_cfgs[R_CFG_NAME_FULL], all_cfgs[R_CFG_NAME_NOLLM]
V8_DECISIONS["retrieval_arms"] = {n: {k: (list(v) if isinstance(v, tuple) else v) for k, v in c.items()}
                                  for n, c in all_cfgs.items()}
_lab = [q for q in dev_ids if _gold_doc(q)]
for n in all_cfgs:
    if _lab:
        V8_DECISIONS["doc_recall_at_doc_k"][n] = round(
            sum(_gold_doc(q) in r_res[q][n]["top_docs"] for q in _lab) / len(_lab), 4)
for q in dev_ids:
    dev_log[q]["retrieval"] = {n: {"top1": (r_res[q][n]["rec"]["cands"][0]["id"]
                                            if r_res[q][n]["rec"]["cands"] else None),
                                   "meteor": round(_top1(r_res[q][n]["rec"]), 4),
                                   "top_docs": r_res[q][n]["top_docs"]} for n in all_cfgs}
dev_state.clear()
checkpoint(f"Xong cổng R (FULL={R_CFG_NAME_FULL}, NOLLM={R_CFG_NAME_NOLLM})")

# ---------------------------------------------------------------------------
# CỔNG T2 — chọn ỨNG VIÊN (Điều trong top-5 CE, hoặc cụm khoản của chúng).
# ---------------------------------------------------------------------------
_AVAIL = ({"bc"} | ({"yn", "list"} if HAS_LLM else set()) | ({"listft"} if HAS_LLM_FT else set())
          | ({"ceft"} if HAS_CEFT else set()) | ({"sub"} if USE_SUBSPAN else set()))
_AVAIL_NOLLM = _AVAIL - LLM_NEEDS
_AVAIL_FULL = _AVAIL if CEFT_WITH_LLM_OK else (_AVAIL - {"ceft"})
_arms_full = [a for a in T2_ARMS if a != "base" and T2_ARMS[a][1] <= _AVAIL_FULL]
_arms_nollm = [a for a in T2_ARMS if a != "base" and T2_ARMS[a][1] <= _AVAIL_NOLLM]
NEEDS_FULL = set().union(*[T2_ARMS[a][1] for a in _arms_full]) if _arms_full else set()
NEEDS_NOLLM = set().union(*[T2_ARMS[a][1] for a in _arms_nollm]) if _arms_nollm else set()
_SAME_R = R_CFG_NAME_NOLLM == R_CFG_NAME_FULL
print(f"  [T2] tín hiệu có: {sorted(_AVAIL)} · {len(_arms_full)} nhánh FULL · {len(_arms_nollm)} nhánh NOLLM")


def _rec_of(qid, path):
    return r_res[qid][R_CFG_NAME_FULL if path == "full" else R_CFG_NAME_NOLLM]["rec"]


def _t2_dev(qid, dev):
    out = {}
    for path in ("full", "nollm"):
        if path == "nollm" and _SAME_R:
            out["nollm"] = out["full"]
            continue
        rec = _rec_of(qid, path)
        if not rec["cands"] or rec["cands"][0]["ce_score"] is None:
            out[path] = None
            continue
        top = [chunk_by_id[c["id"]] for c in rec["cands"]]
        needs = (NEEDS_FULL | NEEDS_NOLLM) if _SAME_R else (NEEDS_FULL if path == "full" else NEEDS_NOLLM)
        out[path] = t2_extras_v8(rec["question"], top, [c["ce_score"] for c in rec["cands"]], dev, needs)
    return out


_t0 = time.time()
t2_ex = _run_on_devices(dev_ids, _t2_dev, "cổng T2 · chấm thêm")
V8_DECISIONS["timing_min"]["gate_t2"] = _elapsed_min(_t0)

# Ngưỡng "CE lưỡng lự" = trung vị margin hạng 1-2 — chỉ từ phân phối điểm, không nhìn gold.
_margins = [t2_ex[q]["full"]["margin"] for q in dev_ids
            if t2_ex[q]["full"] and np.isfinite(t2_ex[q]["full"]["margin"])]
LOWM_THRESH = float(np.median(_margins)) if _margins else 0.0
_POOL_M = {}


def _pool_meteor(qid, cid):
    key = (qid, cid, ANSWER_TEMPLATE)
    if key not in _POOL_M:
        _POOL_M[key] = _meteor(train_data[qid].get("answer", ""),
                               render_answer([chunk_by_id[cid]], 1, train_data[qid]["question"]))
    return _POOL_M[key]


def _t2_cid(qid, path, arm):
    ex = t2_ex[qid][path]
    if not ex:
        return None
    return ex["pool_ids"][_t2_order(ex, arm, ex["n5"])[0]]


def _t2_m(qid, path, arm):
    cid = _t2_cid(qid, path, arm)
    return _pool_meteor(qid, cid) if cid else 0.0


_n_pool = [len(t2_ex[q]["full"]["pool_ids"]) for q in dev_ids if t2_ex[q]["full"]]
_oracle_pool = [max(_pool_meteor(q, c) for c in t2_ex[q]["full"]["pool_ids"]) for q in dev_ids if t2_ex[q]["full"]]
_oracle_5 = [max(_pool_meteor(q, c) for c in t2_ex[q]["full"]["pool_ids"][:t2_ex[q]["full"]["n5"]])
             for q in dev_ids if t2_ex[q]["full"]]
V8_DECISIONS["subspan"] = {
    "pool_size_mean": round(float(np.mean(_n_pool)), 1) if _n_pool else 0,
    "oracle_top5_dieu": round(float(np.mean(_oracle_5)), 4) if _oracle_5 else None,
    "oracle_pool": round(float(np.mean(_oracle_pool)), 4) if _oracle_pool else None,
    "lowm_thresh": round(LOWM_THRESH, 4)}
print(f"  [T2] ứng viên/câu TB {V8_DECISIONS['subspan']['pool_size_mean']} · oracle top-5 Điều "
      f"{V8_DECISIONS['subspan']['oracle_top5_dieu']} -> oracle cả tập (có cụm khoản) "
      f"{V8_DECISIONS['subspan']['oracle_pool']} · ngưỡng lưỡng lự {LOWM_THRESH:.3f}")

T2_ARM_FULL, V8_DECISIONS["gate_t2"] = choose_by_gate(
    "T2 · chọn ứng viên (FULL)", dev_ids, "base",
    {a: (lambda q, a_=a: _t2_m(q, "full", a_)) for a in _arms_full}, lambda q: _t2_m(q, "full", "base"))
T2_ARM_NOLLM, V8_DECISIONS["gate_t2_nollm"] = choose_by_gate(
    "T2 · chọn ứng viên (NOLLM)", dev_ids, "base",
    {a: (lambda q, a_=a: _t2_m(q, "nollm", a_)) for a in _arms_nollm}, lambda q: _t2_m(q, "nollm", "base"))
for q in dev_ids:
    dev_log[q]["t2_full"] = {a: round(_t2_m(q, "full", a), 4) for a in ["base"] + _arms_full}
    dev_log[q]["t2_pick"] = {a: _t2_cid(q, "full", a) for a in ["base"] + _arms_full}
for _p, _a in (("full", T2_ARM_FULL), ("nollm", T2_ARM_NOLLM)):
    _picks = [chunk_by_id[_t2_cid(q, _p, _a)] for q in dev_ids if _t2_cid(q, _p, _a)]
    V8_DECISIONS["subspan"][f"picked_subspan_pct_{_p}"] = round(
        100 * sum(c.get("unit_type") == "dieu_sub" for c in _picks) / max(len(_picks), 1), 1)


def _chosen_chunk(qid, path):
    cid = _t2_cid(qid, path, T2_ARM_FULL if path == "full" else T2_ARM_NOLLM)
    return chunk_by_id[cid] if cid else None


checkpoint(f"Xong cổng T2 (FULL={T2_ARM_FULL}, NOLLM={T2_ARM_NOLLM})")

# ---------------------------------------------------------------------------
# CỔNG CÂU DẪN — 0 GPU, trên ứng viên cấu hình FULL chọn.
# ---------------------------------------------------------------------------
tmpl_m = {}
for q in dev_ids:
    c = _chosen_chunk(q, "full")
    gold, question = train_data[q].get("answer", ""), train_data[q]["question"]
    tmpl_m[q] = {t: (_meteor(gold, render_answer([c], 1, question, template=t)) if c else 0.0)
                 for t in ANSWER_TEMPLATES}
chosen_tmpl, V8_DECISIONS["gate_template"] = choose_by_gate(
    "câu dẫn", dev_ids, ANSWER_TEMPLATE,
    {t: (lambda q, t_=t: tmpl_m[q][t_]) for t in ANSWER_TEMPLATES if t != ANSWER_TEMPLATE},
    lambda q: tmpl_m[q][ANSWER_TEMPLATE])
if chosen_tmpl != ANSWER_TEMPLATE:
    ANSWER_TEMPLATE = chosen_tmpl
    for q in dev_ids:
        for n in {R_CFG_NAME_FULL, R_CFG_NAME_NOLLM}:
            _rescore_meteor(r_res[q][n]["rec"])

# ---------------------------------------------------------------------------
# CỔNG G — dựng câu trả lời bằng LLM (gốc / LoRA) trên ứng viên FULL đã chọn. Thiếu trích dẫn /
# quá ngắn thì compose_answer() tự lùi về template.
# ---------------------------------------------------------------------------
G_ARM_FULL = "base"
concl_raw, free_raw, free_ft_raw = {}, {}, {}


def _gen_of(q, arm, cr=None, fr=None, ffr=None):
    cr, fr, ffr = (concl_raw if cr is None else cr), (free_raw if fr is None else fr), \
        (free_ft_raw if ffr is None else ffr)
    return (cr if arm.startswith("concl") else ffr if arm.startswith("ft") else fr).get(q)


if HAS_LLM and RUN_GEN_GATE:
    if remaining() < PUBLIC_RESERVE_SEC + 60 * 60:
        V8_DECISIONS["gate_gen"] = {"skipped": f"không đủ giờ (còn {remaining()/60:.0f} phút)"}
        print(f"  [cổng G] BỎ QUA — {V8_DECISIONS['gate_gen']['skipped']}")
    else:
        _t0 = time.time()
        items = {}
        for q in dev_ids:
            c = _chosen_chunk(q, "full")
            if c is not None:
                items[q] = (train_data[q]["question"], c)
        _mk_c = lambda it: CONCL_USER.format(q=it[0], ref=unit_ref(it[1]), article=article_text(it[1]))
        _mk_f = lambda it: FREE_USER.format(q=it[0], ref=unit_ref(it[1]), article=article_text(it[1]))
        concl_raw = llm_map(items, _mk_c, LLM_MAX_NEW_CONCL, "câu kết dev")
        free_raw = llm_map(items, _mk_f, LLM_MAX_NEW_FREE, "câu trả lời dev")
        _g_arms = list(G_ARMS)
        if HAS_LLM_FT and remaining() > PUBLIC_RESERVE_SEC + 40 * 60:
            free_ft_raw = llm_map(items, _mk_f, LLM_MAX_NEW_FREE, "câu trả lời dev (LoRA)", ft=True)
            _g_arms = list(G_ARMS_V8)
        g_m = {}
        for q in dev_ids:
            gold = train_data[q].get("answer", "")
            if q not in items:
                g_m[q] = {a: 0.0 for a in _g_arms}
                continue
            question, c = items[q]
            g_m[q] = {a: _meteor(gold, compose_answer_v8(c, question, a, _gen_of(q, a))) for a in _g_arms}
            dev_log[q]["gen"] = {"concl": concl_raw.get(q), "free": free_raw.get(q),
                                 "free_ft": free_ft_raw.get(q),
                                 "meteor": {a: round(v, 4) for a, v in g_m[q].items()}}
        G_ARM_FULL, V8_DECISIONS["gate_gen"] = choose_by_gate(
            "G · dựng câu trả lời", dev_ids, "base",
            {a: (lambda q, a_=a: g_m[q][a_]) for a in _g_arms if a != "base"},
            lambda q: g_m[q]["base"])
        V8_DECISIONS["gate_gen"]["n_fallback"] = {
            a: sum(1 for q in items if compose_answer_v8(items[q][1], items[q][0], a, _gen_of(q, a))
                   == render_answer([items[q][1]], 1, items[q][0]))
            for a in _g_arms if a != "base"}
        V8_DECISIONS["timing_min"]["gate_gen"] = _elapsed_min(_t0)
        V8_DECISIONS["timing_min"]["gen_per_q_sec"] = round((time.time() - _t0) / max(len(items), 1), 2)
checkpoint(f"Xong cổng G (FULL={G_ARM_FULL})")

# ---------------------------------------------------------------------------
# Chốt: FULL (có LLM) chỉ được giữ nếu thắng NOLLM trên CẢ HAI nửa dev.
# ---------------------------------------------------------------------------
LLM_USED_FULL = HAS_LLM and (bool(R_CFG_FULL.get("rw")) or T2_ARM_FULL in T2_LLM_ARMS
                             or G_ARM_FULL != "base")


def _final_dev_answer(q, path):
    c = _chosen_chunk(q, path)
    question = train_data[q]["question"]
    if c is None:
        return "Không tìm thấy thông tin pháp lý cho câu hỏi này."
    if path == "full" and G_ARM_FULL != "base":
        return compose_answer_v8(c, question, G_ARM_FULL, _gen_of(q, G_ARM_FULL))
    return render_answer([c], 1, question)


_dev_pred = {p: {q: _final_dev_answer(q, p) for q in dev_ids} for p in ("full", "nollm")}
_dev_m = {p: {q: _meteor(train_data[q].get("answer", ""), _dev_pred[p][q]) for q in dev_ids}
          for p in ("full", "nollm")}
FULL_IS_NOLLM = True
if LLM_USED_FULL:
    V8_DECISIONS["gate_full_vs_nollm"] = paired_gate(dev_ids, lambda q: _dev_m["nollm"][q],
                                                     lambda q: _dev_m["full"][q])
    _g = V8_DECISIONS["gate_full_vs_nollm"]
    print(f"  FULL vs NOLLM: nửa A {_g['half_a']['delta']:+.4f} · nửa B {_g['half_b']['delta']:+.4f} "
          f"-> {'giữ FULL có LLM' if _g['pass'] else 'FULL = NOLLM'}")
    FULL_IS_NOLLM = not _g["pass"]
if FULL_IS_NOLLM:
    LLM_USED_FULL = False
_EFF_FULL = ("nollm", T2_ARM_NOLLM) if FULL_IS_NOLLM else ("full", T2_ARM_FULL)
CEFT_USED = {"nollm": "ceft" in T2_ARMS[T2_ARM_NOLLM][1], "full": "ceft" in T2_ARMS[_EFF_FULL[1]][1]}
LLM_FT_USED_FULL = LLM_USED_FULL and ("listft" in T2_ARMS[T2_ARM_FULL][1] or G_ARM_FULL.startswith("ft"))

dev_final = {}
for path, src in (("full", _EFF_FULL[0]), ("nollm", "nollm")):
    ms, rs = [], []
    for q in dev_ids:
        gold, pred = train_data[q].get("answer", ""), _dev_pred[src][q]
        ms.append(_meteor(gold, pred))
        rs.append(rouge.score(gold, pred)["rougeL"].fmeasure)
    dev_final[path] = {"meteor": round(sum(ms) / len(ms), 4), "rouge_l": round(sum(rs) / len(rs), 4)}
dev_final["base"] = {"meteor": round(sum(_top1(r_res[q]["base"]["rec"]) for q in dev_ids) / len(dev_ids), 4)}
print(f"  DEV cuối (n={len(dev_ids)}): mốc (R base + T2 base) {dev_final['base']['meteor']:.4f} · "
      f"NOLLM {dev_final['nollm']['meteor']:.4f} · FULL {dev_final['full']['meteor']:.4f} "
      f"(ROUGE-L {dev_final['full']['rouge_l']:.4f})")

V8_DECISIONS["active"] = {
    "full": ({"same_as": "nollm", "uses_llm": False} if FULL_IS_NOLLM else
             {"retrieval": R_CFG_NAME_FULL, "t2": T2_ARM_FULL, "gen": G_ARM_FULL, "uses_llm": LLM_USED_FULL,
              "uses_lora": LLM_FT_USED_FULL, "uses_ce_meteor": CEFT_USED["full"]}),
    "nollm": {"retrieval": R_CFG_NAME_NOLLM, "t2": T2_ARM_NOLLM, "gen": "base", "uses_llm": False,
              "uses_ce_meteor": CEFT_USED["nollm"]},
    "template": ANSWER_TEMPLATE,
    "channels": CHANNEL_NAMES + (["rw_" + n for n in CHANNEL_NAMES] if R_CFG_FULL.get("rw") == "all"
                                 else ["rw_bm25"] if R_CFG_FULL.get("rw") else [])
                + (["prf"] if R_CFG_FULL.get("prf") == "add" else []),
}
V8_DECISIONS["dev_final"] = dev_final
V62_DECISIONS = V8_DECISIONS          # Cell 15 đọc tên cũ
RERANKER_SOURCE = "zeroshot+meteor_ft" if any(CEFT_USED.values()) else "zeroshot"
top_n_answer = 1
use_reranker = HAS_RERANKER
use_adaptive = False

# harvest_records cho phân tích lỗi (Cell 15): cấu hình FULL thực tế, ứng viên xếp theo bộ chọn T2.
harvest_records = {}
for q in dev_ids:
    _p, _a = _EFF_FULL
    rec = _rec_of(q, _p)
    ex = t2_ex[q][_p]
    hr = {"qid": q, "question": rec["question"], "n_cands": rec["n_cands"], "cands": [], "feat": []}
    if ex:
        for i in _t2_order(ex, _a, ex["n5"])[:TOP_K_RERANK]:
            cid = ex["pool_ids"][i]
            c = chunk_by_id[cid]
            hr["cands"].append({"id": cid, "ce_score": float(ex["ce_pool"][i]) if i < len(ex["ce_pool"]) else None,
                                "meteor": _pool_meteor(q, cid), "n_words": len(c["text"].split()),
                                "unit_type": c.get("unit_type", "dieu"), "dieu_so": c.get("dieu_so", "0"),
                                "so_hieu": c.get("so_hieu") or ""})
    harvest_records[q] = hr
harvest_rate_s = r1_sec / max(len(dev_ids), 1)

recall_ids = [q for q in dev_ids if q in train_positive_all]
recall_at_k = {}
if recall_ids:
    for kk in (1, 3, 5):
        hit = sum(train_positive_all[q] in [c["id"] for c in harvest_records[q]["cands"][:kk]]
                  for q in recall_ids)
        recall_at_k[str(kk)] = round(hit / len(recall_ids), 4)
    print(f"  Recall@k Điều trên {len(recall_ids)} câu dev có citation (THAM KHẢO — cụm khoản không "
          f"trùng id Điều gold nên số này thấp hơn thực tế khi T2 chọn cụm khoản): {recall_at_k}")

eval_info = {"meteor": dev_final["full"]["meteor"], "rouge_l": dev_final["full"]["rouge_l"],
             "meteor_nollm": dev_final["nollm"]["meteor"], "rouge_l_nollm": dev_final["nollm"]["rouge_l"],
             "meteor_base": dev_final["base"]["meteor"], "n_dev": len(dev_ids), "recall_at_k": recall_at_k,
             "harvest_rate_s": round(harvest_rate_s, 2), "n_harvest_phase_a": len(dev_ids),
             "dev_arms_min": round(r1_sec / 60, 1)}

if HAS_LLM and not LLM_USED_FULL:
    print("  LLM không được dùng trong bài nộp -> giải phóng.")
    llm_models.clear()
    llm_tokenizers.clear()
    torch.cuda.empty_cache()
if reranker_models_ft and not any(CEFT_USED.values()):
    print("  CE-METEOR không qua cổng -> giải phóng.")
    reranker_models_ft.clear()
    reranker_tokenizers_ft.clear()
    torch.cuda.empty_cache()

with open(os.path.join(OUT_DIR, "v8_decisions.json"), "w", encoding="utf-8") as f:
    json.dump(V8_DECISIONS, f, ensure_ascii=False, indent=2, default=str)
with open(os.path.join(OUT_DIR, "v8_dev_arms.jsonl"), "w", encoding="utf-8") as f:
    for q in dev_ids:
        f.write(json.dumps(dev_log[q], ensure_ascii=False, default=str) + "\n")
print(f"  CẤU HÌNH CHỐT: {V8_DECISIONS['active']}")
checkpoint("Xong các cổng + dev-eval")


In [ ]:
# Cell 13 [v7]: LTR TẮT — v6_2 đo âm cả hai nửa (Δ −0,0068 / −0,0114, eval_harvest_summary.json).
# Giữ các biến mà Cell 15/16 đọc.
ltr_model = None
use_ltr = False
ltr_report = {"trained": False, "reason": "v7: tắt (v6_2 đo âm cả hai nửa)", "n_groups": 0,
              "delta_half_a": None, "delta_half_b": None, "accepted": False}
print(f"=== Bước 6b [v7]: {ltr_report['reason']} ===")


In [ ]:
# Cell 14 [v8]: Bước 7 — sinh public cho CẢ HAI cấu hình rồi đóng gói.
# submission_nollm.zip ghi TRƯỚC (không phụ thuộc LLM); sau đó mới sinh bằng LLM và ghi
# submission.zip. Ngân sách tham số đếm lại bằng numel() trên model thật trước khi ghi.
print("=== Bước 7 [v8]: Sinh câu trả lời cho public-official.json ===")
print(f"  FULL : {V8_DECISIONS['active']['full']}")
print(f"  NOLLM: {V8_DECISIONS['active']['nollm']} · câu dẫn={ANSWER_TEMPLATE}")
FALLBACK_ANSWER = "Không tìm thấy thông tin pháp lý cho câu hỏi này."


def build_submission(answers: dict, expected_ids: set, out_zip: Path) -> None:
    errors = []
    got = set(answers.keys())
    if got != expected_ids:
        errors.append(f"Key lệch: thiếu {len(expected_ids-got)}, thừa {len(got-expected_ids)}")
    for qid, ans in answers.items():
        if not isinstance(ans, str) or not ans.strip():
            errors.append(f"[{qid}] answer rỗng hoặc không phải string")
    if errors:
        raise ValueError("Submission KHÔNG hợp lệ:\n  - " + "\n  - ".join(errors[:20]))
    normalized = {qid: {"answer": str(ans)} for qid, ans in answers.items()}
    json_path = out_zip.with_suffix(".json")
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(normalized, f, ensure_ascii=False)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="submission.json")
    with zipfile.ZipFile(out_zip) as zf:
        assert zf.namelist() == ["submission.json"], f"zip chứa {zf.namelist()}"
        reloaded = json.loads(zf.read("submission.json").decode("utf-8"))
        assert reloaded == normalized
    print(f"  OK — {out_zip} ({len(normalized)} câu trả lời, đã kiểm tra lại từ đĩa)")


with open(PUBLIC_PATH, encoding="utf-8") as f:
    questions = json.load(f)
qids = list(questions.keys())
print(f"  {len(qids)} câu hỏi public.")

pub_rw = {}
if LLM_USED_FULL and R_CFG_FULL.get("rw"):
    _raw = llm_map({q: questions[q]["question"] for q in qids}, lambda x: REWRITE_USER.format(q=x),
                   LLM_MAX_NEW_REWRITE, "viết lại truy vấn public")
    pub_rw = {q: parse_rewrite(_raw.get(q, ""), questions[q]["question"]) for q in qids}

PUB_PATHS = [("nollm", R_CFG_NOLLM, T2_ARM_NOLLM)] + ([] if FULL_IS_NOLLM else [("full", R_CFG_FULL, T2_ARM_FULL)])
_PUB_CFGS = [cfg for _p, cfg, _a in PUB_PATHS]


def _pub_one(qid, dev):
    q = questions[qid]["question"]
    c = build_list_cache(q, pub_rw.get(qid), _PUB_CFGS)
    m, k = reranker_models.get(dev), reranker_tokenizers.get(dev)
    memo1, memo2, done, out = {}, {}, {}, {}
    by_key = {}
    for path, cfg, arm in PUB_PATHS:
        by_key.setdefault(json.dumps(cfg, sort_keys=True, default=str), []).append((path, cfg, arm))
    for key, paths in by_key.items():
        lists, w = lists_for(paths[0][1], c)
        ranked, scores = t2_from_lists(q, lists, w, m, k, memo1, memo2)
        top = (ranked or [])[:TOP_K_RERANK]
        needs = set().union(*[T2_ARMS[a][1] for _p, _c, a in paths])
        ex = (t2_extras_v8(q, top, scores[:len(top)], dev, needs)
              if top and scores is not None and any(a != "base" for _p, _c, a in paths) else None)
        for path, _cfg, arm in paths:
            if not top:
                out[path] = None
            elif arm == "base" or ex is None:
                out[path] = top[0]["id"]
            else:
                out[path] = ex["pool_ids"][_t2_order(ex, arm, ex["n5"])[0]]
    if FULL_IS_NOLLM:
        out["full"] = out["nollm"]
    return out


_t0 = time.time()
picked = _run_on_devices(qids, _pub_one, "Bước 7 · chọn ứng viên")
print(f"  chọn ứng viên cho {len(picked)} câu trong {(time.time()-_t0)/60:.1f} phút · cụm khoản: "
      f"NOLLM {sum(1 for q in qids if picked[q]['nollm'] and '#k' in picked[q]['nollm'])} câu, "
      f"FULL {sum(1 for q in qids if picked[q]['full'] and '#k' in picked[q]['full'])} câu")


def _render(qid, cid):
    return render_answer([chunk_by_id[cid]], 1, questions[qid]["question"]) if cid else FALLBACK_ANSWER


answers_nollm = {q: _render(q, picked[q]["nollm"]) for q in qids}
print("=== Bước 8a [v8]: submission_nollm.zip ===")
build_submission(answers_nollm, set(questions.keys()), Path(OUT_DIR) / "submission_nollm.zip")
checkpoint("✅ submission_nollm.zip đã an toàn trên đĩa")


def _n_params(mdl):
    return sum(p.numel() for p in mdl.parameters())


_ceft_n = _n_params(next(iter(reranker_models_ft.values()))) if reranker_models_ft else 0
PARAMS = {"encoders": sum(_n_params(ch["model"]) for ch in DENSE_CHANNELS),
          "reranker": _n_params(next(iter(reranker_models.values()))) if reranker_models else 0,
          "reranker_meteor_ft": _ceft_n,
          "llm": _n_params(next(iter(llm_models.values()))) if (LLM_USED_FULL and llm_models) else 0}
PARAMS["nollm_total"] = PARAMS["encoders"] + PARAMS["reranker"] + (_ceft_n if CEFT_USED["nollm"] else 0)
PARAMS["full_total"] = (PARAMS["encoders"] + PARAMS["reranker"] + (_ceft_n if CEFT_USED["full"] else 0)
                        + PARAMS["llm"])
print(f"  Tham số: NOLLM {PARAMS['nollm_total']/1e9:.3f}B · FULL {PARAMS['full_total']/1e9:.3f}B "
      f"(trần {PARAM_BUDGET/1e9:.0f}B, rule.md §2.1)")
if PARAMS["nollm_total"] >= PARAM_BUDGET:
    raise SystemExit(f"⛔ NOLLM {PARAMS['nollm_total']/1e9:.3f}B vượt trần — không được xảy ra, kiểm tra Cell 12")

full_gen, n_gen_fallback = {}, 0
if LLM_USED_FULL and G_ARM_FULL != "base":
    _ft = G_ARM_FULL.startswith("ft")
    _tmpl = CONCL_USER if G_ARM_FULL.startswith("concl") else FREE_USER
    _mx = LLM_MAX_NEW_CONCL if G_ARM_FULL.startswith("concl") else LLM_MAX_NEW_FREE
    _items = {q: (questions[q]["question"], chunk_by_id[picked[q]["full"]]) for q in qids if picked[q]["full"]}
    _est = V8_DECISIONS["timing_min"].get("gen_per_q_sec", 3.0) * len(_items) / 2
    if remaining() - _est < 20 * 60:
        print(f"  ⚠️ không đủ giờ cho LLM sinh (ước {_est/60:.0f} phút, còn {remaining()/60:.0f}) -> "
              f"FULL dùng template trên ứng viên FULL")
        G_ARM_FULL = "base"
    else:
        _t0 = time.time()
        full_gen = llm_map(_items, lambda it: _tmpl.format(q=it[0], ref=unit_ref(it[1]),
                                                            article=article_text(it[1])),
                           _mx, "Bước 7 · LLM sinh", ft=_ft)
        print(f"  LLM sinh {len(full_gen)} câu trong {(time.time()-_t0)/60:.1f} phút")

if FULL_IS_NOLLM:
    answers_full = dict(answers_nollm)
else:
    answers_full = {}
    for q in qids:
        cid = picked[q]["full"]
        if not cid:
            answers_full[q] = FALLBACK_ANSWER
            continue
        c = chunk_by_id[cid]
        answers_full[q] = compose_answer_v8(c, questions[q]["question"], G_ARM_FULL, full_gen.get(q))
        if G_ARM_FULL != "base" and answers_full[q] == render_answer([c], 1, questions[q]["question"]):
            n_gen_fallback += 1
    if G_ARM_FULL != "base":
        print(f"  {n_gen_fallback}/{len(qids)} câu lùi về template (LLM thiếu trích dẫn / quá ngắn)")

if PARAMS["full_total"] >= PARAM_BUDGET:
    print(f"  ⛔ FULL {PARAMS['full_total']/1e9:.3f}B >= {PARAM_BUDGET/1e9:.0f}B -> submission.zip dùng bản NOLLM")
    answers_full = dict(answers_nollm)

print("=== Bước 8b [v8]: submission.zip ===")
build_submission(answers_full, set(questions.keys()), Path(OUT_DIR) / "submission.zip")
answers = answers_full
n_diff = sum(answers_full[q] != answers_nollm[q] for q in qids)
print(f"  FULL khác NOLLM ở {n_diff}/{len(qids)} câu")
checkpoint("✅ submission.zip đã an toàn trên đĩa")
if elapsed() > SUBMISSION_DEADLINE_SEC:
    print(f"  [GHI CHÚ] Về đích ở phút {elapsed()/60:.1f}, muộn hơn mốc tự đặt "
          f"{SUBMISSION_DEADLINE_SEC/60:.0f} phút.")


In [ ]:
# Cell 15 [v6_1]: HARVEST PHA C (mở rộng) + PHÂN TÍCH LỖI
#
# Chạy SAU khi submission.zip đã nằm trên đĩa. Mọi thứ ở đây là phần ăn thêm: nếu Kaggle ngắt
# phiên giữa chừng, bài nộp không suy suyển.
#
# ⚠️ VÌ SAO KHÔNG PHẢI 7.000 CÂU — đọc kỹ trước khi tăng HARVEST_TARGET_TOTAL. Đo từ lượt
# thật: ~4,9 giây/câu với 2 GPU song song. 7.000 câu = 572 phút, tức nhiều hơn TOÀN BỘ phần
# fine-tune + encode corpus cộng lại (307 phút). Con số 7.000 không nằm trong bất kỳ phiên
# nào, dù có bỏ hết mọi thứ khác. Cỡ mẫu ở đây do THỜI GIAN CÒN LẠI quyết định.
#
# Và cỡ mẫu lớn hơn KHÔNG mua được nhiều như ta tưởng: với ~3.000 câu, sai số chuẩn của một
# tỉ lệ quanh 40% đã là √(0,4·0,6/3000) = 0,9 điểm phần trăm. Đủ để nói "lỗi xếp hạng chiếm
# 38-42%", mà đó chính là độ phân giải cần cho việc ra quyết định hướng đi. Gấp đôi mẫu chỉ
# thu sai số về 0,63 điểm — không đổi kết luận nào.
print("=== Bước 9 [v8]: Harvest mở rộng + phân tích lỗi ===")

print(f"  v8: phân tích lỗi trên {len(harvest_records)} câu dev (cấu hình FULL), không harvest thêm.")


# ---------------------------------------------------------------------------
# PHÂN LOẠI LỖI. Bốn nhóm, phân biệt được bằng hai con số đã có sẵn trong cache:
#   chosen  = METEOR của ứng viên pipeline THỰC SỰ chọn (hạng 1 sau LTR nếu có)
#   oracle  = METEOR của ứng viên TỐT NHẤT trong top-K
#
#   ok              chosen đã đủ tốt -> không phải lỗi
#   retrieval_fail  oracle thấp -> KHÔNG ứng viên nào cứu được câu này. Bộ chọn bó tay;
#                   muốn chữa phải sửa tầng 1 (chọn văn bản) hoặc tầng cắt Điều.
#   ranking_fail    oracle cao hơn chosen rõ rệt -> đáp án tốt ĐÃ nằm trong tay mà không
#                   chọn. Đây đúng là loại lỗi LTR nhắm tới, và là chỗ +7,7 điểm của §7.
#   extraction_weak chosen ≈ oracle nhưng cả hai đều thấp -> chọn đúng Điều rồi mà điểm vẫn
#                   kém: lỗi ở khâu dựng câu trả lời (template/độ dài/ranh giới Điều), không
#                   phải ở khâu chọn.
#
# Ranh giới giữa bốn nhóm là NGƯỠNG TỰ ĐẶT (ERR_* ở Cell 2), không phải sự thật khách quan.
# Tỉ lệ tuyệt đối vì thế đọc kèm ngưỡng; cái đáng tin là so sánh giữa các lượt CÙNG ngưỡng.
# ---------------------------------------------------------------------------
def classify_error(chosen_m: float, oracle_m: float) -> str:
    if chosen_m >= ERR_OK_METEOR:
        return "ok"
    if oracle_m < ERR_RETRIEVAL_CEIL:
        return "retrieval_fail"
    if oracle_m - chosen_m >= ERR_RANKING_GAP:
        return "ranking_fail"
    return "extraction_weak"


def _chosen_index(rec) -> int:
    """Ứng viên pipeline thật sự chọn — phải khớp ĐÚNG cấu hình đã nộp, kể cả LTR."""
    if use_ltr and ltr_model is not None and rec["feat"]:
        try:
            pred = ltr_model.predict(np.asarray(rec["feat"], dtype=np.float64))
            return int(np.argmax(pred))
        except Exception:
            return 0
    return 0


rows_out, summary_counts = [], {}
m_chosen_all, m_oracle_all, ce_gap_01 = [], [], []
rank_of_best, n_ok_rank1 = {}, 0
by_unit = {}

for qid, rec in harvest_records.items():
    gold = str(train_data[qid].get("answer", ""))
    if not rec["cands"]:
        rows_out.append({"qid": qid, "question": rec["question"], "error_type": "no_candidate",
                          "meteor": 0.0, "oracle_best_meteor": 0.0, "oracle_gap": 0.0})
        summary_counts["no_candidate"] = summary_counts.get("no_candidate", 0) + 1
        continue
    ms = [c["meteor"] for c in rec["cands"]]
    ci = _chosen_index(rec)
    oi = int(np.argmax(ms))
    chosen_m, oracle_m = ms[ci], ms[oi]
    etype = classify_error(chosen_m, oracle_m)

    summary_counts[etype] = summary_counts.get(etype, 0) + 1
    m_chosen_all.append(chosen_m)
    m_oracle_all.append(oracle_m)
    rank_of_best[oi] = rank_of_best.get(oi, 0) + 1
    if ci == oi:
        n_ok_rank1 += 1
    if len(rec["cands"]) > 1 and rec["cands"][0]["ce_score"] is not None \
            and rec["cands"][1]["ce_score"] is not None:
        ce_gap_01.append(rec["cands"][0]["ce_score"] - rec["cands"][1]["ce_score"])
    u = rec["cands"][ci]["unit_type"]
    by_unit.setdefault(u, []).append(chosen_m)

    rows_out.append({
        "qid": qid,
        "question": rec["question"],
        "error_type": etype,
        "meteor": round(chosen_m, 4),
        "oracle_best_meteor": round(oracle_m, 4),
        "oracle_gap": round(oracle_m - chosen_m, 4),
        "chosen_rank": ci,
        "oracle_rank": oi,
        "n_candidates": rec["n_cands"],
        "gold_len_words": len(gold.split()),
        # Đủ để tra ngược một câu cụ thể mà không phải chạy lại GPU: id Điều tra được trong
        # corpus, điểm CE cho biết reranker "tự tin" tới đâu, METEOR cho biết lẽ ra được bao
        # nhiêu nếu chọn ứng viên đó.
        "candidates": [{
            "rank": i, "id": c["id"], "dieu_so": c["dieu_so"], "so_hieu": c["so_hieu"],
            "unit_type": c["unit_type"], "n_words": c["n_words"],
            "ce_score": (round(c["ce_score"], 4) if c["ce_score"] is not None else None),
            "meteor_if_chosen": round(c["meteor"], 4),
        } for i, c in enumerate(rec["cands"])],
    })

n_tot = max(len(m_chosen_all), 1)
summary = {
    "meta": {
        "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
        "n_harvested": len(harvest_records),
        "n_scored": len(m_chosen_all),
        "harvest_rate_s_per_q": round(harvest_rate_s, 2),
        "config": {"top_n": top_n_answer, "agg_mode": AGG_MODE, "use_ltr": bool(use_ltr),
                    "reranker": RERANKER_SOURCE, "top_k_candidates": LTR_TOP_K_CANDIDATES,
                    "answer_template": ANSWER_TEMPLATE,
                    "channels": V62_DECISIONS["active"]["channels"]},
        "thresholds": {"ok": ERR_OK_METEOR, "ranking_gap": ERR_RANKING_GAP,
                        "retrieval_ceil": ERR_RETRIEVAL_CEIL},
        "sample_note": (
            "Mẫu gồm câu dev (đã loại khỏi fine-tune) + câu KHÔNG có nhãn citation (chưa bao "
            "giờ vào fine-tune). Không câu nào bị contamination bởi fine-tune encoder. Điểm "
            "vẫn cao hơn public vì template render_answer khớp văn phong train.json "
            "(result.md §5) — đó là lệch phân phối, không phải rò rỉ."),
    },
    "scores": {
        "meteor_mean": round(sum(m_chosen_all) / n_tot, 4),
        "oracle_meteor_mean": round(sum(m_oracle_all) / n_tot, 4),
        "gap_to_oracle": round((sum(m_oracle_all) - sum(m_chosen_all)) / n_tot, 4),
        "pick_best_rate": round(n_ok_rank1 / n_tot, 4),
        "ce_gap_rank0_rank1_median": (round(float(np.median(ce_gap_01)), 4) if ce_gap_01 else None),
    },
    "error_distribution": {k: {"count": v, "pct": round(100 * v / max(len(rows_out), 1), 1)}
                            for k, v in sorted(summary_counts.items(), key=lambda kv: -kv[1])},
    "where_best_candidate_sits": {str(k): v for k, v in sorted(rank_of_best.items())},
    "meteor_by_unit_type": {k: {"n": len(v), "meteor": round(sum(v) / len(v), 4)}
                             for k, v in sorted(by_unit.items(), key=lambda kv: -len(kv[1]))},
    "ltr": ltr_report,
}

full_path = os.path.join(OUT_DIR, "eval_harvest_full.json")
sum_path = os.path.join(OUT_DIR, "eval_harvest_summary.json")
with open(full_path, "w", encoding="utf-8") as f:
    json.dump(rows_out, f, ensure_ascii=False, indent=1)
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n  --- TỔNG HỢP trên {len(rows_out)} câu ---")
print(f"    METEOR thực tế   {summary['scores']['meteor_mean']:.4f}")
print(f"    METEOR oracle    {summary['scores']['oracle_meteor_mean']:.4f}  "
      f"(dư địa còn {summary['scores']['gap_to_oracle']:+.4f})")
print(f"    chọn đúng ứng viên tốt nhất: {100*summary['scores']['pick_best_rate']:.1f}%")
if summary["scores"]["ce_gap_rank0_rank1_median"] is not None:
    print(f"    khoảng cách điểm CE hạng 1 vs hạng 2 (trung vị): "
          f"{summary['scores']['ce_gap_rank0_rank1_median']:.4f}")
print("    phân bố lỗi:")
for k, v in summary["error_distribution"].items():
    print(f"      {k:<16s} {v['count']:>5d}  ({v['pct']:>4.1f}%)")
print(f"\n  Đã ghi:\n    {full_path}  (chi tiết từng câu — dùng để soi ca lỗi cụ thể)"
      f"\n    {sum_path}  (tổng hợp — dùng để quyết định hướng đi)")
checkpoint("Xong phân tích lỗi")


In [ ]:
# Cell 16 [v8]: Sổ thí nghiệm — 1 dòng JSON/lượt, APPEND (không ghi đè).
# Chỉ đọc biến thực sự tồn tại trong notebook này (bài học v6: cell log crash vì biến đã xoá).
n_empty = sum(1 for a in answers.values() if not a.strip())
record = {
    "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
    "version": "v8",
    "seed": SEED, "use_warmup": USE_WARMUP, "n_warmup_used": n_warmup_used,
    "hardware": (f"kaggle_t4x{N_GPU}" if IS_KAGGLE else f"local_{N_GPU}gpu"),
    "concl": CONCL, "answer_template": ANSWER_TEMPLATE,
    "dense_model_a": BASE_DENSE_MODEL_A, "dense_model_b": BASE_DENSE_MODEL_B,
    "agg_mode": AGG_MODE, "reranker_source": RERANKER_SOURCE,
    "doc_k": DOC_K, "max_dieu_candidates": MAX_DIEU_CANDIDATES, "top_n_answer": top_n_answer,
    "n_dev_excluded_from_train": n_dev_had_label,
    "n_train_pairs_available": finetune_info["n_pairs_available"],
    "n_train_pairs_used": finetune_info["n_pairs_used"],
    "used_finetune": finetune_info["used_finetune"], "finetune_reason": finetune_info["reason"],
    "finetune_models": finetune_info["models"],
    "v8_flags": {"r_gate": V8_RUN_R_GATE, "subspan": USE_SUBSPAN, "subspan_max_win": SUBSPAN_MAX_WIN,
                 "subspan_min_words": SUBSPAN_MIN_WORDS, "ce_meteor_ft": USE_CE_METEOR_FT,
                 "llm_lora": USE_LLM_LORA},
    "labels": label_info,
    "ce_meteor_ft": {k: v for k, v in ce_ft_info.items() if k != "meta"},
    "ce_meteor_ft_meta": {k: v for k, v in (ce_ft_info.get("meta") or {}).items() if k != "loss_log"},
    "llm_lora": {k: v for k, v in lora_info.items() if k != "meta"},
    "llm_lora_meta": {k: v for k, v in (lora_info.get("meta") or {}).items() if k != "loss_log"},
    "llm": llm_info, "llm_used_full": LLM_USED_FULL, "llm_lora_used_full": LLM_FT_USED_FULL,
    "full_is_nollm": FULL_IS_NOLLM, "params": PARAMS,
    "active": V8_DECISIONS["active"], "subspan": V8_DECISIONS["subspan"],
    "dev_meteor": eval_info["meteor"], "dev_rouge_l": eval_info["rouge_l"],
    "dev_meteor_nollm": eval_info["meteor_nollm"], "dev_meteor_base": eval_info["meteor_base"],
    "dev_n": eval_info["n_dev"], "dev_recall_at_k": eval_info["recall_at_k"],
    "harvest_meteor": summary["scores"]["meteor_mean"],
    "harvest_oracle_meteor": summary["scores"]["oracle_meteor_mean"],
    "harvest_pick_best_rate": summary["scores"]["pick_best_rate"],
    "error_distribution": {k: v["pct"] for k, v in summary["error_distribution"].items()},
    "n_empty_answers": n_empty, "n_public_full_differs_nollm": n_diff,
    "n_public_gen_fallback": n_gen_fallback,
    "elapsed_min": round(elapsed() / 60, 1),
    "v8_decisions": V8_DECISIONS,
    "encode_info": encode_info,
    "resource_log": _RESOURCE_LOG,
}
try:
    with open(EXPERIMENT_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    print(f"  Đã ghi thêm 1 dòng vào {EXPERIMENT_LOG_PATH}")
except OSError as e:
    print(f"  [CẢNH BÁO] Không ghi được sổ thí nghiệm ({e}) — không ảnh hưởng submission.")

print("  [SỔ THÍ NGHIỆM — copy dòng dưới đây nếu cần đối chiếu sau này]")
print("  " + json.dumps({k: v for k, v in record.items() if k not in ("v8_decisions", "resource_log")},
                        ensure_ascii=False, default=str))

print(f"\n{'='*78}\nXONG v8 — {elapsed()/60:.1f} phút")
print(f"  submission.zip             FULL  (LLM: {LLM_USED_FULL}, LoRA: {LLM_FT_USED_FULL}"
      f"{', = NOLLM' if FULL_IS_NOLLM else ''}) · {PARAMS['full_total']/1e9:.3f}B tham số")
print(f"  submission_nollm.zip       NOLLM (chỉ model đã đăng ký + bản fine-tune) · {PARAMS['nollm_total']/1e9:.3f}B")
print(f"  dev METEOR: mốc {eval_info['meteor_base']:.4f} · NOLLM {eval_info['meteor_nollm']:.4f} "
      f"· FULL {eval_info['meteor']:.4f}")
print(f"  v8_decisions.json          số đo từng cổng (R / T2 / câu dẫn / G / FULL-vs-NOLLM)")
print(f"  v8_dev_arms.jsonl          từng câu dev × từng nhánh, kèm ứng viên được chọn và văn bản LLM sinh")
print(f"  eval_harvest_*.json        phân tích lỗi trên dev (cấu hình FULL thực tế)")
print(f"  experiment_log.jsonl       sổ thí nghiệm")
print('='*78)
